<a href="https://colab.research.google.com/github/luhcasmacedo/luhcasmacedo.github.io/blob/main/Scopus_Retrieval_Protocol_2026_Systematic_Literature_Review_Master's_Thesis_(Santos_Mac%C3%AAdo%2C_2027).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install requests pandas openpyxl

In [2]:
import getpass

API_KEY = getpass.getpass("Cole sua nova API Key da Elsevier: ")

if not API_KEY:
    raise ValueError("Nenhuma API Key foi inserida.")

print("API Key recebida com sucesso.")

Cole sua nova API Key da Elsevier: ··········
API Key recebida com sucesso.


In [3]:
# ============================================================
# BUSCA SCOPUS — CHINA STUDIES / TECHNOLOGY TRANSFER
# ============================================================

# Lista de periódicos e seus ISSNs
journals = {
    "China Economic Review": ["1043-951X", "1873-7781"],
    "Journal of Contemporary China": ["1067-0564", "1469-9400"],
    "The China Quarterly": ["0305-7410", "1468-2648"],
    "Journal of Chinese Political Science": ["1080-6954", "1874-6357"],
    "China & World Economy": ["1671-2234", "1749-124X"],
    "The Chinese Economy": ["1097-1475", "1558-0954"],
    "Journal of Chinese Economic and Business Studies": ["1476-5284", "1476-5292"],
    "Chinese Sociological Review": ["2162-0555", "2162-0563"],
    "Journal of Chinese Governance": ["2381-2346", "2381-2354"],
    "China Review": ["1680-2012", "1015-6607"],
    "China Economic Journal": ["1753-8963", "1753-8971"],
    "Chinese Political Science Review": ["2365-4244", "2365-4252"],
    "Economic and Political Studies": ["2095-4816", "2470-4024"],
    "The Chinese Journal of International Politics": ["1750-8916", "1750-8924"],
    "Journal of Current Chinese Affairs": ["1868-1026", "1868-4874"],
    "The China Journal": ["1324-9347", "1835-8535"]
}


# ============================================================
# CONSTRUIR A PARTE DOS PERIÓDICOS
# ============================================================

journal_queries = []

for journal, issns in journals.items():

    issn_query = " OR ".join(
        [f'ISSN({issn})' for issn in issns]
    )

    journal_queries.append(f"({issn_query})")


journals_part = " OR ".join(journal_queries)


# ============================================================
# TERMOS DA SUA RSL
# ============================================================

technology_terms = '''
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
'''


# ============================================================
# CONSULTA COMPLETA
# ============================================================

SEARCH_QUERY = f'''
(
    {journals_part}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{technology_terms}
'''.replace("\n", " ").strip()


print("============================================================")
print("CONSULTA SCOPUS")
print("============================================================")
print(SEARCH_QUERY)

CONSULTA SCOPUS
(     (ISSN(1043-951X) OR ISSN(1873-7781)) OR (ISSN(1067-0564) OR ISSN(1469-9400)) OR (ISSN(0305-7410) OR ISSN(1468-2648)) OR (ISSN(1080-6954) OR ISSN(1874-6357)) OR (ISSN(1671-2234) OR ISSN(1749-124X)) OR (ISSN(1097-1475) OR ISSN(1558-0954)) OR (ISSN(1476-5284) OR ISSN(1476-5292)) OR (ISSN(2162-0555) OR ISSN(2162-0563)) OR (ISSN(2381-2346) OR ISSN(2381-2354)) OR (ISSN(1680-2012) OR ISSN(1015-6607)) OR (ISSN(1753-8963) OR ISSN(1753-8971)) OR (ISSN(2365-4244) OR ISSN(2365-4252)) OR (ISSN(2095-4816) OR ISSN(2470-4024)) OR (ISSN(1750-8916) OR ISSN(1750-8924)) OR (ISSN(1868-1026) OR ISSN(1868-4874)) OR (ISSN(1324-9347) OR ISSN(1835-8535)) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" )


In [4]:
print("\nREVISTAS INCLUÍDAS:")
print("=" * 60)

for i, journal in enumerate(journals.keys(), start=1):
    print(f"{i:02d}. {journal}")


REVISTAS INCLUÍDAS:
01. China Economic Review
02. Journal of Contemporary China
03. The China Quarterly
04. Journal of Chinese Political Science
05. China & World Economy
06. The Chinese Economy
07. Journal of Chinese Economic and Business Studies
08. Chinese Sociological Review
09. Journal of Chinese Governance
10. China Review
11. China Economic Journal
12. Chinese Political Science Review
13. Economic and Political Studies
14. The Chinese Journal of International Politics
15. Journal of Current Chinese Affairs
16. The China Journal


In [5]:
# ============================================================
# TESTAR A CONSULTA NA SCOPUS API
# ============================================================

import requests

url = "https://api.elsevier.com/content/search/scopus"

headers = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json"
}

params = {
    "query": SEARCH_QUERY,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data = response.json()

    total_results = int(
        data["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(f"Total de resultados encontrados: {total_results}")

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 32


In [6]:
# ============================================================
# BAIXAR TODOS OS RESULTADOS DA SCOPUS
# ============================================================

import requests
import time

all_results = []

count = 25
start = 0

while start < total_results:

    params = {
        "query": SEARCH_QUERY,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data["search-results"].get("entry", [])

    all_results.extend(entries)

    print(
        f"Baixados: {len(all_results)} / {total_results}"
    )

    start += count

    time.sleep(1)

print("\n✅ Busca concluída!")
print(f"Total de registros baixados: {len(all_results)}")

Baixados: 25 / 32
Baixados: 32 / 32

✅ Busca concluída!
Total de registros baixados: 32


In [7]:
# ============================================================
# ORGANIZAR OS RESULTADOS
# ============================================================

import pandas as pd

# Criar dicionário ISSN → nome da revista
issn_to_journal = {}

for journal, issns in journals.items():
    for issn in issns:
        issn_to_journal[issn.replace("-", "")] = journal


records = []

for item in all_results:

    # ISSN
    issn = item.get("prism:issn", "")
    issn_clean = issn.replace("-", "")

    # Identificar revista
    journal_name = issn_to_journal.get(
        issn_clean,
        item.get("prism:publicationName", "")
    )

    # Data
    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    # DOI
    doi = item.get("prism:doi", "")

    # Scopus URL
    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    record = {
        "Journal": journal_name,
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": issn,
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": doi,
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    }

    records.append(record)


df = pd.DataFrame(records)

print(f"Total de artigos: {len(df)}")

display(df)

Total de artigos: 32


,Journal,Title,Authors,Year,ISSN,Volume,Issue,Pages,DOI,EID,Cited by,Scopus URL
0,China Review,Does Chinese Capital Help Industrial Developme...,Chen L.,2026,16802012,26,3,73-100,,2-s2.0-105047479470,0,https://www.scopus.com/inward/record.uri?partn...
1,China Review,Negotiating Global China Beyond the State–Dias...,Cao N.,2026,16802012,26,3,1-17,,2-s2.0-105047391936,1,https://www.scopus.com/inward/record.uri?partn...
2,China Economic Journal,"Intelligent manufacturing, persistent innovati...",Pu Z.,2026,17538963,,,None,10.1080/17538963.2026.2731874,2-s2.0-105050746125,0,https://www.scopus.com/inward/record.uri?partn...
3,The Chinese Economy,Made in China 2025 and the Catalytic Role of O...,Yuan Y.,2026,10971475,,,None,10.1080/10971475.2026.2658941,2-s2.0-105038119443,0,https://www.scopus.com/inward/record.uri?partn...
4,China Economic Review,A blessing and a curse: Identifying how knowle...,Qin X.,2025,1043951X,91,,None,10.1016/j.chieco.2025.102416,2-s2.0-105002816487,6,https://www.scopus.com/inward/record.uri?partn...
5,China & World Economy,Impact of National Technology Transfer Markets...,Li J.,2025,16712234,33,3,76-108,10.1111/cwe.12590,2-s2.0-105005112865,4,https://www.scopus.com/inward/record.uri?partn...
6,Journal of Chinese Economic and Business Studies,Reconsidering the pollution haven hypothesis: ...,Ekesiobi C.,2025,14765284,23,2,233-257,10.1080/14765284.2024.2435227,2-s2.0-85212219175,14,https://www.scopus.com/inward/record.uri?partn...
7,China Economic Review,Research on contracting institutions and conve...,Li J.,2024,1043951X,84,,None,10.1016/j.chieco.2024.102129,2-s2.0-85183972955,2,https://www.scopus.com/inward/record.uri?partn...
8,China Economic Review,Marginal returns on Chinese agricultural techn...,Olasehinde T.S.,2023,1043951X,78,,None,10.1016/j.chieco.2023.101935,2-s2.0-85147868074,9,https://www.scopus.com/inward/record.uri?partn...
9,Economic and Political Studies,Does technology transfer from the US to China ...,Menaldo V.,2021,20954816,9,4,417-446,10.1080/20954816.2021.1933768,2-s2.0-85108658301,5,https://www.scopus.com/inward/record.uri?partn...


In [8]:
# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts = (
    df["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts)

,Journal,Number of articles
0,Journal of Chinese Economic and Business Studies,9
1,China Economic Review,8
2,Economic and Political Studies,4
3,China Economic Journal,4
4,China Review,2
5,The China Quarterly,2
6,The Chinese Economy,1
7,China & World Economy,1
8,Journal of Chinese Political Science,1


In [9]:
# ============================================================
# EXPORTAR RESULTADOS PARA EXCEL
# ============================================================

filename = "Scopus_China_Journals_Technology_Transfer_2004_2026.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    # --------------------------------------------------------
    # ABA 1 — TODOS OS ARTIGOS
    # --------------------------------------------------------
    df.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    # --------------------------------------------------------
    # ABA 2 — CONTAGEM POR REVISTA
    # --------------------------------------------------------
    journal_counts.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    # --------------------------------------------------------
    # ABA 3 — LOG DA BUSCA
    # --------------------------------------------------------

    search_log = pd.DataFrame({
        "Field": [
            "Database",
            "Search date",
            "Number of journals",
            "Journals",
            "Publication years",
            "Document type",
            "Search field",
            "Search terms",
            "Source identification",
            "Total results"
        ],

        "Value": [
            "Scopus",
            pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
            len(journals),
            "; ".join(journals.keys()),
            "2003–2027",
            "Article",
            "TITLE-ABS-KEY",
            '"Technology Transfer" OR "Transfer of Technology" OR "Technological Transfer"',
            "ISSN",
            len(df)
        ]
    })

    search_log.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("✅ Excel criado:")
print(filename)

✅ Excel criado:
Scopus_China_Journals_Technology_Transfer_2004_2026.xlsx


In [10]:
# ============================================================
# BAIXAR O ARQUIVO
# ============================================================

from google.colab import files

files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# ============================================================
# TERMOS — TECHNOLOGY INTERNALIZATION
# ============================================================

technology_terms = '''
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
'''

In [12]:
# ============================================================
# CONSULTA COMPLETA
# ============================================================

SEARCH_QUERY = f'''
(
    {journals_part}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{technology_terms}
'''.replace("\n", " ").strip()


print("============================================================")
print("CONSULTA SCOPUS")
print("============================================================")
print(SEARCH_QUERY)

CONSULTA SCOPUS
(     (ISSN(1043-951X) OR ISSN(1873-7781)) OR (ISSN(1067-0564) OR ISSN(1469-9400)) OR (ISSN(0305-7410) OR ISSN(1468-2648)) OR (ISSN(1080-6954) OR ISSN(1874-6357)) OR (ISSN(1671-2234) OR ISSN(1749-124X)) OR (ISSN(1097-1475) OR ISSN(1558-0954)) OR (ISSN(1476-5284) OR ISSN(1476-5292)) OR (ISSN(2162-0555) OR ISSN(2162-0563)) OR (ISSN(2381-2346) OR ISSN(2381-2354)) OR (ISSN(1680-2012) OR ISSN(1015-6607)) OR (ISSN(1753-8963) OR ISSN(1753-8971)) OR (ISSN(2365-4244) OR ISSN(2365-4252)) OR (ISSN(2095-4816) OR ISSN(2470-4024)) OR (ISSN(1750-8916) OR ISSN(1750-8924)) OR (ISSN(1868-1026) OR ISSN(1868-4874)) OR (ISSN(1324-9347) OR ISSN(1835-8535)) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" )


In [13]:
SEARCH_QUERY

'(     (ISSN(1043-951X) OR ISSN(1873-7781)) OR (ISSN(1067-0564) OR ISSN(1469-9400)) OR (ISSN(0305-7410) OR ISSN(1468-2648)) OR (ISSN(1080-6954) OR ISSN(1874-6357)) OR (ISSN(1671-2234) OR ISSN(1749-124X)) OR (ISSN(1097-1475) OR ISSN(1558-0954)) OR (ISSN(1476-5284) OR ISSN(1476-5292)) OR (ISSN(2162-0555) OR ISSN(2162-0563)) OR (ISSN(2381-2346) OR ISSN(2381-2354)) OR (ISSN(1680-2012) OR ISSN(1015-6607)) OR (ISSN(1753-8963) OR ISSN(1753-8971)) OR (ISSN(2365-4244) OR ISSN(2365-4252)) OR (ISSN(2095-4816) OR ISSN(2470-4024)) OR (ISSN(1750-8916) OR ISSN(1750-8924)) OR (ISSN(1868-1026) OR ISSN(1868-4874)) OR (ISSN(1324-9347) OR ISSN(1835-8535)) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" )'

In [14]:
all_results = []

count = 25
start = 0

while start < total_results:

    params = {
        "query": SEARCH_QUERY,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    page_data = response.json()
    entries = page_data["search-results"].get("entry", [])

    all_results.extend(entries)

    print(f"Baixados: {len(all_results)} / {total_results}")

    start += count
    time.sleep(1)

print("✅ Busca concluída!")

Baixados: 1 / 32
Baixados: 2 / 32
✅ Busca concluída!


In [15]:
'"Technology Internalization" OR "Internalization of Technology" OR "Technological Internalization"',

('"Technology Internalization" OR "Internalization of Technology" OR "Technological Internalization"',)

In [16]:
filename = "Scopus_China_Journals_Technology_Internalization_2004_2026.xlsx"

In [17]:
params = {
    "query": SEARCH_QUERY,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data = response.json()

    total_results = int(
        data["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(f"Total de resultados encontrados: {total_results}")

else:
    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [18]:
all_results = []

count = 25
start = 0

while start < total_results:

    params = {
        "query": SEARCH_QUERY,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data["search-results"].get("entry", [])

    all_results.extend(entries)

    print(f"Baixados: {len(all_results)} / {total_results}")

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(f"Total de registros baixados: {len(all_results)}")


✅ Busca concluída!
Total de registros baixados: 0


In [19]:
print("Total de resultados da busca:", total_results)
print("Total baixado:", len(all_results))

Total de resultados da busca: 0
Total baixado: 0


In [20]:
# ============================================================
# ORGANIZAR OS RESULTADOS DA BUSCA
# ============================================================

records = []

for item in all_results:

    # ISSN
    issn = item.get("prism:issn", "")
    issn_clean = issn.replace("-", "")

    # Identificar periódico
    journal_name = issn_to_journal.get(
        issn_clean,
        item.get("prism:publicationName", "")
    )

    # Data
    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    # DOI
    doi = item.get("prism:doi", "")

    # URL do Scopus
    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records.append({
        "Journal": journal_name,
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": issn,
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": doi,
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df = pd.DataFrame(records)

print(f"✅ Tabela criada!")
print(f"Total de artigos: {len(df)}")

display(df)

✅ Tabela criada!
Total de artigos: 0


""


In [21]:
# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts = pd.DataFrame({
    "Journal": list(journals.keys()),
    "Number of articles": [0] * len(journals)
})

display(journal_counts)

,Journal,Number of articles
0,China Economic Review,0
1,Journal of Contemporary China,0
2,The China Quarterly,0
3,Journal of Chinese Political Science,0
4,China & World Economy,0
5,The Chinese Economy,0
6,Journal of Chinese Economic and Business Studies,0
7,Chinese Sociological Review,0
8,Journal of Chinese Governance,0
9,China Review,0


In [22]:
# ============================================================
# EXPORTAR BUSCA TECHNOLOGY INTERNALIZATION
# ============================================================

filename = "Scopus_China_Journals_Technology_Internalization_2004_2026.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    # Resultados — vazio, pois a busca retornou 0
    df.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    # Contagem por periódico
    journal_counts.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    # Search Log
    search_log.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename)

✅ Excel criado com sucesso!
Scopus_China_Journals_Technology_Internalization_2004_2026.xlsx


In [23]:
from google.colab import files

files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# ============================================================
# VERIFICAR REVISTAS NO SCOPUS PELO ISSN
# ============================================================

journals_to_check = {
    "Review of International Political Economy": ["0969-2290", "1466-4526"],
    "International Organization": ["0020-8183", "1531-5088"],
    "International Affairs": ["0020-5850", "1468-2346"],
    "Foreign Affairs": ["0015-7120", "2327-505X"],
    "Journal of Peace Research": ["0022-3433", "1460-3578"],
    "Journal of Democracy": ["1045-5736", "1086-3214"],
    "Journal of Conflict Resolution": ["0022-0027", "1552-8766"],
    "Journal of European Integration": ["0703-6337", "1477-2280"],
    "Third World Quarterly": ["0143-6597", "1360-2241"],
    "International Studies Review": ["1521-9488", "1468-2486"],
    "International Studies Quarterly": ["0020-8833", "1468-2478"],
    "Geopolitics": ["1465-0045", "1557-3028"],
    "European Journal of International Relations": ["1354-0661", "1460-3691"],
    "Global Policy": ["1758-5880", "1758-5899"],
    "Review of International Studies": ["0260-2105", "1469-9044"],
    "The Review of International Organizations": ["1559-7431", "1559-744X"],
    "Contemporary Security Policy": ["1352-3260", "1743-8764"],
    "The Pacific Review": ["0951-2748", "1470-1332"]
}

print(f"{len(journals_to_check)} revistas serão verificadas.")

18 revistas serão verificadas.


In [25]:
# ============================================================
# CONSULTAR SCOPUS SERIAL TITLE API
# ============================================================

import requests
import time
import pandas as pd

serial_url = "https://api.elsevier.com/content/serial/title"

headers_serial = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json"
}

scopus_sources = []

for journal, issns in journals_to_check.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_sources.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if not found:

        scopus_sources.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")

    else:
        print(f"✅ Encontrado: {journal}")


✅ Encontrado: Review of International Political Economy
✅ Encontrado: International Organization
✅ Encontrado: International Affairs
✅ Encontrado: Foreign Affairs
✅ Encontrado: Journal of Peace Research
✅ Encontrado: Journal of Democracy
✅ Encontrado: Journal of Conflict Resolution
✅ Encontrado: Journal of European Integration
✅ Encontrado: Third World Quarterly
✅ Encontrado: International Studies Review
✅ Encontrado: International Studies Quarterly
✅ Encontrado: Geopolitics
✅ Encontrado: European Journal of International Relations
✅ Encontrado: Global Policy
✅ Encontrado: Review of International Studies
✅ Encontrado: The Review of International Organizations
✅ Encontrado: Contemporary Security Policy
✅ Encontrado: The Pacific Review


In [26]:
# ============================================================
# BUSCA: TECHNOLOGY TRANSFER
# 18 REVISTAS CONFIRMADAS NO SCOPUS
# ============================================================

# Transformar os resultados da verificação em DataFrame
scopus_sources_df = pd.DataFrame(scopus_sources)

# Selecionar somente as revistas confirmadas
confirmed_journals = scopus_sources_df[
    scopus_sources_df["Status"] == "FOUND IN SCOPUS"
].copy()

print(f"Revistas confirmadas no Scopus: {len(confirmed_journals)}")

# Criar consulta com os ISSNs confirmados
issn_queries = []

for _, row in confirmed_journals.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries.append(f'ISSN({issn})')

journals_part = " OR ".join(issn_queries)

# Termos da busca
technology_terms = '''
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
'''

# Consulta completa
SEARCH_QUERY = f'''
(
    {journals_part}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{technology_terms}
'''.replace("\n", " ").strip()

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY)

print("\n============================================================")
print(f"Revistas confirmadas: {len(confirmed_journals)}")
print(f"18 revistas? {len(confirmed_journals) == 18}")
print("============================================================")

Revistas confirmadas no Scopus: 18

CONSULTA GERADA
(     ISSN(0969-2290) OR ISSN(0020-8183) OR ISSN(0020-5850) OR ISSN(0015-7120) OR ISSN(0022-3433) OR ISSN(1045-5736) OR ISSN(0022-0027) OR ISSN(0703-6337) OR ISSN(0143-6597) OR ISSN(1521-9488) OR ISSN(0020-8833) OR ISSN(1465-0045) OR ISSN(1354-0661) OR ISSN(1758-5880) OR ISSN(0260-2105) OR ISSN(1559-7431) OR ISSN(1352-3260) OR ISSN(0951-2748) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" )

Revistas confirmadas: 18
18 revistas? True


In [27]:
# ============================================================
# EXECUTAR BUSCA NO SCOPUS — TECHNOLOGY TRANSFER
# ============================================================

params = {
    "query": SEARCH_QUERY,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data = response.json()

    total_results = int(
        data["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(f"Total de resultados encontrados: {total_results}")

else:
    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 23


In [28]:
# ============================================================
# BAIXAR OS 23 RESULTADOS
# ============================================================

all_results = []

count = 25
start = 0

while start < total_results:

    params = {
        "query": SEARCH_QUERY,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data["search-results"].get("entry", [])

    all_results.extend(entries)

    print(f"Baixados: {len(all_results)} / {total_results}")

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(f"Total de registros baixados: {len(all_results)}")

Baixados: 23 / 23

✅ Busca concluída!
Total de registros baixados: 23


In [29]:
# ============================================================
# ORGANIZAR OS 23 RESULTADOS EM DATAFRAME
# ============================================================

records = []

for item in all_results:

    # ISSN
    issn = item.get("prism:issn", "")

    # Periódico
    journal_name = item.get(
        "prism:publicationName",
        ""
    )

    # Data / ano
    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    # DOI
    doi = item.get("prism:doi", "")

    # Autores
    authors = item.get("dc:creator", "")

    # URL do Scopus
    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records.append({
        "Journal": journal_name,
        "Title": item.get("dc:title", ""),
        "Authors": authors,
        "Year": year,
        "ISSN": issn,
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": doi,
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

# Criar DataFrame
df = pd.DataFrame(records)

print("✅ Tabela criada!")
print(f"Total de artigos: {len(df)}")

display(df)

✅ Tabela criada!
Total de artigos: 23


,Journal,Title,Authors,Year,ISSN,Volume,Issue,Pages,DOI,EID,Cited by,Scopus URL
0,International Studies Quarterly,"Divide and Conquer: Industry Market Structure,...",Minnich J.,2026,00208833,70,3,None,10.1093/isq/sqag047,2-s2.0-105042168715,0,https://www.scopus.com/inward/record.uri?partn...
1,International Affairs,The US as ‘leader’ and China as ‘collaborator’...,Chen Y.,2026,00205850,102,2,617-638,10.1093/ia/iiaf268,2-s2.0-105032463447,0,https://www.scopus.com/inward/record.uri?partn...
2,Geopolitics,"The United States, China, and Strategic Techno...",Haar R.N.,2026,14650045,,,None,10.1080/14650045.2026.2626266,2-s2.0-105031586612,1,https://www.scopus.com/inward/record.uri?partn...
3,Geopolitics,"Green Windows or Old Traps? Geopoliticization,...",Cerioli L.,2026,14650045,,,None,10.1080/14650045.2026.2659603,2-s2.0-105035913932,0,https://www.scopus.com/inward/record.uri?partn...
4,Geopolitics,A New Era of Vaccine Diplomacy: Navigating the...,Surianta A.,2025,14650045,30,5,2408-2435,10.1080/14650045.2025.2485206,2-s2.0-105002706388,4,https://www.scopus.com/inward/record.uri?partn...
5,European Journal of International Relations,Keep your enemies safer: technical cooperation...,Ding J.,2024,13540661,30,4,918-945,10.1177/13540661241246622,2-s2.0-85191713035,2,https://www.scopus.com/inward/record.uri?partn...
6,Pacific Review,"North Korean climate diplomacy: engagement, pr...",Scartozzi C.M.,2024,09512748,37,4,825-852,10.1080/09512748.2023.2261646,2-s2.0-85173772511,2,https://www.scopus.com/inward/record.uri?partn...
7,Review of International Organizations,Behind the screen: Understanding national supp...,Chan Z.T.,2022,15597431,17,3,513-541,10.1007/s11558-021-09436-y,2-s2.0-85110775641,67,https://www.scopus.com/inward/record.uri?partn...
8,Global Policy,Knowledge Transfer in the Global South: Reusin...,Alves A.C.,2022,17585880,13,S1,45-57,10.1111/1758-5899.13060,2-s2.0-85127727653,22,https://www.scopus.com/inward/record.uri?partn...
9,Third World Quarterly,ICT for development and the novel principles o...,Rothe F.F.,2022,01436597,43,6,1495-1514,10.1080/01436597.2022.2060202,2-s2.0-85128744878,9,https://www.scopus.com/inward/record.uri?partn...


In [30]:
# ============================================================
# CONTAGEM DE ARTIGOS POR PERIÓDICO
# ============================================================

journal_counts = (
    df["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts)

,Journal,Number of articles
0,Global Policy,7
1,Third World Quarterly,5
2,Geopolitics,3
3,Review of International Organizations,2
4,Pacific Review,2
5,International Studies Quarterly,1
6,International Affairs,1
7,European Journal of International Relations,1
8,Review of International Political Economy,1


In [31]:
# ============================================================
# SEARCH LOG
# ============================================================

search_log = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_journals),
        "; ".join(confirmed_journals["Journal"].tolist()),
        "2003–2027",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Transfer" OR "Transfer of Technology" OR "Technological Transfer"',
        "ISSN",
        len(df)
    ]
})

display(search_log)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 00:41:19
2,Number of journals,18
3,Journals,Review of International Political Economy; Int...
4,Publication years,2003–2027
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Transfer"" OR ""Transfer of Technolo..."
8,Source identification,ISSN
9,Total results,23


In [32]:
# ============================================================
# EXPORTAR DATASET
# ============================================================

filename = "Scopus_Diplomacy_International_Relations_Technology_Transfer_2004_2026.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename)

✅ Excel criado com sucesso!
Scopus_Diplomacy_International_Relations_Technology_Transfer_2004_2026.xlsx


In [33]:
from google.colab import files

files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
# ============================================================
# BUSCA: TECHNOLOGY INTERNALIZATION
# 18 REVISTAS CONFIRMADAS NO SCOPUS
# ============================================================

# Usar as mesmas 18 revistas já confirmadas
confirmed_journals = scopus_sources_df[
    scopus_sources_df["Status"] == "FOUND IN SCOPUS"
].copy()

print(f"Revistas confirmadas no Scopus: {len(confirmed_journals)}")

# Criar a parte da consulta com os ISSNs
issn_queries = []

for _, row in confirmed_journals.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries.append(f'ISSN({issn})')

journals_part = " OR ".join(issn_queries)

# ============================================================
# TERMOS DA BUSCA
# ============================================================

internalization_terms = '''
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
'''

# ============================================================
# CONSULTA COMPLETA
# ============================================================

SEARCH_QUERY_INTERNALIZATION = f'''
(
    {journals_part}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{internalization_terms}
'''.replace("\n", " ").strip()

print("\n============================================================")
print("CONSULTA GERADA — TECHNOLOGY INTERNALIZATION")
print("============================================================")
print(SEARCH_QUERY_INTERNALIZATION)

print("\n============================================================")
print(f"Revistas confirmadas: {len(confirmed_journals)}")
print(f"18 revistas? {len(confirmed_journals) == 18}")
print("============================================================")

Revistas confirmadas no Scopus: 18

CONSULTA GERADA — TECHNOLOGY INTERNALIZATION
(     ISSN(0969-2290) OR ISSN(0020-8183) OR ISSN(0020-5850) OR ISSN(0015-7120) OR ISSN(0022-3433) OR ISSN(1045-5736) OR ISSN(0022-0027) OR ISSN(0703-6337) OR ISSN(0143-6597) OR ISSN(1521-9488) OR ISSN(0020-8833) OR ISSN(1465-0045) OR ISSN(1354-0661) OR ISSN(1758-5880) OR ISSN(0260-2105) OR ISSN(1559-7431) OR ISSN(1352-3260) OR ISSN(0951-2748) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" )

Revistas confirmadas: 18
18 revistas? True


In [35]:
# ============================================================
# EXECUTAR BUSCA NO SCOPUS
# TECHNOLOGY INTERNALIZATION
# ============================================================

params = {
    "query": SEARCH_QUERY_INTERNALIZATION,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_internalization = response.json()

    total_results_internalization = int(
        data_internalization["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_internalization}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [36]:
# ============================================================
# DATASET — TECHNOLOGY INTERNALIZATION
# ============================================================

# DataFrame vazio com as mesmas colunas do dataset anterior

df_internalization = pd.DataFrame(columns=[
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
])

print("Total de artigos:", len(df_internalization))

Total de artigos: 0


In [37]:
# ============================================================
# SEARCH LOG — TECHNOLOGY INTERNALIZATION
# ============================================================

search_log_internalization = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_journals),
        "; ".join(confirmed_journals["Journal"].tolist()),
        "2003–2027",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Internalization" OR "Internalization of Technology" OR "Technological Internalization"',
        "ISSN",
        0
    ]
})

display(search_log_internalization)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 00:41:20
2,Number of journals,18
3,Journals,Review of International Political Economy; Int...
4,Publication years,2003–2027
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Internalization"" OR ""Internalizati..."
8,Source identification,ISSN
9,Total results,0


In [38]:
# ============================================================
# EXPORTAR DATASET
# ============================================================

filename_internalization = (
    "Scopus_Diplomacy_International_Relations_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_internalization,
    engine="openpyxl"
) as writer:

    df_internalization.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_internalization.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado!")
print(filename_internalization)

✅ Excel criado!
Scopus_Diplomacy_International_Relations_Technology_Internalization_2004_2026.xlsx


In [39]:
from google.colab import files

files.download(filename_internalization)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
# ============================================================
# NOVO BLOCO — INTERNATIONAL LAW
# ============================================================

international_law_journals = {
    "German Law Journal": ["2071-8322"],

    "American Journal of International Law": [
        "0002-9300",
        "2161-7953"
    ],

    "Journal of International Economic Law": [
        "1369-3034",
        "1464-3758"
    ],

    "The International Journal of Human Rights": [
        "1364-2987",
        "1744-053X"
    ],

    "Business and Human Rights Journal": [
        "2057-0198",
        "2057-0201"
    ],

    "Common Market Law Review": [
        "0165-0750",
        "1875-8320"
    ],

    "European Journal of International Law": [
        "0938-5428",
        "1468-0386"
    ],

    "International and Comparative Law Quarterly": [
        "0020-5893",
        "1471-6895"
    ],

    "International Journal of Constitutional Law": [
        "1474-2640",
        "1474-2640"
    ],

    "IIC-International Review of Intellectual Property and Competition Law": [
        "0018-9855",
        "2195-0237"
    ],

    "World Trade Review": [
        "1474-7456",
        "1475-3138"
    ],

    "Human Rights Law Review": [
        "1461-7781",
        "1744-1021"
    ],

    "Journal of Human Rights": [
        "1475-4835",
        "1475-4843"
    ],

    "ERA Forum": [
        "1612-3093",
        "1863-9038"
    ],

    "Leiden Journal of International Law": [
        "0922-1565",
        "1478-9698"
    ],

    "American Journal of International Law Unbound": [
        "2398-7723"
    ],

    "Global Constitutionalism": [
        "2045-3817",
        "2045-3825"
    ]
}

print(
    f"{len(international_law_journals)} revistas "
    "foram cadastradas."
)

17 revistas foram cadastradas.


In [41]:
# ============================================================
# VERIFICAR INTERNATIONAL LAW NO SCOPUS
# ============================================================

scopus_international_law = []

for journal, issns in international_law_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_international_law.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if not found:

        scopus_international_law.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")

    else:

        print(f"✅ Encontrado: {journal}")

✅ Encontrado: German Law Journal
✅ Encontrado: American Journal of International Law
✅ Encontrado: Journal of International Economic Law
✅ Encontrado: The International Journal of Human Rights
✅ Encontrado: Business and Human Rights Journal
✅ Encontrado: Common Market Law Review
✅ Encontrado: European Journal of International Law
✅ Encontrado: International and Comparative Law Quarterly
✅ Encontrado: International Journal of Constitutional Law
✅ Encontrado: IIC-International Review of Intellectual Property and Competition Law
✅ Encontrado: World Trade Review
✅ Encontrado: Human Rights Law Review
✅ Encontrado: Journal of Human Rights
✅ Encontrado: ERA Forum
✅ Encontrado: Leiden Journal of International Law
✅ Encontrado: American Journal of International Law Unbound
✅ Encontrado: Global Constitutionalism


In [42]:
# ============================================================
# RESULTADO DA VERIFICAÇÃO
# ============================================================

scopus_international_law_df = pd.DataFrame(
    scopus_international_law
)

display(scopus_international_law_df)

,Journal,ISSN searched,Scopus title,Source ID,Publisher,Status
0,German Law Journal,2071-8322,German Law Journal,21101023020,Cambridge University Press,FOUND IN SCOPUS
1,American Journal of International Law,0002-9300,American Journal of International Law,25050,Cambridge University Press,FOUND IN SCOPUS
2,Journal of International Economic Law,1369-3034,Journal of International Economic Law,26836,Oxford University Press,FOUND IN SCOPUS
3,The International Journal of Human Rights,1364-2987,International Journal of Human Rights,5600155017,Routledge,FOUND IN SCOPUS
4,Business and Human Rights Journal,2057-0198,Business and Human Rights Journal,21100830164,Cambridge University Press,FOUND IN SCOPUS
5,Common Market Law Review,0165-0750,Common Market Law Review,25666,Kluwer Law International,FOUND IN SCOPUS
6,European Journal of International Law,0938-5428,European Journal of International Law,145598,Oxford University Press,FOUND IN SCOPUS
7,International and Comparative Law Quarterly,0020-5893,International and Comparative Law Quarterly,26339,Cambridge University Press,FOUND IN SCOPUS
8,International Journal of Constitutional Law,1474-2640,International Journal of Constitutional Law,4700152463,Oxford University Press,FOUND IN SCOPUS
9,IIC-International Review of Intellectual Prope...,0018-9855,IIC International Review of Intellectual Prope...,39526,Springer Science and Business Media Deutschlan...,FOUND IN SCOPUS


In [43]:
# ============================================================
# BUSCA — INTERNATIONAL LAW
# TECHNOLOGY TRANSFER
# ============================================================

confirmed_international_law = scopus_international_law_df[
    scopus_international_law_df["Status"] == "FOUND IN SCOPUS"
].copy()

print(
    f"Revistas confirmadas no Scopus: "
    f"{len(confirmed_international_law)}"
)

# ISSNs confirmados
issn_queries = []

for _, row in confirmed_international_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries.append(f'ISSN({issn})')

journals_part_international_law = " OR ".join(
    issn_queries
)

# Termos
technology_transfer_terms = '''
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
'''

# Consulta completa
SEARCH_QUERY_IL_TT = f'''
(
    {journals_part_international_law}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{technology_transfer_terms}
'''.replace("\n", " ").strip()

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_IL_TT)

print("\n============================================================")
print(
    f"17 revistas? "
    f"{len(confirmed_international_law) == 17}"
)
print("============================================================")

Revistas confirmadas no Scopus: 17

CONSULTA GERADA
(     ISSN(2071-8322) OR ISSN(0002-9300) OR ISSN(1369-3034) OR ISSN(1364-2987) OR ISSN(2057-0198) OR ISSN(0165-0750) OR ISSN(0938-5428) OR ISSN(0020-5893) OR ISSN(1474-2640) OR ISSN(0018-9855) OR ISSN(1474-7456) OR ISSN(1461-7781) OR ISSN(1475-4835) OR ISSN(1612-3093) OR ISSN(0922-1565) OR ISSN(2398-7723) OR ISSN(2045-3817) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" )

17 revistas? True


In [44]:
# ============================================================
# EXECUTAR BUSCA NO SCOPUS
# ============================================================

params = {
    "query": SEARCH_QUERY_IL_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_il_tt = response.json()

    total_results_il_tt = int(
        data_il_tt["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_il_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 16


In [45]:
# ============================================================
# BAIXAR OS RESULTADOS — INTERNATIONAL LAW / TECHNOLOGY TRANSFER
# ============================================================

all_results_il_tt = []

count = 25
start = 0

while start < total_results_il_tt:

    params = {
        "query": SEARCH_QUERY_IL_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data["search-results"].get("entry", [])

    all_results_il_tt.extend(entries)

    print(
        f"Baixados: {len(all_results_il_tt)} / "
        f"{total_results_il_tt}"
    )

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(
    f"Total de registros baixados: "
    f"{len(all_results_il_tt)}"
)

Baixados: 16 / 16

✅ Busca concluída!
Total de registros baixados: 16


In [46]:
# ============================================================
# ORGANIZAR OS 16 RESULTADOS
# INTERNATIONAL LAW / TECHNOLOGY TRANSFER
# ============================================================

records_il_tt = []

for item in all_results_il_tt:

    # ISSN
    issn = item.get("prism:issn", "")

    # Periódico
    journal_name = item.get(
        "prism:publicationName",
        ""
    )

    # Data / ano
    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    # DOI
    doi = item.get("prism:doi", "")

    # Autores
    authors = item.get("dc:creator", "")

    # URL do Scopus
    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_il_tt.append({
        "Journal": journal_name,
        "Title": item.get("dc:title", ""),
        "Authors": authors,
        "Year": year,
        "ISSN": issn,
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": doi,
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_il_tt = pd.DataFrame(records_il_tt)

print("✅ Tabela criada!")
print(f"Total de artigos: {len(df_il_tt)}")

display(df_il_tt)

✅ Tabela criada!
Total de artigos: 16


,Journal,Title,Authors,Year,ISSN,Volume,Issue,Pages,DOI,EID,Cited by,Scopus URL
0,Journal of International Economic Law,A co-regulatory approach to bilateral trade: a...,Tian G.Y.,2026,13693034,29,2,264-291,10.1093/jiel/jgag019,2-s2.0-105045424288,0,https://www.scopus.com/inward/record.uri?partn...
1,International Journal of Human Rights,Not fit for purpose: the limits of the UN guid...,Vijeyarasa R.,2026,13642987,,,None,10.1080/13642987.2026.2680040,2-s2.0-105041115422,0,https://www.scopus.com/inward/record.uri?partn...
2,Journal of International Economic Law,Aligning climate needs and intellectual proper...,Tundang R.E.,2025,13693034,28,3,441-459,10.1093/jiel/jgaf030,2-s2.0-105021324944,1,https://www.scopus.com/inward/record.uri?partn...
3,World Trade Review,MRNA Technology Transfer Hub and Intellectual ...,Adekola T.A.,2025,14747456,24,2,302-320,10.1017/S147474562400020X,2-s2.0-85216848973,7,https://www.scopus.com/inward/record.uri?partn...
4,American Journal of International Law,Four Treaties in One: The Biodiversity Beyond ...,Bodansky D.,2024,00029300,118,2,299-323,10.1017/ajil.2024.9,2-s2.0-85187706927,35,https://www.scopus.com/inward/record.uri?partn...
5,German Law Journal,The Biden Administration's Trade Policy: Promi...,Schoenbaum T.J.,2023,,24,1,102-124,10.1017/glj.2023.7,2-s2.0-85151453140,24,https://www.scopus.com/inward/record.uri?partn...
6,IIC International Review of Intellectual Prope...,It’s Unfair! Non-price Exploitation in ICT Pat...,Galli N.,2023,00189855,54,2,200-222,10.1007/s40319-023-01287-x,2-s2.0-85147939099,0,https://www.scopus.com/inward/record.uri?partn...
7,World Trade Review,The WTO at the Crossroads: How to Avoid the Ch...,Mavroidis P.C.,2022,14747456,21,3,359-366,10.1017/S147474562200009X,2-s2.0-85127867272,4,https://www.scopus.com/inward/record.uri?partn...
8,Journal of International Economic Law,Secrecy by Default: How Regional Trade Agreeme...,Słok-Wódkowska M.,2022,13693034,25,1,91-109,10.1093/jiel/jgac005,2-s2.0-85128139153,9,https://www.scopus.com/inward/record.uri?partn...
9,Journal of International Economic Law,Making the WTO (Not So) great again: The case ...,Howse R.,2020,13693034,23,2,371-389,10.1093/jiel/jgaa017,2-s2.0-85097195408,21,https://www.scopus.com/inward/record.uri?partn...


In [47]:
# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_il_tt = (
    df_il_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts_il_tt)

,Journal,Number of articles
0,Journal of International Economic Law,6
1,IIC International Review of Intellectual Prope...,4
2,World Trade Review,2
3,International Journal of Human Rights,1
4,American Journal of International Law,1
5,German Law Journal,1
6,Leiden Journal of International Law,1


In [48]:
# ============================================================
# SEARCH LOG
# ============================================================

search_log_il_tt = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_international_law),
        "; ".join(
            confirmed_international_law["Journal"].tolist()
        ),
        "2003–2027",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Transfer" OR "Transfer of Technology" OR "Technological Transfer"',
        "ISSN",
        len(df_il_tt)
    ]
})

display(search_log_il_tt)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 00:41:28
2,Number of journals,17
3,Journals,German Law Journal; American Journal of Intern...
4,Publication years,2003–2027
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Transfer"" OR ""Transfer of Technolo..."
8,Source identification,ISSN
9,Total results,16


In [49]:
# ============================================================
# EXPORTAR DATASET
# ============================================================

filename_il_tt = (
    "Scopus_International_Law_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_il_tt,
    engine="openpyxl"
) as writer:

    df_il_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_il_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_il_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename_il_tt)

✅ Excel criado com sucesso!
Scopus_International_Law_Technology_Transfer_2004_2026.xlsx


In [50]:
from google.colab import files

files.download(filename_il_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
# ============================================================
# BUSCA — INTERNATIONAL LAW
# TECHNOLOGY INTERNALIZATION
# ============================================================

# Usar as 17 revistas já confirmadas
confirmed_international_law = scopus_international_law_df[
    scopus_international_law_df["Status"] == "FOUND IN SCOPUS"
].copy()

print(
    f"Revistas confirmadas no Scopus: "
    f"{len(confirmed_international_law)}"
)

# ISSNs confirmados
issn_queries = []

for _, row in confirmed_international_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries.append(f'ISSN({issn})')

journals_part_international_law = " OR ".join(
    issn_queries
)

# ============================================================
# TERMOS DA BUSCA
# ============================================================

internalization_terms_il = '''
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
'''

# ============================================================
# CONSULTA COMPLETA
# ============================================================

SEARCH_QUERY_IL_TI = f'''
(
    {journals_part_international_law}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{internalization_terms_il}
'''.replace("\n", " ").strip()

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_IL_TI)

print("\n============================================================")
print(
    f"17 revistas? "
    f"{len(confirmed_international_law) == 17}"
)
print("============================================================")

Revistas confirmadas no Scopus: 17

CONSULTA GERADA
(     ISSN(2071-8322) OR ISSN(0002-9300) OR ISSN(1369-3034) OR ISSN(1364-2987) OR ISSN(2057-0198) OR ISSN(0165-0750) OR ISSN(0938-5428) OR ISSN(0020-5893) OR ISSN(1474-2640) OR ISSN(0018-9855) OR ISSN(1474-7456) OR ISSN(1461-7781) OR ISSN(1475-4835) OR ISSN(1612-3093) OR ISSN(0922-1565) OR ISSN(2398-7723) OR ISSN(2045-3817) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" )

17 revistas? True


In [52]:
# ============================================================
# EXECUTAR BUSCA NO SCOPUS
# ============================================================

params = {
    "query": SEARCH_QUERY_IL_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_il_ti = response.json()

    total_results_il_ti = int(
        data_il_ti["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_il_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [53]:
# ============================================================
# DATASET — INTERNATIONAL LAW / TECHNOLOGY INTERNALIZATION
# ============================================================

df_il_ti = pd.DataFrame(columns=[
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
])

print("Total de artigos:", len(df_il_ti))

Total de artigos: 0


In [54]:
# ============================================================
# SEARCH LOG
# INTERNATIONAL LAW / TECHNOLOGY INTERNALIZATION
# ============================================================

search_log_il_ti = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_international_law),
        "; ".join(
            confirmed_international_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Internalization" OR "Internalization of Technology" OR "Technological Internalization"',
        "ISSN",
        0
    ]
})

display(search_log_il_ti)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 00:41:29
2,Number of journals,17
3,Journals,German Law Journal; American Journal of Intern...
4,Publication years,2004–2026
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Internalization"" OR ""Internalizati..."
8,Source identification,ISSN
9,Total results,0


In [55]:
# ============================================================
# EXPORTAR DATASET
# ============================================================

filename_il_ti = (
    "Scopus_International_Law_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_il_ti,
    engine="openpyxl"
) as writer:

    df_il_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_il_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename_il_ti)

✅ Excel criado com sucesso!
Scopus_International_Law_Technology_Internalization_2004_2026.xlsx


In [56]:
from google.colab import files

files.download(filename_il_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
# ============================================================
# NOVO BLOCO — TECHNOLOGY LAW
# ============================================================

technology_law_journals = {
    "Computer Law & Security Review": [
        "0267-3649",
        "1873-5351"
    ],

    "Internet Policy Review": [
        "2197-6775"
    ],

    "Berkeley Technology Law Journal": [
        "1086-4349",
        "1933-7721"
    ],

    "World Patent Information": [
        "0172-2190",
        "1874-748X"
    ],

    "International Review of Law, Computers & Technology": [
        "1360-0869",
        "1360-0877"
    ],

    "International Journal of Law and Information Technology": [
        "0967-0769",
        "1464-036X"
    ],

    "Vanderbilt Journal of Entertainment and Technology Law": [
        "1942-242X"
    ],

    "Journal of Intellectual Property Law & Practice": [
        "1747-1532",
        "1747-1540"
    ],

    "GRUR International": [
        "2632-8623",
        "2632-8615"
    ],

    "The Journal of World Intellectual Property": [
        "1422-2213",
        "1422-1357"
    ],

    "Journal of Antitrust Enforcement": [
        "2050-0693",
        "2050-0707"
    ],

    "European Competition Journal": [
        "1757-8396",
        "1757-840X"
    ],

    "Columbia Journal of Law & the Arts": [
        "0734-9149",
        "2161-9271"
    ],

    "Digital Law Journal": [
        "2686-9136",
        "2686-9144"
    ],

    "Journal of Intellectual Property, Information Technology & Electronic Communication Law": [
        "2190-3383",
        "2190-3391"
    ],

    "Cybersecurity and Law": [
        "2544-9945"
    ]
}

print(
    f"{len(technology_law_journals)} revistas "
    "foram cadastradas."
)

16 revistas foram cadastradas.


In [58]:
# ============================================================
# VERIFICAR TECHNOLOGY LAW NO SCOPUS
# ============================================================

scopus_technology_law = []

for journal, issns in technology_law_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_technology_law.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if not found:

        scopus_technology_law.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")

    else:

        print(f"✅ Encontrado: {journal}")

⚠️ Não encontrado: Computer Law & Security Review
✅ Encontrado: Internet Policy Review
⚠️ Não encontrado: Berkeley Technology Law Journal
✅ Encontrado: World Patent Information
✅ Encontrado: International Review of Law, Computers & Technology
✅ Encontrado: International Journal of Law and Information Technology
⚠️ Não encontrado: Vanderbilt Journal of Entertainment and Technology Law
✅ Encontrado: Journal of Intellectual Property Law & Practice
✅ Encontrado: GRUR International
✅ Encontrado: The Journal of World Intellectual Property
⚠️ Não encontrado: Journal of Antitrust Enforcement
✅ Encontrado: European Competition Journal
✅ Encontrado: Columbia Journal of Law & the Arts
⚠️ Não encontrado: Digital Law Journal
⚠️ Não encontrado: Journal of Intellectual Property, Information Technology & Electronic Communication Law
⚠️ Não encontrado: Cybersecurity and Law


In [59]:
# ============================================================
# RESULTADO DA VERIFICAÇÃO
# ============================================================

scopus_technology_law_df = pd.DataFrame(
    scopus_technology_law
)

display(scopus_technology_law_df)

,Journal,ISSN searched,Scopus title,Source ID,Publisher,Status
0,Computer Law & Security Review,,,,,NOT FOUND
1,Internet Policy Review,2197-6775,Internet Policy Review,21100898535,Alexander von Humboldt Institute for Internet ...,FOUND IN SCOPUS
2,Berkeley Technology Law Journal,,,,,NOT FOUND
3,World Patent Information,0172-2190,World Patent Information,15069,Elsevier Ltd,FOUND IN SCOPUS
4,"International Review of Law, Computers & Techn...",1360-0869,"International Review of Law, Computers and Tec...",18100156704,Routledge,FOUND IN SCOPUS
5,International Journal of Law and Information T...,0967-0769,International Journal of Law and Information T...,145600,Oxford University Press,FOUND IN SCOPUS
6,Vanderbilt Journal of Entertainment and Techno...,,,,,NOT FOUND
7,Journal of Intellectual Property Law & Practice,1747-1532,Journal of Intellectual Property Law and Practice,21100937573,Oxford University Press,FOUND IN SCOPUS
8,GRUR International,2632-8623,GRUR International,21101144867,Oxford University Press,FOUND IN SCOPUS
9,The Journal of World Intellectual Property,1422-2213,Journal of World Intellectual Property,21100316000,John Wiley and Sons Ltd,FOUND IN SCOPUS


In [60]:
technology_law_journals = {

    "Computer Law & Security Review": [
        "2212-473X",
        "2212-4748"
    ],

    "Internet Policy Review": [
        "2197-6775"
    ],

    "Berkeley Technology Law Journal": [
        "1086-3818",
        "2380-4742"
    ],

    "World Patent Information": [
        "0172-2190",
        "1874-690X"
    ],

    "International Review of Law, Computers & Technology": [
        "1360-0869",
        "1360-0877"
    ],

    "International Journal of Law and Information Technology": [
        "0967-0769",
        "1464-036X"
    ],

    "Vanderbilt Journal of Entertainment and Technology Law": [
        "1942-678X",
        "1942-6771"
    ],

    "Journal of Intellectual Property Law & Practice": [
        "1747-1532",
        "1747-1540"
    ],

    "GRUR International": [
        "2632-8623",
        "2632-8615"
    ],

    "The Journal of World Intellectual Property": [
        "1422-2213",
        "1422-1357"
    ],

    "Journal of Antitrust Enforcement": [
        "2050-0688",
        "2050-0696"
    ],

    "European Competition Journal": [
        "1757-8396",
        "1757-840X"
    ],

    "Columbia Journal of Law & the Arts": [
        "0734-9149",
        "2161-9271"
    ],

    "Digital Law Journal": [
        "2686-9136"
    ],

    "Journal of Intellectual Property, Information Technology and E-Commerce Law": [
    "2190-3387"
],

    "Cybersecurity and Law": [
        "2658-1493",
        "3072-0532"
    ]
}

In [61]:
# ============================================================
# BUSCA — TECHNOLOGY LAW
# TECHNOLOGY TRANSFER
# ============================================================

# Apenas revistas confirmadas no Scopus
confirmed_technology_law = scopus_technology_law_df[
    scopus_technology_law_df["Status"] == "FOUND IN SCOPUS"
].copy()

print(
    f"Revistas confirmadas no Scopus: "
    f"{len(confirmed_technology_law)}"
)

# ISSNs confirmados
issn_queries_tl = []

for _, row in confirmed_technology_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_tl.append(f'ISSN({issn})')

journals_part_technology_law = " OR ".join(
    issn_queries_tl
)

# ============================================================
# TERMOS DA BUSCA
# ============================================================

technology_transfer_terms_tl = '''
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
'''

# ============================================================
# CONSULTA COMPLETA
# ============================================================

SEARCH_QUERY_TL_TT = f'''
(
    {journals_part_technology_law}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{technology_transfer_terms_tl}
'''.replace("\n", " ").strip()

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_TL_TT)

print("\n============================================================")
print(
    f"Revistas utilizadas: "
    f"{len(confirmed_technology_law)}"
)
print("============================================================")

Revistas confirmadas no Scopus: 9

CONSULTA GERADA
(     ISSN(2197-6775) OR ISSN(0172-2190) OR ISSN(1360-0869) OR ISSN(0967-0769) OR ISSN(1747-1532) OR ISSN(2632-8623) OR ISSN(1422-2213) OR ISSN(1757-8396) OR ISSN(0734-9149) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" )

Revistas utilizadas: 9


In [62]:
# ============================================================
# EXECUTAR BUSCA NO SCOPUS
# ============================================================

params = {
    "query": SEARCH_QUERY_TL_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_tl_tt = response.json()

    total_results_tl_tt = int(
        data_tl_tt["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_tl_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 34


In [63]:
# ============================================================
# BAIXAR OS RESULTADOS — TECHNOLOGY LAW / TECHNOLOGY TRANSFER
# ============================================================

all_results_tl_tt = []

count = 25
start = 0

while start < total_results_tl_tt:

    params = {
        "query": SEARCH_QUERY_TL_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data["search-results"].get("entry", [])

    all_results_tl_tt.extend(entries)

    print(
        f"Baixados: {len(all_results_tl_tt)} / "
        f"{total_results_tl_tt}"
    )

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(
    f"Total de registros baixados: "
    f"{len(all_results_tl_tt)}"
)

Baixados: 25 / 34
Baixados: 34 / 34

✅ Busca concluída!
Total de registros baixados: 34


In [64]:
# ============================================================
# ORGANIZAR OS 36 RESULTADOS
# TECHNOLOGY LAW / TECHNOLOGY TRANSFER
# ============================================================

records_tl_tt = []

for item in all_results_tl_tt:

    issn = item.get("prism:issn", "")

    journal_name = item.get(
        "prism:publicationName",
        ""
    )

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    doi = item.get("prism:doi", "")

    authors = item.get("dc:creator", "")

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_tl_tt.append({
        "Journal": journal_name,
        "Title": item.get("dc:title", ""),
        "Authors": authors,
        "Year": year,
        "ISSN": issn,
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": doi,
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_tl_tt = pd.DataFrame(records_tl_tt)

print("✅ Tabela criada!")
print(f"Total de artigos: {len(df_tl_tt)}")

display(df_tl_tt)

✅ Tabela criada!
Total de artigos: 34


,Journal,Title,Authors,Year,ISSN,Volume,Issue,Pages,DOI,EID,Cited by,Scopus URL
0,World Patent Information,The distribution of patent impact in federal r...,Marinakis Y.D.,2026,01722190,85,,None,10.1016/j.wpi.2026.102459,2-s2.0-105035874856,0,https://www.scopus.com/inward/record.uri?partn...
1,Grur International,Breeding the Winning Unicorn: Spin-Offs and Te...,Piselli R.,2026,26328623,75,4,329-339,10.1093/grurint/ikag005,2-s2.0-105039384218,0,https://www.scopus.com/inward/record.uri?partn...
2,Journal of World Intellectual Property,Catalyzing circularity: Tackling intellectual ...,Choudhary L.,2026,14222213,29,1,177-200,10.1111/jwip.70006,2-s2.0-105017810499,0,https://www.scopus.com/inward/record.uri?partn...
3,Journal of World Intellectual Property,Determining an appropriate patent working requ...,Hawin M.,2026,14222213,29,1,101-113,10.1111/jwip.70003,2-s2.0-105010450288,0,https://www.scopus.com/inward/record.uri?partn...
4,Grur International,Regulation of Technology Transfer in Brazil: A...,Chamas C.,2026,26328623,75,2,163-171,10.1093/grurint/ikaf112,2-s2.0-105028882518,0,https://www.scopus.com/inward/record.uri?partn...
5,European Competition Journal,Licensing negotiation groups: the new antitrus...,Colangelo G.,2026,17441056,,,None,10.1080/17441056.2026.2683221,2-s2.0-105041040961,0,https://www.scopus.com/inward/record.uri?partn...
6,World Patent Information,Technological and patent landscape of biosenso...,Otálora Trujillo M.C.,2025,01722190,83,,None,10.1016/j.wpi.2025.102405,2-s2.0-105022207681,3,https://www.scopus.com/inward/record.uri?partn...
7,Journal of World Intellectual Property,The problems with the nature of the obligation...,Kolawole O.I.,2025,14222213,28,3,762-782,10.1111/jwip.12354,2-s2.0-86000658472,0,https://www.scopus.com/inward/record.uri?partn...
8,Grur International,Position Statement of the Max Planck Institute...,Drexl J.,2025,26328623,74,8,736-750,10.1093/grurint/ikaf088,2-s2.0-105015448243,0,https://www.scopus.com/inward/record.uri?partn...
9,Journal of World Intellectual Property,The flexibilization of intellectual property r...,Varella M.D.,2024,14222213,27,3,563-575,10.1111/jwip.12317,2-s2.0-85197426164,0,https://www.scopus.com/inward/record.uri?partn...


In [65]:
# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_tl_tt = (
    df_tl_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts_tl_tt)

,Journal,Number of articles
0,World Patent Information,16
1,Journal of World Intellectual Property,10
2,Grur International,7
3,European Competition Journal,1


In [66]:
# ============================================================
# SEARCH LOG
# TECHNOLOGY LAW / TECHNOLOGY TRANSFER
# ============================================================

search_log_tl_tt = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_technology_law),
        "; ".join(
            confirmed_technology_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Transfer" OR "Transfer of Technology" OR "Technological Transfer"',
        "ISSN",
        len(df_tl_tt)
    ]
})

display(search_log_tl_tt)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 00:41:42
2,Number of journals,9
3,Journals,Internet Policy Review; World Patent Informati...
4,Publication years,2004–2026
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Transfer"" OR ""Transfer of Technolo..."
8,Source identification,ISSN
9,Total results,34


In [67]:
# ============================================================
# EXPORTAR DATASET
# ============================================================

filename_tl_tt = (
    "Scopus_Technology_Law_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_tl_tt,
    engine="openpyxl"
) as writer:

    df_tl_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_tl_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_tl_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename_tl_tt)

✅ Excel criado com sucesso!
Scopus_Technology_Law_Technology_Transfer_2004_2026.xlsx


In [68]:
from google.colab import files

files.download(filename_tl_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [69]:
# ============================================================
# BUSCA — TECHNOLOGY LAW
# TECHNOLOGY INTERNALIZATION
# ============================================================

confirmed_technology_law = scopus_technology_law_df[
    scopus_technology_law_df["Status"] == "FOUND IN SCOPUS"
].copy()

print(
    f"Revistas confirmadas no Scopus: "
    f"{len(confirmed_technology_law)}"
)

# ISSNs confirmados
issn_queries_tl_ti = []

for _, row in confirmed_technology_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_tl_ti.append(f'ISSN({issn})')

journals_part_technology_law_ti = " OR ".join(
    issn_queries_tl_ti
)

# ============================================================
# TERMOS DA BUSCA
# ============================================================

technology_internalization_terms_tl = '''
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
'''

# ============================================================
# CONSULTA COMPLETA
# ============================================================

SEARCH_QUERY_TL_TI = f'''
(
    {journals_part_technology_law_ti}
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
AND
{technology_internalization_terms_tl}
'''.replace("\n", " ").strip()

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_TL_TI)

print("\n============================================================")
print(
    f"Revistas utilizadas: "
    f"{len(confirmed_technology_law)}"
)
print("============================================================")

Revistas confirmadas no Scopus: 9

CONSULTA GERADA
(     ISSN(2197-6775) OR ISSN(0172-2190) OR ISSN(1360-0869) OR ISSN(0967-0769) OR ISSN(1747-1532) OR ISSN(2632-8623) OR ISSN(1422-2213) OR ISSN(1757-8396) OR ISSN(0734-9149) ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar) AND  TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" )

Revistas utilizadas: 9


In [70]:
# ============================================================
# EXECUTAR BUSCA NO SCOPUS
# ============================================================

params = {
    "query": SEARCH_QUERY_TL_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_tl_ti = response.json()

    total_results_tl_ti = int(
        data_tl_ti["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_tl_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [71]:
# ============================================================
# NOVO BLOCO — AFRICAN LAW JOURNALS
# ============================================================

african_law_journals = {

    "South African Journal of Bioethics and Law": [
        "1999-7639"
    ],

    "African Human Mobility Review": [
        "2411-6955",
        "2410-7972"
    ],

    "African Human Rights Law Journal": [
        "1609-073X",
        "1996-2096"
    ],

    "Journal of Sustainable Development Law and Policy": [
        "2467-8406",
        "2467-8392"
    ],

    "Potchefstroom Electronic Law Journal": [
        "1727-3781"
    ],

    "South African Law Journal": [
        "0258-2503",
        "1996-2177"
    ],

    "Mizan Law Review": [
        "1998-9881",
        "2309-902X"
    ],

    "Fundamina": [
        "1021-545X",
        "2411-7870"
    ],

    "Tydskrif vir die Suid-Afrikaanse Reg": [
        "0257-7747",
        "1996-2207"
    ],

    "African Human Rights Yearbook": [
        "2523-1367",
        "2663-323X"
    ]
}

print(
    f"{len(african_law_journals)} periódicos cadastrados."
)

10 periódicos cadastrados.


In [72]:
# ============================================================
# VERIFICAR AFRICAN LAW JOURNALS NO SCOPUS
# ============================================================

scopus_african_law = []

for journal, issns in african_law_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_african_law.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_african_law.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")

✅ Encontrado: South African Journal of Bioethics and Law
✅ Encontrado: African Human Mobility Review
✅ Encontrado: African Human Rights Law Journal
✅ Encontrado: Journal of Sustainable Development Law and Policy
✅ Encontrado: Potchefstroom Electronic Law Journal
✅ Encontrado: South African Law Journal
✅ Encontrado: Mizan Law Review
✅ Encontrado: Fundamina
✅ Encontrado: Tydskrif vir die Suid-Afrikaanse Reg
✅ Encontrado: African Human Rights Yearbook


In [73]:
# ============================================================
# RESULTADO DA VERIFICAÇÃO
# ============================================================

scopus_african_law_df = pd.DataFrame(
    scopus_african_law
)

display(scopus_african_law_df)

print(
    "\nConfirmados no Scopus:",
    (
        scopus_african_law_df["Status"]
        == "FOUND IN SCOPUS"
    ).sum(),
    "/",
    len(scopus_african_law_df)
)

,Journal,ISSN searched,Scopus title,Source ID,Publisher,Status
0,South African Journal of Bioethics and Law,1999-7639,South African Journal of Bioethics and Law,21101040698,South African Medical Association,FOUND IN SCOPUS
1,African Human Mobility Review,2411-6955,African Human Mobility Review,21101154265,University of the Western Cape,FOUND IN SCOPUS
2,African Human Rights Law Journal,1609-073X,African Human Rights Law Journal,21101037907,Pretoria University Law Press,FOUND IN SCOPUS
3,Journal of Sustainable Development Law and Policy,2467-8406,Journal of Sustainable Development Law and Policy,21101189040,"Institute for Oil, Gas, Environment, Energy an...",FOUND IN SCOPUS
4,Potchefstroom Electronic Law Journal,1727-3781,Potchefstroom Electronic Law Journal,21100797848,North-West Unversity,FOUND IN SCOPUS
5,South African Law Journal,0258-2503,South African Law Journal,19157,Juta and Company Ltd,FOUND IN SCOPUS
6,Mizan Law Review,1998-9881,Mizan Law Review,21101124071,St Mary's University,FOUND IN SCOPUS
7,Fundamina,1021-545X,Fundamina,21101224866,Juta and Company Ltd,FOUND IN SCOPUS
8,Tydskrif vir die Suid-Afrikaanse Reg,0257-7747,Tydskrif vir die Suid-Afrikaanse Reg,19900191877,Juta and Company Ltd,FOUND IN SCOPUS
9,African Human Rights Yearbook,2523-1367,African Human Rights Yearbook,21101183493,Pretoria University Law Press,FOUND IN SCOPUS



Confirmados no Scopus: 10 / 10


In [77]:
# ============================================================
# CRIAR LISTA DE ISSNs DOS PERIÓDICOS CONFIRMADOS NO SCOPUS
# ============================================================

confirmed_african_law = scopus_african_law_df[
    scopus_african_law_df["Status"] == "FOUND IN SCOPUS"
].copy()

issn_queries_african_law = []

for _, row in confirmed_african_law.iterrows():
    issn = row["ISSN searched"]

    if issn:
        issn_queries_african_law.append(
            f'ISSN({issn})'
        )

journals_part_african_law = " OR ".join(
    issn_queries_african_law
)

print(
    f"Periódicos confirmados no Scopus: "
    f"{len(confirmed_african_law)}"
)

print("\nParte da consulta referente aos periódicos:")
print(journals_part_african_law)

Periódicos confirmados no Scopus: 10

Parte da consulta referente aos periódicos:
ISSN(1999-7639) OR ISSN(2411-6955) OR ISSN(1609-073X) OR ISSN(2467-8406) OR ISSN(1727-3781) OR ISSN(0258-2503) OR ISSN(1998-9881) OR ISSN(1021-545X) OR ISSN(0257-7747) OR ISSN(2523-1367)


In [78]:
# ============================================================
# AFRICAN LAW — TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

SEARCH_QUERY_AFRICA_TT = f'''
(
    {journals_part_african_law}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_AFRICA_TT)

CONSULTA GERADA
(     ISSN(1999-7639) OR ISSN(2411-6955) OR ISSN(1609-073X) OR ISSN(2467-8406) OR ISSN(1727-3781) OR ISSN(0258-2503) OR ISSN(1998-9881) OR ISSN(1021-545X) OR ISSN(0257-7747) OR ISSN(2523-1367) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)


In [79]:
# ============================================================
# EXECUTAR CONSULTA NO SCOPUS
# ============================================================

params = {
    "query": SEARCH_QUERY_AFRICA_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_africa_tt = response.json()

    total_results_africa_tt = int(
        data_africa_tt["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_africa_tt}"
    )

else:
    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 3


In [80]:
# ============================================================
# BAIXAR RESULTADOS — AFRICAN LAW / TECHNOLOGY TRANSFER
# ============================================================

all_results_africa_tt = []

count = 25
start = 0

while start < total_results_africa_tt:

    params = {
        "query": SEARCH_QUERY_AFRICA_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data["search-results"].get("entry", [])

    all_results_africa_tt.extend(entries)

    print(
        f"Baixados: {len(all_results_africa_tt)} / "
        f"{total_results_africa_tt}"
    )

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(
    f"Total de registros baixados: "
    f"{len(all_results_africa_tt)}"
)

Baixados: 3 / 3

✅ Busca concluída!
Total de registros baixados: 3


In [81]:
# ============================================================
# ORGANIZAR RESULTADOS
# AFRICAN LAW / TECHNOLOGY TRANSFER
# ============================================================

records_africa_tt = []

for item in all_results_africa_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_africa_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_africa_tt = pd.DataFrame(records_africa_tt)

print("✅ Tabela criada!")
print(f"Total de artigos: {len(df_africa_tt)}")

display(df_africa_tt)

✅ Tabela criada!
Total de artigos: 3


,Journal,Title,Authors,Year,ISSN,Volume,Issue,Pages,DOI,EID,Cited by,Scopus URL
0,African Human Mobility Review,Human Capital Flight: The Japa Phenomenon and ...,Bashir A.M.,2026,24116955,12,2,107-126,10.14426/7g6bjs83,2-s2.0-105051326449,0,https://www.scopus.com/inward/record.uri?partn...
1,Mizan Law Review,The Room for Imposing Performance Requirements...,Dejene Y.M.,2022,19989881,16,2,391-422,10.4314/mlr.v16i2.6,2-s2.0-85148239316,0,https://www.scopus.com/inward/record.uri?partn...
2,Potchefstroom Electronic Law Journal,Transboundary movements of genetically modifie...,Lim Tung O.J.,2014,,17,5,1739-1787,10.17159/1727-3781/2014/v17i5a2153,2-s2.0-85032266597,6,https://www.scopus.com/inward/record.uri?partn...


In [82]:
# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_africa_tt = (
    df_africa_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts_africa_tt)

,Journal,Number of articles
0,African Human Mobility Review,1
1,Mizan Law Review,1
2,Potchefstroom Electronic Law Journal,1


In [83]:
# ============================================================
# SEARCH LOG
# ============================================================

search_log_africa_tt = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_african_law),
        "; ".join(
            confirmed_african_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Transfer" OR "Transfer of Technology" OR "Technological Transfer"',
        "ISSN",
        len(df_africa_tt)
    ]
})

display(search_log_africa_tt)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 00:55:05
2,Number of journals,10
3,Journals,South African Journal of Bioethics and Law; Af...
4,Publication years,2004–2026
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Transfer"" OR ""Transfer of Technolo..."
8,Source identification,ISSN
9,Total results,3


In [84]:
# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_africa_tt = (
    "Scopus_African_Law_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_africa_tt,
    engine="openpyxl"
) as writer:

    df_africa_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_africa_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_africa_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename_africa_tt)

✅ Excel criado com sucesso!
Scopus_African_Law_Technology_Transfer_2004_2026.xlsx


In [85]:
from google.colab import files

files.download(filename_africa_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [86]:
# ============================================================
# AFRICAN LAW — TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_AFRICA_TI = f'''
(
    {journals_part_african_law}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_AFRICA_TI)

CONSULTA GERADA
(     ISSN(1999-7639) OR ISSN(2411-6955) OR ISSN(1609-073X) OR ISSN(2467-8406) OR ISSN(1727-3781) OR ISSN(0258-2503) OR ISSN(1998-9881) OR ISSN(1021-545X) OR ISSN(0257-7747) OR ISSN(2523-1367) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)


In [87]:
# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_AFRICA_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_africa_ti = response.json()

    total_results_africa_ti = int(
        data_africa_ti["search-results"]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_africa_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [89]:
# ============================================================
# REGISTRAR BUSCA COM ZERO RESULTADOS
# AFRICAN LAW / TECHNOLOGY INTERNALIZATION
# ============================================================

# Planilha vazia mantendo as mesmas colunas
df_africa_ti = pd.DataFrame(columns=[
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
])

# Search Log
search_log_africa_ti = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],
    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_african_law),
        "; ".join(
            confirmed_african_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Internalization" OR "Internalization of Technology" OR "Technological Internalization"',
        "ISSN",
        0
    ]
})

# Nome do arquivo
filename_africa_ti = (
    "Scopus_African_Law_"
    "Technology_Internalization_2004_2026.xlsx"
)

# Exportar
with pd.ExcelWriter(
    filename_africa_ti,
    engine="openpyxl"
) as writer:

    df_africa_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_africa_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print("Total de resultados: 0")
print(filename_africa_ti)

✅ Excel criado com sucesso!
Total de resultados: 0
Scopus_African_Law_Technology_Internalization_2004_2026.xlsx


In [90]:
from google.colab import files

files.download(filename_africa_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [91]:
# ============================================================
# NOVO BLOCO — CENTRAL / EASTERN EUROPE + BRICS
# VERIFICAÇÃO DOS PERIÓDICOS NO SCOPUS
# ============================================================

europe_brics_journals = {

    "Masaryk University Journal of Law and Technology": [
        "1802-5943",
        "1802-5951"
    ],

    "Environmental Economics": [
        "1998-6041",
        "1998-605X"
    ],

    "Forum Scientiae Oeconomia": [
        "2300-5947"
    ],

    "Bialostockie Studia Prawnicze": [
        "1689-7404",
        "2719-9452"
    ],

    "TalTech Journal of European Studies": [
        "2674-4600",
        "2674-4619"
    ],

    "Central European Public Administration Review": [
        "2591-2240",
        "2591-2259"
    ],

    "Transactions on Maritime Science": [
        "1848-3305",
        "1848-3313"
    ],

    "Studia Iuridica Lublinensia": [
        "1731-6375",
        "2449-8289"
    ],

    "BRICS Law Journal": [
        "2409-9058",
        "2412-2343"
    ],

    "Banks and Bank Systems": [
        "1816-7403",
        "1991-7074"
    ],

    "Lex Portus": [
        "2524-101X",
        "2617-541X"
    ],

    "Investment Management and Financial Innovations": [
        "1810-4967",
        "1812-9358"
    ],

    "Juridical Tribune": [
        "2247-7195",
        "2248-0382"
    ],

    "Prawo i Wiez": [
        "2299-405X"
    ],

    "Croatian Yearbook of European Law and Policy": [
        "1845-5662",
        "1848-9958"
    ]
}

print(
    f"Total de periódicos cadastrados: "
    f"{len(europe_brics_journals)}"
)

Total de periódicos cadastrados: 15


In [92]:
# ============================================================
# VERIFICAR OS 15 PERIÓDICOS NO SCOPUS
# ============================================================

scopus_europe_brics = []

for journal, issns in europe_brics_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_europe_brics.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_europe_brics.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")

✅ Encontrado: Masaryk University Journal of Law and Technology
✅ Encontrado: Environmental Economics
✅ Encontrado: Forum Scientiae Oeconomia
✅ Encontrado: Bialostockie Studia Prawnicze
✅ Encontrado: TalTech Journal of European Studies
✅ Encontrado: Central European Public Administration Review
✅ Encontrado: Transactions on Maritime Science
✅ Encontrado: Studia Iuridica Lublinensia
✅ Encontrado: BRICS Law Journal
✅ Encontrado: Banks and Bank Systems
✅ Encontrado: Lex Portus
✅ Encontrado: Investment Management and Financial Innovations
✅ Encontrado: Juridical Tribune
✅ Encontrado: Prawo i Wiez
✅ Encontrado: Croatian Yearbook of European Law and Policy


In [93]:
# ============================================================
# TABELA DE VERIFICAÇÃO
# ============================================================

scopus_europe_brics_df = pd.DataFrame(
    scopus_europe_brics
)

display(scopus_europe_brics_df)

total_confirmed = (
    scopus_europe_brics_df["Status"]
    == "FOUND IN SCOPUS"
).sum()

print(
    f"\n✅ Confirmados no Scopus: "
    f"{total_confirmed} / "
    f"{len(scopus_europe_brics_df)}"
)

,Journal,ISSN searched,Scopus title,Source ID,Publisher,Status
0,Masaryk University Journal of Law and Technology,1802-5943,Masaryk University Journal of Law and Technology,21100255068,"Faculty of Law, Masaryk University",FOUND IN SCOPUS
1,Environmental Economics,1998-6041,Environmental Economics,21101071896,LLC CPC Business Perspectives,FOUND IN SCOPUS
2,Forum Scientiae Oeconomia,2300-5947,Forum Scientiae Oeconomia,21101040211,WSB University,FOUND IN SCOPUS
3,Bialostockie Studia Prawnicze,1689-7404,Bialostockie Studia Prawnicze,21101149930,Paradigm Publishing Services,FOUND IN SCOPUS
4,TalTech Journal of European Studies,2674-4600,TalTech Journal of European Studies,21101190201,Paradigm Publishing Services,FOUND IN SCOPUS
5,Central European Public Administration Review,2591-2240,Central European Public Administration Review,21101185807,University of Ljubljana Faculty of Public Admi...,FOUND IN SCOPUS
6,Transactions on Maritime Science,1848-3305,Transactions on Maritime Science,21100840720,Faculty of Maritime Studies,FOUND IN SCOPUS
7,Studia Iuridica Lublinensia,1731-6375,Studia Iuridica Lublinensia,21101037295,Wydawnictwo Uniwersytetu Marii Curie-Sklodowsk...,FOUND IN SCOPUS
8,BRICS Law Journal,2409-9058,BRICS Law Journal,21100857559,University of Tyumen,FOUND IN SCOPUS
9,Banks and Bank Systems,1816-7403,Banks and Bank Systems,21100407655,LLC CPC Business Perspectives,FOUND IN SCOPUS



✅ Confirmados no Scopus: 15 / 15


In [94]:
# ============================================================
# CRIAR UNIVERSO DOS 15 PERIÓDICOS CONFIRMADOS
# ============================================================

confirmed_europe_brics = scopus_europe_brics_df[
    scopus_europe_brics_df["Status"] == "FOUND IN SCOPUS"
].copy()

issn_queries_europe_brics = []

for _, row in confirmed_europe_brics.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_europe_brics.append(
            f'ISSN({issn})'
        )

journals_part_europe_brics = " OR ".join(
    issn_queries_europe_brics
)

print(
    f"Periódicos utilizados: "
    f"{len(confirmed_europe_brics)}"
)

print("\nISSNs utilizados na consulta:")
print(journals_part_europe_brics)

Periódicos utilizados: 15

ISSNs utilizados na consulta:
ISSN(1802-5943) OR ISSN(1998-6041) OR ISSN(2300-5947) OR ISSN(1689-7404) OR ISSN(2674-4600) OR ISSN(2591-2240) OR ISSN(1848-3305) OR ISSN(1731-6375) OR ISSN(2409-9058) OR ISSN(1816-7403) OR ISSN(2524-101X) OR ISSN(1810-4967) OR ISSN(2247-7195) OR ISSN(2299-405X) OR ISSN(1845-5662)


In [95]:
# ============================================================
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

SEARCH_QUERY_EUROPE_BRICS_TT = f'''
(
    {journals_part_europe_brics}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_EUROPE_BRICS_TT)

CONSULTA GERADA
(     ISSN(1802-5943) OR ISSN(1998-6041) OR ISSN(2300-5947) OR ISSN(1689-7404) OR ISSN(2674-4600) OR ISSN(2591-2240) OR ISSN(1848-3305) OR ISSN(1731-6375) OR ISSN(2409-9058) OR ISSN(1816-7403) OR ISSN(2524-101X) OR ISSN(1810-4967) OR ISSN(2247-7195) OR ISSN(2299-405X) OR ISSN(1845-5662) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)


In [96]:
# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_EUROPE_BRICS_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_europe_brics_tt = response.json()

    total_results_europe_brics_tt = int(
        data_europe_brics_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_europe_brics_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 5


In [97]:
# ============================================================
# BAIXAR RESULTADOS
# EUROPE/BRICS — TECHNOLOGY TRANSFER
# ============================================================

all_results_europe_brics_tt = []

count = 25
start = 0

while start < total_results_europe_brics_tt:

    params = {
        "query": SEARCH_QUERY_EUROPE_BRICS_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_europe_brics_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_europe_brics_tt)} / "
        f"{total_results_europe_brics_tt}"
    )

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(
    f"Total de registros baixados: "
    f"{len(all_results_europe_brics_tt)}"
)

Baixados: 5 / 5

✅ Busca concluída!
Total de registros baixados: 5


In [98]:
# ============================================================
# ORGANIZAR RESULTADOS
# EUROPE/BRICS — TECHNOLOGY TRANSFER
# ============================================================

records_europe_brics_tt = []

for item in all_results_europe_brics_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_europe_brics_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_europe_brics_tt = pd.DataFrame(
    records_europe_brics_tt
)

print("✅ Tabela criada!")
print(
    f"Total de artigos: "
    f"{len(df_europe_brics_tt)}"
)

display(df_europe_brics_tt)

✅ Tabela criada!
Total de artigos: 5


,Journal,Title,Authors,Year,ISSN,Volume,Issue,Pages,DOI,EID,Cited by,Scopus URL
0,Brics Law Journal,Harmonization of Robotics Legislation as a Fac...,Begishev I.,2026,24099058,13,1,29-55,10.21684/2412-2343-2026-13-1-29-55,2-s2.0-105039211553,0,https://www.scopus.com/inward/record.uri?partn...
1,Environmental Economics,Foreign direct investment and manufacturing CO...,Tran-Nam Q.,2026,19986041,17,2,1-14,10.21511/ee.17(2).2026.01,2-s2.0-105037973710,0,https://www.scopus.com/inward/record.uri?partn...
2,Environmental Economics,Foreign direct investment and carbon emissions...,Anh-Tu N.,2026,19986041,17,1,78-93,10.21511/ee.17(1).2026.07,2-s2.0-105036305837,0,https://www.scopus.com/inward/record.uri?partn...
3,Investment Management and Financial Innovations,The impact of financial globalization on the e...,Hrubliak O.,2025,18104967,22,4,209-226,10.21511/imfi.22(4).2025.17,2-s2.0-105023378193,5,https://www.scopus.com/inward/record.uri?partn...
4,Investment Management and Financial Innovations,Impact of FDI on innovation in Tunisia's high-...,Seghir S.,2012,18104967,9,4,96-103,,2-s2.0-84873352793,0,https://www.scopus.com/inward/record.uri?partn...


In [99]:
# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_europe_brics_tt = (
    df_europe_brics_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts_europe_brics_tt)

,Journal,Number of articles
0,Environmental Economics,2
1,Investment Management and Financial Innovations,2
2,Brics Law Journal,1


In [100]:
# ============================================================
# SEARCH LOG
# ============================================================

search_log_europe_brics_tt = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
        len(confirmed_europe_brics),
        "; ".join(
            confirmed_europe_brics[
                "Journal"
            ].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Transfer" OR "Transfer of Technology" OR "Technological Transfer"',
        "ISSN",
        len(df_europe_brics_tt)
    ]
})

display(search_log_europe_brics_tt)

,Field,Value
0,Database,Scopus
1,Search date,2026-09-30 01:11:04
2,Number of journals,15
3,Journals,Masaryk University Journal of Law and Technolo...
4,Publication years,2004–2026
5,Document type,Article
6,Search field,TITLE-ABS-KEY
7,Search terms,"""Technology Transfer"" OR ""Transfer of Technolo..."
8,Source identification,ISSN
9,Total results,5


In [101]:
# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_europe_brics_tt = (
    "Scopus_Europe_BRICS_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_europe_brics_tt,
    engine="openpyxl"
) as writer:

    df_europe_brics_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_europe_brics_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_europe_brics_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print(filename_europe_brics_tt)

✅ Excel criado com sucesso!
Scopus_Europe_BRICS_Technology_Transfer_2004_2026.xlsx


In [102]:
from google.colab import files

files.download(filename_europe_brics_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [103]:
# ============================================================
# EUROPE/BRICS — TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_EUROPE_BRICS_TI = f'''
(
    {journals_part_europe_brics}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")
print(SEARCH_QUERY_EUROPE_BRICS_TI)

print("\n============================================================")
print(
    f"Periódicos utilizados: "
    f"{len(confirmed_europe_brics)}"
)
print("============================================================")

CONSULTA GERADA
(     ISSN(1802-5943) OR ISSN(1998-6041) OR ISSN(2300-5947) OR ISSN(1689-7404) OR ISSN(2674-4600) OR ISSN(2591-2240) OR ISSN(1848-3305) OR ISSN(1731-6375) OR ISSN(2409-9058) OR ISSN(1816-7403) OR ISSN(2524-101X) OR ISSN(1810-4967) OR ISSN(2247-7195) OR ISSN(2299-405X) OR ISSN(1845-5662) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

Periódicos utilizados: 15


In [104]:
# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_EUROPE_BRICS_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_europe_brics_ti = response.json()

    total_results_europe_brics_ti = int(
        data_europe_brics_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")
    print(
        f"Total de resultados encontrados: "
        f"{total_results_europe_brics_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [106]:
# ============================================================
# EUROPE/BRICS — TECHNOLOGY INTERNALIZATION
# ZERO RESULTADOS
# ============================================================

# Tabela vazia com a estrutura padrão
df_europe_brics_ti = pd.DataFrame(columns=[
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
])

# Search Log
search_log_europe_brics_ti = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_europe_brics),
        "; ".join(
            confirmed_europe_brics["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        '"Technology Internalization" OR "Internalization of Technology" OR "Technological Internalization"',
        "ISSN",
        0
    ]
})

# Nome do arquivo
filename_europe_brics_ti = (
    "Scopus_Europe_BRICS_"
    "Technology_Internalization_2004_2026.xlsx"
)

# Exportar
with pd.ExcelWriter(
    filename_europe_brics_ti,
    engine="openpyxl"
) as writer:

    df_europe_brics_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_europe_brics_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Excel criado com sucesso!")
print("Total de resultados: 0")
print(filename_europe_brics_ti)

✅ Excel criado com sucesso!
Total de resultados: 0
Scopus_Europe_BRICS_Technology_Internalization_2004_2026.xlsx


In [107]:
from google.colab import files

files.download(filename_europe_brics_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [120]:
# ============================================================
# LATIN AMERICAN LAW JOURNALS
# VERSÃO CORRIGIDA
# ============================================================

import requests
import pandas as pd
import time


# ============================================================
# 1. LISTA CORRIGIDA DOS 16 PERIÓDICOS
# ============================================================

latin_american_law_journals = {

    "Novum Jus": [
        "1692-6013",
        "2500-8692"
    ],

    "Revista de Investigacoes Constitucionais": [
        "2359-5639"
    ],

    "Revista Eurolatinoamericana de Derecho Administrativo": [
        "2362-583X"
    ],

    "Via Inveniendi et Iudicandi": [
        "1909-0528"
    ],

    "A e C - Revista de Direito Administrativo e Constitucional": [
        "1516-3210",
        "1984-4182"
    ],

    "Pro Jure Revista de Derecho": [
        "2810-7659",
        "0718-6851",
        "0716-1883"
    ],

    "Revista Chilena de Derecho": [
        "0716-0747",
        "0718-3437"
    ],

    # Revista de Derecho (Valdivia, Chile)
    "Revista de Derecho": [
        "0716-9132",
        "0718-0950"
    ],

    "Revista Cientifica General Jose Maria Cordova": [
        "1900-6586",
        "2500-7645"
    ],

    "Revista Derecho del Estado": [
        "0122-9893",
        "2346-2051"
    ],

    "Ius et Praxis": [
        "0717-2877",
        "0718-0012"
    ],

    "Revista de Derecho Ambiental (Chile)": [
        "0719-4633"
    ],

    "Dilemas": [
        "1983-5922",
        "2178-2792"
    ],

    "Cuestiones Constitucionales": [
        "1405-9193",
        "2448-4881"
    ],

    "Vniversitas": [
        "0041-9060",
        "2011-1711"
    ],

    # CORRIGIDO
    "Relacoes Internacionais no Mundo Atual": [
        "1518-9368",
        "2316-2880"
    ]
}

print("============================================================")
print("PERIÓDICOS CADASTRADOS")
print("============================================================")
print(
    f"Total de periódicos: "
    f"{len(latin_american_law_journals)}"
)


# ============================================================
# 2. VERIFICAR OS 16 NO SCOPUS
# ============================================================

scopus_latin_american_law = []

for journal, issns in latin_american_law_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_latin_american_law.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if found:
        print(f"✅ Encontrado: {journal}")

    else:

        scopus_latin_american_law.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# 3. CRIAR TABELA DE VERIFICAÇÃO
# ============================================================

scopus_latin_american_law_df = pd.DataFrame(
    scopus_latin_american_law
)

display(
    scopus_latin_american_law_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)

total_confirmed_latin = (
    scopus_latin_american_law_df["Status"]
    == "FOUND IN SCOPUS"
).sum()

print("\n============================================================")
print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_latin} / 16"
)
print("============================================================")


# ============================================================
# 4. SELECIONAR SOMENTE OS CONFIRMADOS
# ============================================================

confirmed_latin_american_law = (
    scopus_latin_american_law_df[
        scopus_latin_american_law_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)


# ============================================================
# 5. CRIAR FILTRO DOS PERIÓDICOS
# ============================================================

issn_queries_latin = []

for _, row in confirmed_latin_american_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_latin.append(
            f'ISSN({issn})'
        )

journals_part_latin = " OR ".join(
    issn_queries_latin
)

print(
    f"\nPeriódicos utilizados na busca: "
    f"{len(confirmed_latin_american_law)}"
)


# ============================================================
# 6. BUSCA MULTILÍNGUE — TECHNOLOGY TRANSFER
# ENGLISH + PORTUGUESE + SPANISH
# 2004–2026
# ============================================================

SEARCH_QUERY_LATIN_TT = f'''
(
    {journals_part_latin}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
    OR
    "Transferência de Tecnologia"
    OR
    "Transferencia de Tecnología"
    OR
    "Transferência Tecnológica"
    OR
    "Transferencia Tecnológica"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_LATIN_TT)


# ============================================================
# 7. EXECUTAR A CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_LATIN_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_latin_tt = response.json()

    total_results_latin_tt = int(
        data_latin_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_latin_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

PERIÓDICOS CADASTRADOS
Total de periódicos: 16
✅ Encontrado: Novum Jus
✅ Encontrado: Revista de Investigacoes Constitucionais
✅ Encontrado: Revista Eurolatinoamericana de Derecho Administrativo
✅ Encontrado: Via Inveniendi et Iudicandi
✅ Encontrado: A e C - Revista de Direito Administrativo e Constitucional
✅ Encontrado: Pro Jure Revista de Derecho
✅ Encontrado: Revista Chilena de Derecho
✅ Encontrado: Revista de Derecho
✅ Encontrado: Revista Cientifica General Jose Maria Cordova
✅ Encontrado: Revista Derecho del Estado
✅ Encontrado: Ius et Praxis
✅ Encontrado: Revista de Derecho Ambiental (Chile)
✅ Encontrado: Dilemas
✅ Encontrado: Cuestiones Constitucionales
✅ Encontrado: Vniversitas
✅ Encontrado: Relacoes Internacionais no Mundo Atual


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Novum Jus,1692-6013,Novum Jus,21101060255,FOUND IN SCOPUS
1,Revista de Investigacoes Constitucionais,2359-5639,Revista de Investigacoes Constitucionais,21100855986,FOUND IN SCOPUS
2,Revista Eurolatinoamericana de Derecho Adminis...,2362-583X,Revista Eurolatinoamericana de Derecho Adminis...,21101079115,FOUND IN SCOPUS
3,Via Inveniendi et Iudicandi,1909-0528,Via Inveniendi et Iudicandi,21101072371,FOUND IN SCOPUS
4,A e C - Revista de Direito Administrativo e Co...,1516-3210,A e C - Revista de Direito Administrativo e Co...,21101076713,FOUND IN SCOPUS
5,Pro Jure Revista de Derecho,2810-7659,Pro Jure Revista de Derecho,21101326198,FOUND IN SCOPUS
6,Revista Chilena de Derecho,0716-0747,Revista Chilena de Derecho,11700154202,FOUND IN SCOPUS
7,Revista de Derecho,0716-9132,Revista de Derecho,5000160302,FOUND IN SCOPUS
8,Revista Cientifica General Jose Maria Cordova,1900-6586,Revista Cientifica General Jose Maria Cordova,21100996939,FOUND IN SCOPUS
9,Revista Derecho del Estado,0122-9893,Revista Derecho del Estado,21100432126,FOUND IN SCOPUS



✅ Confirmados no Scopus: 16 / 16

Periódicos utilizados na busca: 16

CONSULTA GERADA
(     ISSN(1692-6013) OR ISSN(2359-5639) OR ISSN(2362-583X) OR ISSN(1909-0528) OR ISSN(1516-3210) OR ISSN(2810-7659) OR ISSN(0716-0747) OR ISSN(0716-9132) OR ISSN(1900-6586) OR ISSN(0122-9893) OR ISSN(0717-2877) OR ISSN(0719-4633) OR ISSN(1983-5922) OR ISSN(1405-9193) OR ISSN(0041-9060) OR ISSN(2316-2880) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer"     OR     "Transferência de Tecnologia"     OR     "Transferencia de Tecnología"     OR     "Transferência Tecnológica"     OR     "Transferencia Tecnológica" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 4


In [121]:
# ============================================================
# BAIXAR OS 4 RESULTADOS CORRIGIDOS
# LATIN AMERICAN LAW — TECHNOLOGY TRANSFER
# ============================================================

all_results_latin_tt = []

count = 25
start = 0

while start < total_results_latin_tt:

    params = {
        "query": SEARCH_QUERY_LATIN_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_latin_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_latin_tt)} / "
        f"{total_results_latin_tt}"
    )

    start += count
    time.sleep(1)

print("\n✅ Busca concluída!")
print(
    f"Total de registros baixados: "
    f"{len(all_results_latin_tt)}"
)

# ============================================================
# CONFERÊNCIA DOS PERIÓDICOS E TÍTULOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(all_results_latin_tt, start=1):

    print(f"\n{i}. {item.get('dc:title', '')}")

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   ISSN:",
        item.get("prism:issn", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

Baixados: 4 / 4

✅ Busca concluída!
Total de registros baixados: 4

ARTIGOS RECUPERADOS

1. Legal Mechanisms for the Protection and Transfer of Technology: Toward an Effective Reform in Chile, Second Project and Pending Challenges
   Periódico: Novum Jus
   ISSN: 16926013
   Data: 2025-09-01

2. THE IMPORTANCE OF RESETTING THE SOCIO-LEGAL CULTURE OF TECHNOLOGY TRANSFER IN CHILE
   Periódico: Novum Jus
   ISSN: 16926013
   Data: 2022-10-01

3. COLLABORATIVE INTELLIGENCE AND INTELLECTUAL PROPERTY: HUMANS AND HUMANOIDS IN THE LIGHT OF THE HEALTH CARE SYSTEM
   Periódico: Novum Jus
   ISSN: 16926013
   Data: 2021-01-01

4. THE DEVELOPMENTALISM FUNCTION OF THE STATE IN THE AEGIS OF TECHNOLOGY AND KNOW-HOW TRANSFER AGREEMENT
   Periódico: Relacoes Internacionais no Mundo Atual
   ISSN: 
   Data: 2019-07-01


In [122]:
# ============================================================
# ORGANIZAR RESULTADOS DEFINITIVOS
# LATIN AMERICAN LAW — TECHNOLOGY TRANSFER
# ============================================================

records_latin_tt = []

for item in all_results_latin_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_latin_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_latin_tt = pd.DataFrame(records_latin_tt)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_latin_tt = (
    df_latin_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_latin_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer" OR '
    '"Transferência de Tecnologia" OR '
    '"Transferencia de Tecnología" OR '
    '"Transferência Tecnológica" OR '
    '"Transferencia Tecnológica"'
)

search_log_latin_tt = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        16,
        "; ".join(
            confirmed_latin_american_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English; Portuguese; Spanish",
        7,
        search_terms_latin_tt,
        "ISSN",
        len(df_latin_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_latin_tt = (
    "Scopus_Latin_American_Law_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_latin_tt,
    engine="openpyxl"
) as writer:

    df_latin_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_latin_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_latin_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(f"Total de artigos: {len(df_latin_tt)}")
print("\nArtigos por periódico:")

display(journal_counts_latin_tt)

print("\nArquivo:")
print(filename_latin_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 4

Artigos por periódico:


,Journal,Number of articles
0,Novum Jus,3
1,Relacoes Internacionais no Mundo Atual,1



Arquivo:
Scopus_Latin_American_Law_Technology_Transfer_2004_2026.xlsx


In [123]:
from google.colab import files

files.download(filename_latin_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [124]:
# ============================================================
# LATIN AMERICAN LAW — TECHNOLOGY INTERNALIZATION
# ENGLISH + PORTUGUESE + SPANISH
# 2004–2026
# ============================================================


# ============================================================
# 1. CRIAR A CONSULTA
# ============================================================

SEARCH_QUERY_LATIN_TI = f'''
(
    {journals_part_latin}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
    OR
    "Internalização de Tecnologia"
    OR
    "Internalización de Tecnología"
    OR
    "Internalização Tecnológica"
    OR
    "Internalización Tecnológica"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_LATIN_TI)


# ============================================================
# 2. EXECUTAR A CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_LATIN_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_latin_ti = response.json()

    total_results_latin_ti = int(
        data_latin_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_latin_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(1692-6013) OR ISSN(2359-5639) OR ISSN(2362-583X) OR ISSN(1909-0528) OR ISSN(1516-3210) OR ISSN(2810-7659) OR ISSN(0716-0747) OR ISSN(0716-9132) OR ISSN(1900-6586) OR ISSN(0122-9893) OR ISSN(0717-2877) OR ISSN(0719-4633) OR ISSN(1983-5922) OR ISSN(1405-9193) OR ISSN(0041-9060) OR ISSN(2316-2880) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization"     OR     "Internalização de Tecnologia"     OR     "Internalización de Tecnología"     OR     "Internalização Tecnológica"     OR     "Internalización Tecnológica" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [125]:
# ============================================================
# GERAR EXCEL — TECHNOLOGY INTERNALIZATION
# LATIN AMERICAN LAW
# 2004–2026
# RESULTADO: 0
# ============================================================

import pandas as pd


# ============================================================
# 1. RESULTS — TABELA VAZIA
# ============================================================

columns_latin_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_latin_ti = pd.DataFrame(
    columns=columns_latin_ti
)


# ============================================================
# 2. SEARCH TERMS
# ============================================================

search_terms_latin_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization" OR '
    '"Internalização de Tecnologia" OR '
    '"Internalización de Tecnología" OR '
    '"Internalização Tecnológica" OR '
    '"Internalización Tecnológica"'
)


# ============================================================
# 3. SEARCH LOG
# ============================================================

search_log_latin_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        16,
        "; ".join(
            confirmed_latin_american_law[
                "Journal"
            ].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English; Portuguese; Spanish",
        7,
        search_terms_latin_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# 4. EXPORTAR
# ============================================================

filename_latin_ti = (
    "Scopus_Latin_American_Law_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_latin_ti,
    engine="openpyxl"
) as writer:

    df_latin_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_latin_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# 5. CONFIRMAÇÃO
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print("Número de periódicos: 16")
print("Idiomas: English; Portuguese; Spanish")
print("Número de expressões: 7")
print("Período: 2004–2026")

print("\nArquivo:")
print(filename_latin_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Número de periódicos: 16
Idiomas: English; Portuguese; Spanish
Número de expressões: 7
Período: 2004–2026

Arquivo:
Scopus_Latin_American_Law_Technology_Internalization_2004_2026.xlsx


In [126]:
from google.colab import files

files.download(
    "Scopus_Latin_American_Law_Technology_Internalization_2004_2026.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [127]:
# ============================================================
# NORTHERN AMERICA / US LAW JOURNALS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

import requests
import pandas as pd
import time


# ============================================================
# 1. PERIÓDICOS + ISSNs
# ============================================================

us_law_journals = {

    "International Security": [
        "0162-2889",
        "1531-4804"
    ],

    "Yale Law Journal": [
        "0044-0094"
    ],

    "Virginia Law Review": [
        "0042-6601"
    ],

    "Harvard Law Review": [
        "0017-811X"
    ],

    "Stanford Law Review": [
        "0038-9765"
    ],

    "Journal of Law and Economics": [
        "0022-2186",
        "1537-5285"
    ],

    "Columbia Law Review": [
        "0010-1958"
    ],

    "University of Pennsylvania Law Review": [
        "0041-9907"
    ],

    "Duke Law Journal": [
        "0012-7086"
    ],

    "California Law Review": [
        "0008-1221"
    ],

    "Michigan Law Review": [
        "0026-2234"
    ],

    "Police Quarterly": [
        "1098-6111",
        "1552-745X"
    ],

    "Texas Law Review": [
        "0040-4411"
    ],

    "Vanderbilt Law Review": [
        "0042-2533"
    ],

    "Social Science Computer Review": [
        "0894-4393",
        "1552-8286"
    ],

    "Yale Journal on Regulation": [
        "0741-9457",
        "2376-5925"
    ]
}


print("============================================================")
print("PERIÓDICOS CADASTRADOS")
print("============================================================")

print(
    f"Total de periódicos: "
    f"{len(us_law_journals)}"
)


# ============================================================
# 2. VERIFICAR CADA PERIÓDICO NO SCOPUS
# ============================================================

scopus_us_law = []

for journal, issns in us_law_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_us_law.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get(
                        "dc:title",
                        ""
                    ),
                    "Source ID": entry.get(
                        "source-id",
                        ""
                    ),
                    "Publisher": entry.get(
                        "dc:publisher",
                        ""
                    ),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if found:

        print(
            f"✅ Encontrado: {journal}"
        )

    else:

        scopus_us_law.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(
            f"⚠️ Não encontrado: {journal}"
        )


# ============================================================
# 3. DATAFRAME
# ============================================================

scopus_us_law_df = pd.DataFrame(
    scopus_us_law
)


# ============================================================
# 4. MOSTRAR RESULTADOS
# ============================================================

display(
    scopus_us_law_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# 5. CONTAGEM FINAL
# ============================================================

total_confirmed_us = (
    scopus_us_law_df["Status"]
    == "FOUND IN SCOPUS"
).sum()

print("\n============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_us} / "
    f"{len(us_law_journals)}"
)

print("============================================================")

PERIÓDICOS CADASTRADOS
Total de periódicos: 16
✅ Encontrado: International Security
✅ Encontrado: Yale Law Journal
✅ Encontrado: Virginia Law Review
✅ Encontrado: Harvard Law Review
✅ Encontrado: Stanford Law Review
✅ Encontrado: Journal of Law and Economics
✅ Encontrado: Columbia Law Review
✅ Encontrado: University of Pennsylvania Law Review
✅ Encontrado: Duke Law Journal
✅ Encontrado: California Law Review
✅ Encontrado: Michigan Law Review
✅ Encontrado: Police Quarterly
✅ Encontrado: Texas Law Review
✅ Encontrado: Vanderbilt Law Review
✅ Encontrado: Social Science Computer Review
✅ Encontrado: Yale Journal on Regulation


,Journal,ISSN searched,Scopus title,Source ID,Status
0,International Security,0162-2889,International Security,25835,FOUND IN SCOPUS
1,Yale Law Journal,0044-0094,Yale Law Journal,22856,FOUND IN SCOPUS
2,Virginia Law Review,0042-6601,Virginia Law Review,22132,FOUND IN SCOPUS
3,Harvard Law Review,0017-811X,Harvard Law Review,14316,FOUND IN SCOPUS
4,Stanford Law Review,0038-9765,Stanford Law Review,19751,FOUND IN SCOPUS
5,Journal of Law and Economics,0022-2186,Journal of Law and Economics,16220,FOUND IN SCOPUS
6,Columbia Law Review,0010-1958,Columbia Law Review,13143,FOUND IN SCOPUS
7,University of Pennsylvania Law Review,0041-9907,University of Pennsylvania Law Review,21681,FOUND IN SCOPUS
8,Duke Law Journal,0012-7086,Duke Law Journal,15184,FOUND IN SCOPUS
9,California Law Review,0008-1221,California Law Review,29868,FOUND IN SCOPUS



✅ Confirmados no Scopus: 16 / 16


In [129]:
# ============================================================
# US / NORTH AMERICAN JOURNALS
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================


# ============================================================
# 1. SOMENTE PERIÓDICOS CONFIRMADOS
# ============================================================

confirmed_us_law = (
    scopus_us_law_df[
        scopus_us_law_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_us_law)}"
)


# ============================================================
# 2. CRIAR FILTRO POR ISSN
# ============================================================

issn_queries_us = []

for _, row in confirmed_us_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_us.append(
            f'ISSN({issn})'
        )

journals_part_us = " OR ".join(
    issn_queries_us
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_us)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER — 3 EXPRESSÕES
# ============================================================

SEARCH_QUERY_US_TT = f'''
(
    {journals_part_us}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_US_TT)


# ============================================================
# 4. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_US_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_us_tt = response.json()

    total_results_us_tt = int(
        data_us_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_us_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 16
Periódicos utilizados na busca: 16

CONSULTA GERADA
(     ISSN(0162-2889) OR ISSN(0044-0094) OR ISSN(0042-6601) OR ISSN(0017-811X) OR ISSN(0038-9765) OR ISSN(0022-2186) OR ISSN(0010-1958) OR ISSN(0041-9907) OR ISSN(0012-7086) OR ISSN(0008-1221) OR ISSN(0026-2234) OR ISSN(1098-6111) OR ISSN(0040-4411) OR ISSN(0042-2533) OR ISSN(0894-4393) OR ISSN(0741-9457) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 3


In [130]:
# ============================================================
# BAIXAR E CONFERIR OS RESULTADOS
# US / NORTH AMERICAN JOURNALS — TECHNOLOGY TRANSFER
# ============================================================

all_results_us_tt = []

count = 25
start = 0

while start < total_results_us_tt:

    params = {
        "query": SEARCH_QUERY_US_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_us_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_us_tt)} / "
        f"{total_results_us_tt}"
    )

    start += count
    time.sleep(1)


print("\n✅ Busca concluída!")

print(
    f"Total de registros baixados: "
    f"{len(all_results_us_tt)}"
)


# ============================================================
# CONFERÊNCIA DOS ARTIGOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_us_tt,
    start=1
):

    print(
        f"\n{i}. "
        f"{item.get('dc:title', '')}"
    )

    print(
        "   Periódico:",
        item.get(
            "prism:publicationName",
            ""
        )
    )

    print(
        "   ISSN:",
        item.get(
            "prism:issn",
            ""
        )
    )

    print(
        "   Data:",
        item.get(
            "prism:coverDate",
            ""
        )
    )

    print(
        "   DOI:",
        item.get(
            "prism:doi",
            ""
        )
    )

Baixados: 3 / 3

✅ Busca concluída!
Total de registros baixados: 3

ARTIGOS RECUPERADOS

1. Innovation and the Firm: A new synthesis
   Periódico: Stanford Law Review
   ISSN: 00389765
   Data: 2018-05-01
   DOI: 

2. Achieving world trade organization compliance for export processing zones while maintaining economic competitiveness for developing countries
   Periódico: Duke Law Journal
   ISSN: 00127086
   Data: 2013-11-01
   DOI: 

3. Intellectual property protection and technology licensing: The case of developing countries
   Periódico: Journal of Law and Economics
   ISSN: 00222186
   Data: 2012-08-01
   DOI: 10.1086/665566


In [131]:
# ============================================================
# ORGANIZAR E EXPORTAR RESULTADOS
# US / NORTH AMERICAN JOURNALS — TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

records_us_tt = []

for item in all_results_us_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_us_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_us_tt = pd.DataFrame(records_us_tt)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_us_tt = (
    df_us_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_us_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_us_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_us_law),
        "; ".join(
            confirmed_us_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_us_tt,
        "ISSN",
        len(df_us_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_us_tt = (
    "Scopus_US_Law_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_us_tt,
    engine="openpyxl"
) as writer:

    df_us_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_us_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_us_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(f"Total de artigos: {len(df_us_tt)}")

print("\nArtigos por periódico:")
display(journal_counts_us_tt)

print("\nArquivo:")
print(filename_us_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 3

Artigos por periódico:


,Journal,Number of articles
0,Stanford Law Review,1
1,Duke Law Journal,1
2,Journal of Law and Economics,1



Arquivo:
Scopus_US_Law_Technology_Transfer_2004_2026.xlsx


In [132]:
from google.colab import files

files.download(filename_us_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [133]:
# ============================================================
# US / NORTH AMERICAN JOURNALS
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_US_TI = f'''
(
    {journals_part_us}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_US_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_US_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)

if response.status_code == 200:

    data_us_ti = response.json()

    total_results_us_ti = int(
        data_us_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_us_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0162-2889) OR ISSN(0044-0094) OR ISSN(0042-6601) OR ISSN(0017-811X) OR ISSN(0038-9765) OR ISSN(0022-2186) OR ISSN(0010-1958) OR ISSN(0041-9907) OR ISSN(0012-7086) OR ISSN(0008-1221) OR ISSN(0026-2234) OR ISSN(1098-6111) OR ISSN(0040-4411) OR ISSN(0042-2533) OR ISSN(0894-4393) OR ISSN(0741-9457) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [134]:
# ============================================================
# GERAR EXCEL — TECHNOLOGY INTERNALIZATION
# US / NORTH AMERICAN JOURNALS
# 2004–2026
# RESULTADO: 0
# ============================================================

# Results vazio
columns_us_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_us_ti = pd.DataFrame(columns=columns_us_ti)


# ============================================================
# SEARCH TERMS
# ============================================================

search_terms_us_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_us_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_us_law),
        "; ".join(
            confirmed_us_law["Journal"].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_us_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_us_ti = (
    "Scopus_US_Law_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_us_ti,
    engine="openpyxl"
) as writer:

    df_us_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_us_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print(f"Número de periódicos: {len(confirmed_us_law)}")
print("Idioma: English")
print("Número de expressões: 3")
print("Período: 2004–2026")

print("\nArquivo:")
print(filename_us_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Número de periódicos: 16
Idioma: English
Número de expressões: 3
Período: 2004–2026

Arquivo:
Scopus_US_Law_Technology_Internalization_2004_2026.xlsx


In [135]:
from google.colab import files

files.download(filename_us_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [136]:
# ============================================================
# PACIFIC REGION LAW JOURNALS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

import requests
import pandas as pd
import time


# ============================================================
# 1. PERIÓDICOS + ISSNs
# ============================================================

pacific_region_law_journals = {

    "Law, Technology and Humans": [
        "2652-4074"
    ],

    "University of New South Wales Law Journal": [
        "0313-0096",
        "1839-2881"
    ],

    "Cosmopolitan Civil Societies": [
        "1837-5391"
    ],

    "Australian Journal of Human Rights": [
        "1323-238X",
        "2573-573X"
    ],

    "Australian Year Book of International Law": [
        "0084-7658"
    ],

    "Global Biosecurity": [
        "2652-0036"
    ],

    "Journal of Law and Medicine": [
        "1320-159X"
    ],

    "University of Western Australia Law Review": [
        "0042-0328"
    ],

    "Sydney Law Review": [
        "0082-0512"
    ],

    "Melbourne University Law Review": [
        "0025-8938"
    ],

    "Adelaide Law Review": [
        "0065-1915"
    ],

    "Australian Journal of Asian Law": [
        "1443-0738"
    ],

    "Journal of the Australasian Tax Teachers Association": [
        "1832-911X"
    ],

    "University of Queensland Law Journal": [
        "0083-4041"
    ],

    "Journal of Equity": [
        "1833-2137"
    ],

    "Victoria University of Wellington Law Review": [
        "1171-042X"
    ]
}


print("============================================================")
print("PACIFIC REGION LAW JOURNALS")
print("============================================================")

print(
    f"Total de periódicos cadastrados: "
    f"{len(pacific_region_law_journals)}"
)


# ============================================================
# 2. VERIFICAR CADA PERIÓDICO NO SCOPUS
# ============================================================

scopus_pacific_region_law = []

for journal, issns in pacific_region_law_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_pacific_region_law.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get(
                        "dc:title",
                        ""
                    ),
                    "Source ID": entry.get(
                        "source-id",
                        ""
                    ),
                    "Publisher": entry.get(
                        "dc:publisher",
                        ""
                    ),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)

    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_pacific_region_law.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# 3. CRIAR DATAFRAME
# ============================================================

scopus_pacific_region_law_df = pd.DataFrame(
    scopus_pacific_region_law
)


# ============================================================
# 4. MOSTRAR RESULTADOS
# ============================================================

display(
    scopus_pacific_region_law_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# 5. CONTAGEM FINAL
# ============================================================

total_confirmed_pacific_region = (
    scopus_pacific_region_law_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("PACIFIC REGION LAW JOURNALS")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_pacific_region} / "
    f"{len(pacific_region_law_journals)}"
)

print("============================================================")

PACIFIC REGION LAW JOURNALS
Total de periódicos cadastrados: 16
✅ Encontrado: Law, Technology and Humans
✅ Encontrado: University of New South Wales Law Journal
✅ Encontrado: Cosmopolitan Civil Societies
✅ Encontrado: Australian Journal of Human Rights
✅ Encontrado: Australian Year Book of International Law
✅ Encontrado: Global Biosecurity
✅ Encontrado: Journal of Law and Medicine
✅ Encontrado: University of Western Australia Law Review
✅ Encontrado: Sydney Law Review
✅ Encontrado: Melbourne University Law Review
✅ Encontrado: Adelaide Law Review
✅ Encontrado: Australian Journal of Asian Law
✅ Encontrado: Journal of the Australasian Tax Teachers Association
✅ Encontrado: University of Queensland Law Journal
✅ Encontrado: Journal of Equity
✅ Encontrado: Victoria University of Wellington Law Review


,Journal,ISSN searched,Scopus title,Source ID,Status
0,"Law, Technology and Humans",2652-4074,"Law, Technology and Humans",21101089371,FOUND IN SCOPUS
1,University of New South Wales Law Journal,0313-0096,University of New South Wales Law Journal,21674,FOUND IN SCOPUS
2,Cosmopolitan Civil Societies,1837-5391,Cosmopolitan Civil Societies,21100828130,FOUND IN SCOPUS
3,Australian Journal of Human Rights,1323-238X,Australian Journal of Human Rights,21100855754,FOUND IN SCOPUS
4,Australian Year Book of International Law,0084-7658,Australian Year Book of International Law,21101071253,FOUND IN SCOPUS
5,Global Biosecurity,2652-0036,Global Biosecurity,21101140347,FOUND IN SCOPUS
6,Journal of Law and Medicine,1320-159X,Journal of Law and Medicine,16227,FOUND IN SCOPUS
7,University of Western Australia Law Review,0042-0328,University of Western Australia Law Review,21101057019,FOUND IN SCOPUS
8,Sydney Law Review,0082-0512,Sydney Law Review,20345,FOUND IN SCOPUS
9,Melbourne University Law Review,0025-8938,Melbourne University Law Review,16183,FOUND IN SCOPUS



PACIFIC REGION LAW JOURNALS
✅ Confirmados no Scopus: 16 / 16


In [137]:
# ============================================================
# PACIFIC REGION LAW JOURNALS
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================


# ============================================================
# 1. SELECIONAR SOMENTE OS CONFIRMADOS
# ============================================================

confirmed_pacific_region_law = (
    scopus_pacific_region_law_df[
        scopus_pacific_region_law_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_pacific_region_law)}"
)


# ============================================================
# 2. CRIAR FILTRO DOS PERIÓDICOS POR ISSN
# ============================================================

issn_queries_pacific = []

for _, row in confirmed_pacific_region_law.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_pacific.append(
            f'ISSN({issn})'
        )

journals_part_pacific = " OR ".join(
    issn_queries_pacific
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_pacific)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_PACIFIC_TT = f'''
(
    {journals_part_pacific}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_PACIFIC_TT)


# ============================================================
# 4. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_PACIFIC_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_pacific_tt = response.json()

    total_results_pacific_tt = int(
        data_pacific_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_pacific_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 16
Periódicos utilizados na busca: 16

CONSULTA GERADA
(     ISSN(2652-4074) OR ISSN(0313-0096) OR ISSN(1837-5391) OR ISSN(1323-238X) OR ISSN(0084-7658) OR ISSN(2652-0036) OR ISSN(1320-159X) OR ISSN(0042-0328) OR ISSN(0082-0512) OR ISSN(0025-8938) OR ISSN(0065-1915) OR ISSN(1443-0738) OR ISSN(1832-911X) OR ISSN(0083-4041) OR ISSN(1833-2137) OR ISSN(1171-042X) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [138]:
# ============================================================
# GERAR EXCEL — TECHNOLOGY TRANSFER
# PACIFIC REGION LAW JOURNALS
# 2004–2026
# RESULTADO: 0
# ============================================================

# Results vazio
columns_pacific_tt = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_pacific_tt = pd.DataFrame(
    columns=columns_pacific_tt
)


# ============================================================
# SEARCH TERMS
# ============================================================

search_terms_pacific_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_pacific_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_pacific_region_law),
        "; ".join(
            confirmed_pacific_region_law[
                "Journal"
            ].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_pacific_tt,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_pacific_tt = (
    "Scopus_Pacific_Region_Law_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_pacific_tt,
    engine="openpyxl"
) as writer:

    df_pacific_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_pacific_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print(
    f"Número de periódicos: "
    f"{len(confirmed_pacific_region_law)}"
)
print("Idioma: English")
print("Número de expressões: 3")
print("Período: 2004–2026")

print("\nArquivo:")
print(filename_pacific_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Número de periódicos: 16
Idioma: English
Número de expressões: 3
Período: 2004–2026

Arquivo:
Scopus_Pacific_Region_Law_Technology_Transfer_2004_2026.xlsx


In [139]:
from google.colab import files

files.download(filename_pacific_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [140]:
# ============================================================
# PACIFIC REGION LAW JOURNALS
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_PACIFIC_TI = f'''
(
    {journals_part_pacific}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_PACIFIC_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_PACIFIC_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_pacific_ti = response.json()

    total_results_pacific_ti = int(
        data_pacific_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_pacific_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(2652-4074) OR ISSN(0313-0096) OR ISSN(1837-5391) OR ISSN(1323-238X) OR ISSN(0084-7658) OR ISSN(2652-0036) OR ISSN(1320-159X) OR ISSN(0042-0328) OR ISSN(0082-0512) OR ISSN(0025-8938) OR ISSN(0065-1915) OR ISSN(1443-0738) OR ISSN(1832-911X) OR ISSN(0083-4041) OR ISSN(1833-2137) OR ISSN(1171-042X) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [141]:
# ============================================================
# GERAR EXCEL — TECHNOLOGY INTERNALIZATION
# PACIFIC REGION LAW JOURNALS
# 2004–2026
# RESULTADO: 0
# ============================================================

# Results vazio
columns_pacific_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_pacific_ti = pd.DataFrame(
    columns=columns_pacific_ti
)


# ============================================================
# SEARCH TERMS
# ============================================================

search_terms_pacific_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_pacific_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_pacific_region_law),
        "; ".join(
            confirmed_pacific_region_law[
                "Journal"
            ].tolist()
        ),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_pacific_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_pacific_ti = (
    "Scopus_Pacific_Region_Law_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_pacific_ti,
    engine="openpyxl"
) as writer:

    df_pacific_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_pacific_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print(
    f"Número de periódicos: "
    f"{len(confirmed_pacific_region_law)}"
)
print("Idioma: English")
print("Número de expressões: 3")
print("Período: 2004–2026")

print("\nArquivo:")
print(filename_pacific_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Número de periódicos: 16
Idioma: English
Número de expressões: 3
Período: 2004–2026

Arquivo:
Scopus_Pacific_Region_Law_Technology_Internalization_2004_2026.xlsx


In [142]:
from google.colab import files

files.download(filename_pacific_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [144]:
# ============================================================
# MULTIDISCIPLINARY JOURNALS
# VERIFICAÇÃO NO SCOPUS — VERSÃO CORRIGIDA
# ============================================================

import requests
import pandas as pd
import time


# ============================================================
# 1. PERIÓDICOS + ISSNs
# ============================================================

multidisciplinary_journals = {

    "Government Information Quarterly": [
        "0740-624X",
        "1872-9517"
    ],

    "Journal of Business Ethics": [
        "0167-4544",
        "1573-0697"
    ],

    "Telematics and Informatics": [
        "0736-5853",
        "1879-324X"
    ],

    "Resources Policy": [
        "0301-4207",
        "1873-7641"
    ],

    "Cambridge Prisms: Coastal Futures": [
        "2754-7205"
    ],

    # CORRIGIDO
    "Life Medicine": [
        "2755-1733"
    ],

    "Computers and Security": [
        "0167-4048"
    ],

    "Regulation and Governance": [
        "1748-5983",
        "1748-5991"
    ],

    # CORRIGIDO
    "Europe and the World": [
        "2399-2875"
    ],

    "Transnational Environmental Law": [
        "2047-1025",
        "2047-1033"
    ],

    "Transport Policy": [
        "0967-070X",
        "1879-310X"
    ],

    "Justice Quarterly": [
        "0741-8825",
        "1745-9109"
    ],

    "Marine Policy": [
        "0308-597X",
        "1872-9460"
    ]
}


# ============================================================
# 2. CONTAGEM INICIAL
# ============================================================

print("============================================================")
print("MULTIDISCIPLINARY JOURNALS")
print("============================================================")

print(
    f"Total de periódicos cadastrados: "
    f"{len(multidisciplinary_journals)}"
)


# ============================================================
# 3. VERIFICAR CADA PERIÓDICO NO SCOPUS
# ============================================================

scopus_multidisciplinary = []

for journal, issns in multidisciplinary_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_multidisciplinary.append({

                    "Journal": journal,

                    "ISSN searched": issn,

                    "Scopus title": entry.get(
                        "dc:title",
                        ""
                    ),

                    "Source ID": entry.get(
                        "source-id",
                        ""
                    ),

                    "Publisher": entry.get(
                        "dc:publisher",
                        ""
                    ),

                    "Status": "FOUND IN SCOPUS"
                })

                found = True

                break

        time.sleep(0.2)


    if found:

        print(
            f"✅ Encontrado: {journal}"
        )

    else:

        scopus_multidisciplinary.append({

            "Journal": journal,

            "ISSN searched": "",

            "Scopus title": "",

            "Source ID": "",

            "Publisher": "",

            "Status": "NOT FOUND"
        })

        print(
            f"⚠️ Não encontrado: {journal}"
        )


# ============================================================
# 4. CRIAR DATAFRAME
# ============================================================

scopus_multidisciplinary_df = pd.DataFrame(
    scopus_multidisciplinary
)


# ============================================================
# 5. MOSTRAR TABELA
# ============================================================

display(
    scopus_multidisciplinary_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# 6. CONTAGEM FINAL
# ============================================================

total_confirmed_multidisciplinary = (

    scopus_multidisciplinary_df[
        "Status"
    ]

    == "FOUND IN SCOPUS"

).sum()


print("\n============================================================")
print("MULTIDISCIPLINARY JOURNALS")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_multidisciplinary} / "
    f"{len(multidisciplinary_journals)}"
)

print("============================================================")


# ============================================================
# 7. MOSTRAR EVENTUAIS NÃO ENCONTRADOS
# ============================================================

not_found_multidisciplinary = (
    scopus_multidisciplinary_df[
        scopus_multidisciplinary_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_multidisciplinary) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print(
        "\n⚠️ Periódicos que ainda precisam ser verificados:"
    )

    display(
        not_found_multidisciplinary[
            [
                "Journal",
                "Status"
            ]
        ]
    )

MULTIDISCIPLINARY JOURNALS
Total de periódicos cadastrados: 13
✅ Encontrado: Government Information Quarterly
✅ Encontrado: Journal of Business Ethics
✅ Encontrado: Telematics and Informatics
✅ Encontrado: Resources Policy
✅ Encontrado: Cambridge Prisms: Coastal Futures
✅ Encontrado: Life Medicine
✅ Encontrado: Computers and Security
✅ Encontrado: Regulation and Governance
✅ Encontrado: Europe and the World
✅ Encontrado: Transnational Environmental Law
✅ Encontrado: Transport Policy
✅ Encontrado: Justice Quarterly
✅ Encontrado: Marine Policy


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Government Information Quarterly,0740-624X,Government Information Quarterly,14735,FOUND IN SCOPUS
1,Journal of Business Ethics,0167-4544,Journal of Business Ethics,23859,FOUND IN SCOPUS
2,Telematics and Informatics,0736-5853,Telematics and Informatics,20896,FOUND IN SCOPUS
3,Resources Policy,0301-4207,Resources Policy,110031,FOUND IN SCOPUS
4,Cambridge Prisms: Coastal Futures,2754-7205,Cambridge Prisms: Coastal Futures,21101270281,FOUND IN SCOPUS
5,Life Medicine,2755-1733,Life Medicine,21101257044,FOUND IN SCOPUS
6,Computers and Security,0167-4048,Computers and Security,28898,FOUND IN SCOPUS
7,Regulation and Governance,1748-5983,Regulation and Governance,14000156253,FOUND IN SCOPUS
8,Europe and the World,2399-2875,Europe and the World,21101168024,FOUND IN SCOPUS
9,Transnational Environmental Law,2047-1025,Transnational Environmental Law,21100269613,FOUND IN SCOPUS



MULTIDISCIPLINARY JOURNALS
✅ Confirmados no Scopus: 13 / 13

✅ Todos os periódicos foram encontrados no Scopus.


In [145]:
# ============================================================
# MULTIDISCIPLINARY JOURNALS
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================


# ============================================================
# 1. SELECIONAR SOMENTE OS CONFIRMADOS
# ============================================================

confirmed_multidisciplinary = (
    scopus_multidisciplinary_df[
        scopus_multidisciplinary_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_multidisciplinary)}"
)


# ============================================================
# 2. CRIAR FILTRO DOS PERIÓDICOS POR ISSN
# ============================================================

issn_queries_multidisciplinary = []

for _, row in confirmed_multidisciplinary.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_multidisciplinary.append(
            f'ISSN({issn})'
        )

journals_part_multidisciplinary = " OR ".join(
    issn_queries_multidisciplinary
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_multidisciplinary)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_MULTIDISCIPLINARY_TT = f'''
(
    {journals_part_multidisciplinary}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_MULTIDISCIPLINARY_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_MULTIDISCIPLINARY_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_multidisciplinary_tt = response.json()

    total_results_multidisciplinary_tt = int(
        data_multidisciplinary_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_multidisciplinary_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 13
Periódicos utilizados na busca: 13

CONSULTA GERADA
(     ISSN(0740-624X) OR ISSN(0167-4544) OR ISSN(0736-5853) OR ISSN(0301-4207) OR ISSN(2754-7205) OR ISSN(2755-1733) OR ISSN(0167-4048) OR ISSN(1748-5983) OR ISSN(2399-2875) OR ISSN(2047-1025) OR ISSN(0967-070X) OR ISSN(0741-8825) OR ISSN(0308-597X) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 54


In [146]:
# ============================================================
# BAIXAR TODOS OS RESULTADOS
# MULTIDISCIPLINARY — TECHNOLOGY TRANSFER
# ============================================================

all_results_multidisciplinary_tt = []

count = 25
start = 0

while start < total_results_multidisciplinary_tt:

    params = {
        "query": SEARCH_QUERY_MULTIDISCIPLINARY_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_multidisciplinary_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_multidisciplinary_tt)} / "
        f"{total_results_multidisciplinary_tt}"
    )

    start += count
    time.sleep(1)


print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_multidisciplinary_tt)}"
)


# ============================================================
# CONFERÊNCIA — QUANTOS POR PERIÓDICO
# ============================================================

journals_retrieved = []

for item in all_results_multidisciplinary_tt:

    journals_retrieved.append(
        item.get(
            "prism:publicationName",
            ""
        )
    )

journal_check_multidisciplinary_tt = (
    pd.Series(journals_retrieved)
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

print("\n============================================================")
print("RESULTADOS POR PERIÓDICO")
print("============================================================")

display(
    journal_check_multidisciplinary_tt
)


# ============================================================
# CONFERÊNCIA DOS 54 TÍTULOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_multidisciplinary_tt,
    start=1
):

    print(
        f"\n{i}. "
        f"{item.get('dc:title', '')}"
    )

    print(
        "   Periódico:",
        item.get(
            "prism:publicationName",
            ""
        )
    )

    print(
        "   Data:",
        item.get(
            "prism:coverDate",
            ""
        )
    )

    print(
        "   DOI:",
        item.get(
            "prism:doi",
            ""
        )
    )

Baixados: 25 / 54
Baixados: 50 / 54
Baixados: 54 / 54

DOWNLOAD CONCLUÍDO
Total de registros baixados: 54

RESULTADOS POR PERIÓDICO


,Journal,Number of articles
0,Resources Policy,19
1,Marine Policy,15
2,Transport Policy,5
3,Telematics and Informatics,5
4,Journal of Business Ethics,3
5,Computers and Security,2
6,Transnational Environmental Law,2
7,Government Information Quarterly,2
8,Regulation and Governance,1



ARTIGOS RECUPERADOS

1. Mind the gap: Towards an equitable transfer of marine technology
   Periódico: Marine Policy
   Data: 2026-10-01
   DOI: 10.1016/j.marpol.2026.107182

2. Reframing environmental strategies in international shipping: A semantic network analysis of the shift toward clean technologies
   Periódico: Marine Policy
   Data: 2026-10-01
   DOI: 10.1016/j.marpol.2026.107185

3. Advancing marine policy in uncharted waters: Reflections and prospects from the launch of the BBNJ Symposium series
   Periódico: Marine Policy
   Data: 2026-08-01
   DOI: 10.1016/j.marpol.2026.107143

4. Older people's perceptions of emerging transport technologies and intentions to use them: Are mobility innovations like AVs and MaaS ageing-friendly?
   Periódico: Transport Policy
   Data: 2026-07-01
   DOI: 10.1016/j.tranpol.2026.104156

5. From connectivity to complexity: The influence of high-speed rail on urban knowledge complexity
   Periódico: Transport Policy
   Data: 2026-05-01
   DOI: 

In [147]:
# ============================================================
# ORGANIZAR E EXPORTAR RESULTADOS
# MULTIDISCIPLINARY JOURNALS — TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

records_multidisciplinary_tt = []

for item in all_results_multidisciplinary_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_multidisciplinary_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_multidisciplinary_tt = pd.DataFrame(
    records_multidisciplinary_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_multidisciplinary_tt = (
    df_multidisciplinary_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH TERMS
# ============================================================

search_terms_multidisciplinary_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_multidisciplinary_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_multidisciplinary),

        "; ".join(
            confirmed_multidisciplinary[
                "Journal"
            ].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_multidisciplinary_tt,
        "ISSN",
        len(df_multidisciplinary_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_multidisciplinary_tt = (
    "Scopus_Multidisciplinary_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_multidisciplinary_tt,
    engine="openpyxl"
) as writer:

    df_multidisciplinary_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_multidisciplinary_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_multidisciplinary_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# RESULTADO
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{len(df_multidisciplinary_tt)}"
)

print(
    f"Número de periódicos pesquisados: "
    f"{len(confirmed_multidisciplinary)}"
)

print("\nArtigos por periódico:")

display(
    journal_counts_multidisciplinary_tt
)

print("\nArquivo:")
print(filename_multidisciplinary_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 54
Número de periódicos pesquisados: 13

Artigos por periódico:


,Journal,Number of articles
0,Resources Policy,19
1,Marine Policy,15
2,Transport Policy,5
3,Telematics and Informatics,5
4,Journal of Business Ethics,3
5,Computers and Security,2
6,Transnational Environmental Law,2
7,Government Information Quarterly,2
8,Regulation and Governance,1



Arquivo:
Scopus_Multidisciplinary_Technology_Transfer_2004_2026.xlsx


In [148]:
from google.colab import files

files.download(
    filename_multidisciplinary_tt
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [149]:
# ============================================================
# MULTIDISCIPLINARY JOURNALS
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_MULTIDISCIPLINARY_TI = f'''
(
    {journals_part_multidisciplinary}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_MULTIDISCIPLINARY_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_MULTIDISCIPLINARY_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_multidisciplinary_ti = response.json()

    total_results_multidisciplinary_ti = int(
        data_multidisciplinary_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_multidisciplinary_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0740-624X) OR ISSN(0167-4544) OR ISSN(0736-5853) OR ISSN(0301-4207) OR ISSN(2754-7205) OR ISSN(2755-1733) OR ISSN(0167-4048) OR ISSN(1748-5983) OR ISSN(2399-2875) OR ISSN(2047-1025) OR ISSN(0967-070X) OR ISSN(0741-8825) OR ISSN(0308-597X) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [150]:
# ============================================================
# GERAR EXCEL — TECHNOLOGY INTERNALIZATION
# MULTIDISCIPLINARY JOURNALS
# 2004–2026
# RESULTADO: 0
# ============================================================

# Results vazio
columns_multidisciplinary_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_multidisciplinary_ti = pd.DataFrame(
    columns=columns_multidisciplinary_ti
)


# ============================================================
# SEARCH TERMS
# ============================================================

search_terms_multidisciplinary_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_multidisciplinary_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_multidisciplinary),

        "; ".join(
            confirmed_multidisciplinary[
                "Journal"
            ].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_multidisciplinary_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_multidisciplinary_ti = (
    "Scopus_Multidisciplinary_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_multidisciplinary_ti,
    engine="openpyxl"
) as writer:

    df_multidisciplinary_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_multidisciplinary_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# CONFIRMAÇÃO
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print(
    f"Número de periódicos: "
    f"{len(confirmed_multidisciplinary)}"
)
print("Idioma: English")
print("Número de expressões: 3")
print("Período: 2004–2026")

print("\nArquivo:")
print(filename_multidisciplinary_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Número de periódicos: 13
Idioma: English
Número de expressões: 3
Período: 2004–2026

Arquivo:
Scopus_Multidisciplinary_Technology_Internalization_2004_2026.xlsx


In [151]:
from google.colab import files

files.download(
    filename_multidisciplinary_ti
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [152]:
# ============================================================
# CHINA — ADDITIONAL JOURNALS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

import requests
import pandas as pd
import time


# ============================================================
# 1. PERIÓDICOS + ISSNs
# ============================================================

china_additional_journals = {

    "Global Public Policy and Governance": [
        "2730-6291",
        "2730-6305"
    ],

    "Asian Review of Political Economy": [
        "2731-5835"
    ],

    "World Regional Studies": [
        "1004-9479"
    ]
}


print("============================================================")
print("CHINA — ADDITIONAL JOURNALS")
print("============================================================")

print(
    f"Total de periódicos cadastrados: "
    f"{len(china_additional_journals)}"
)


# ============================================================
# 2. VERIFICAR CADA PERIÓDICO NO SCOPUS
# ============================================================

scopus_china_additional = []

for journal, issns in china_additional_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_china_additional.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get(
                        "dc:title", ""
                    ),
                    "Source ID": entry.get(
                        "source-id", ""
                    ),
                    "Publisher": entry.get(
                        "dc:publisher", ""
                    ),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_china_additional.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# 3. CRIAR DATAFRAME
# ============================================================

scopus_china_additional_df = pd.DataFrame(
    scopus_china_additional
)


# ============================================================
# 4. MOSTRAR RESULTADOS
# ============================================================

display(
    scopus_china_additional_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# 5. CONTAGEM FINAL
# ============================================================

total_confirmed_china_additional = (
    scopus_china_additional_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("CHINA — ADDITIONAL JOURNALS")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_china_additional} / "
    f"{len(china_additional_journals)}"
)

print("============================================================")


# ============================================================
# 6. EVENTUAIS NÃO ENCONTRADOS
# ============================================================

not_found_china_additional = (
    scopus_china_additional_df[
        scopus_china_additional_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_china_additional) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print(
        "\n⚠️ Periódicos que ainda precisam ser verificados:"
    )

    display(
        not_found_china_additional[
            ["Journal", "Status"]
        ]
    )

CHINA — ADDITIONAL JOURNALS
Total de periódicos cadastrados: 3
✅ Encontrado: Global Public Policy and Governance
✅ Encontrado: Asian Review of Political Economy
✅ Encontrado: World Regional Studies


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Global Public Policy and Governance,2730-6291,Global Public Policy and Governance,21101108743,FOUND IN SCOPUS
1,Asian Review of Political Economy,2731-5835,Asian Review of Political Economy,21101390133,FOUND IN SCOPUS
2,World Regional Studies,1004-9479,World Regional Studies,21101185482,FOUND IN SCOPUS



CHINA — ADDITIONAL JOURNALS
✅ Confirmados no Scopus: 3 / 3

✅ Todos os periódicos foram encontrados no Scopus.


In [153]:
# ============================================================
# CHINA — ADDITIONAL JOURNALS
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================


# ============================================================
# 1. SELECIONAR SOMENTE OS CONFIRMADOS
# ============================================================

confirmed_china_additional = (
    scopus_china_additional_df[
        scopus_china_additional_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_china_additional)}"
)


# ============================================================
# 2. CRIAR FILTRO DOS PERIÓDICOS POR ISSN
# ============================================================

issn_queries_china_additional = []

for _, row in confirmed_china_additional.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_china_additional.append(
            f'ISSN({issn})'
        )

journals_part_china_additional = " OR ".join(
    issn_queries_china_additional
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_china_additional)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_CHINA_ADDITIONAL_TT = f'''
(
    {journals_part_china_additional}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_CHINA_ADDITIONAL_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_CHINA_ADDITIONAL_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_china_additional_tt = response.json()

    total_results_china_additional_tt = int(
        data_china_additional_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_china_additional_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 3
Periódicos utilizados na busca: 3

CONSULTA GERADA
(     ISSN(2730-6291) OR ISSN(2731-5835) OR ISSN(1004-9479) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 5


In [154]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# CHINA — ADDITIONAL JOURNALS
# TECHNOLOGY TRANSFER
# ============================================================

all_results_china_additional_tt = []

count = 25
start = 0

while start < total_results_china_additional_tt:

    params = {
        "query": SEARCH_QUERY_CHINA_ADDITIONAL_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_china_additional_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_china_additional_tt)} / "
        f"{total_results_china_additional_tt}"
    )

    start += count
    time.sleep(1)


print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_china_additional_tt)}"
)


# ============================================================
# MOSTRAR OS ARTIGOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_china_additional_tt,
    start=1
):

    print(
        f"\n{i}. "
        f"{item.get('dc:title', '')}"
    )

    print(
        "   Periódico:",
        item.get(
            "prism:publicationName",
            ""
        )
    )

    print(
        "   ISSN:",
        item.get(
            "prism:issn",
            ""
        )
    )

    print(
        "   Data:",
        item.get(
            "prism:coverDate",
            ""
        )
    )

    print(
        "   DOI:",
        item.get(
            "prism:doi",
            ""
        )
    )

Baixados: 5 / 5

DOWNLOAD CONCLUÍDO
Total de registros baixados: 5

ARTIGOS RECUPERADOS

1. Trump, U.S.-China competition, and the future of technology transfer
   Periódico: Global Public Policy and Governance
   ISSN: 27306291
   Data: 2025-12-01
   DOI: 10.1007/s43508-025-00125-9

2. Research on the network evolution of technology innovation flow in the central cities of the Guangdong-Hong Kong-Macao Greater Bay Area: A case of Shenzhen
   Periódico: World Regional Studies
   ISSN: 10049479
   Data: 2024-11-15
   DOI: 10.3969/j.issn.1004-9479.2024.11.20222497

3. The evolution pattern and endogenous mechanism of China's innovation network: From the "glocal" perspective
   Periódico: World Regional Studies
   ISSN: 10049479
   Data: 2024-09-15
   DOI: 10.3969/j.issn.1004-9479.2024.09.20230048

4. Research on technological innovation flow in Guangdong-Hong Kong-Macao Greater Bay Area based on patent measurement
   Periódico: World Regional Studies
   ISSN: 10049479
   Data: 2023-12-15

In [155]:
# ============================================================
# ORGANIZAR E EXPORTAR RESULTADOS
# CHINA — ADDITIONAL JOURNALS
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

records_china_additional_tt = []

for item in all_results_china_additional_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_china_additional_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })

df_china_additional_tt = pd.DataFrame(
    records_china_additional_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_china_additional_tt = (
    df_china_additional_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_china_additional_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_china_additional_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_china_additional),

        "; ".join(
            confirmed_china_additional[
                "Journal"
            ].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_china_additional_tt,
        "ISSN",
        len(df_china_additional_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_china_additional_tt = (
    "Scopus_China_Additional_Journals_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_china_additional_tt,
    engine="openpyxl"
) as writer:

    df_china_additional_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_china_additional_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_china_additional_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# RESULTADO
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{len(df_china_additional_tt)}"
)

print("\nArtigos por periódico:")
display(journal_counts_china_additional_tt)

print("\nArquivo:")
print(filename_china_additional_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 5

Artigos por periódico:


,Journal,Number of articles
0,World Regional Studies,4
1,Global Public Policy and Governance,1



Arquivo:
Scopus_China_Additional_Journals_Technology_Transfer_2004_2026.xlsx


In [156]:
from google.colab import files

files.download(filename_china_additional_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [157]:
# ============================================================
# CHINA — ADDITIONAL JOURNALS
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_CHINA_ADDITIONAL_TI = f'''
(
    {journals_part_china_additional}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_CHINA_ADDITIONAL_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_CHINA_ADDITIONAL_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)

if response.status_code == 200:

    data_china_additional_ti = response.json()

    total_results_china_additional_ti = int(
        data_china_additional_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_china_additional_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(2730-6291) OR ISSN(2731-5835) OR ISSN(1004-9479) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [158]:
# ============================================================
# GERAR EXCEL — TECHNOLOGY INTERNALIZATION
# CHINA — ADDITIONAL JOURNALS
# 2004–2026
# RESULTADO: 0
# ============================================================

# Results vazio
columns_china_additional_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_china_additional_ti = pd.DataFrame(
    columns=columns_china_additional_ti
)


# ============================================================
# SEARCH TERMS
# ============================================================

search_terms_china_additional_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_china_additional_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_china_additional),

        "; ".join(
            confirmed_china_additional[
                "Journal"
            ].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_china_additional_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_china_additional_ti = (
    "Scopus_China_Additional_Journals_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_china_additional_ti,
    engine="openpyxl"
) as writer:

    df_china_additional_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_china_additional_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# CONFIRMAÇÃO
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print(
    f"Número de periódicos pesquisados: "
    f"{len(confirmed_china_additional)}"
)
print("Período: 2004–2026")
print("Document type: Article")
print("Search field: TITLE-ABS-KEY")
print("Número de expressões: 3")

print("\nArquivo:")
print(filename_china_additional_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Número de periódicos pesquisados: 3
Período: 2004–2026
Document type: Article
Search field: TITLE-ABS-KEY
Número de expressões: 3

Arquivo:
Scopus_China_Additional_Journals_Technology_Internalization_2004_2026.xlsx


In [159]:
from google.colab import files

files.download(
    filename_china_additional_ti
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [160]:
# ============================================================
# AFRICA REGION
# VERIFICAÇÃO DOS PERIÓDICOS NO SCOPUS
# ============================================================

africa_region_journals = {
    "Africa Development/Afrique et Developpement": [
        "0850-3907"
    ]
}


# ============================================================
# VERIFICAR NO SCOPUS
# ============================================================

scopus_africa_region = []

for journal, issns in africa_region_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_africa_region.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_africa_region.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# DATAFRAME
# ============================================================

scopus_africa_region_df = pd.DataFrame(
    scopus_africa_region
)

display(
    scopus_africa_region_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# CONTAGEM
# ============================================================

total_confirmed_africa_region = (
    scopus_africa_region_df["Status"]
    == "FOUND IN SCOPUS"
).sum()

print("\n============================================================")
print("AFRICA REGION")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_africa_region} / "
    f"{len(africa_region_journals)}"
)

✅ Encontrado: Africa Development/Afrique et Developpement


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Africa Development/Afrique et Developpement,0850-3907,Africa Development/Afrique et Developpement,13474,FOUND IN SCOPUS



AFRICA REGION
✅ Confirmados no Scopus: 1 / 1


In [161]:
# ============================================================
# AFRICA REGION
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

# 1. Selecionar periódico confirmado
confirmed_africa_region = (
    scopus_africa_region_df[
        scopus_africa_region_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)


# 2. Criar filtro por ISSN
issn_queries_africa_region = []

for _, row in confirmed_africa_region.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_africa_region.append(
            f'ISSN({issn})'
        )

journals_part_africa_region = " OR ".join(
    issn_queries_africa_region
)


# 3. Criar consulta
SEARCH_QUERY_AFRICA_REGION_TT = f'''
(
    {journals_part_africa_region}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# 4. Mostrar consulta
print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_AFRICA_REGION_TT)


# 5. Executar consulta
params = {
    "query": SEARCH_QUERY_AFRICA_REGION_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_africa_region_tt = response.json()

    total_results_africa_region_tt = int(
        data_africa_region_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_africa_region_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0850-3907) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 2


In [163]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# AFRICA REGION
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

all_results_africa_region_tt = []

count = 25
start = 0

while start < total_results_africa_region_tt:

    params = {
        "query": SEARCH_QUERY_AFRICA_REGION_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_africa_region_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_africa_region_tt)} / "
        f"{total_results_africa_region_tt}"
    )

    start += count
    time.sleep(1)


# ============================================================
# CONFERÊNCIA
# ============================================================

print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_africa_region_tt)}"
)


print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_africa_region_tt,
    start=1
):

    print(
        f"\n{i}. {item.get('dc:title', '')}"
    )

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   ISSN:",
        item.get("prism:issn", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

    print(
        "   DOI:",
        item.get("prism:doi", "")
    )

Baixados: 2 / 2

DOWNLOAD CONCLUÍDO
Total de registros baixados: 2

ARTIGOS RECUPERADOS

1. Investissements directs étrangers au Maroc: Impact sur la productivité totale des facteurs selon le pays d’origine (1980-2012)
   Periódico: Africa Development
   ISSN: 08503907
   Data: 2016-01-01
   DOI: 

2. Technology transfer, foreign direct investment and economic growth in Nigeria
   Periódico: Africa Development
   ISSN: 08503907
   Data: 2014-01-01
   DOI: 


In [164]:
# ============================================================
# ORGANIZAR E EXPORTAR RESULTADOS
# AFRICA REGION — TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

records_africa_region_tt = []

for item in all_results_africa_region_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_africa_region_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_africa_region_tt = pd.DataFrame(
    records_africa_region_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_africa_region_tt = (
    df_africa_region_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_africa_region_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_africa_region_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_africa_region),

        "; ".join(
            confirmed_africa_region[
                "Journal"
            ].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_africa_region_tt,
        "ISSN",
        len(df_africa_region_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_africa_region_tt = (
    "Scopus_Africa_Region_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_africa_region_tt,
    engine="openpyxl"
) as writer:

    df_africa_region_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_africa_region_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_africa_region_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# CONFERÊNCIA FINAL
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{len(df_africa_region_tt)}"
)

print("\nArtigos por periódico:")
display(journal_counts_africa_region_tt)

print("\nArquivo:")
print(filename_africa_region_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 2

Artigos por periódico:


,Journal,Number of articles
0,Africa Development,2



Arquivo:
Scopus_Africa_Region_Technology_Transfer_2004_2026.xlsx


In [165]:
from google.colab import files

files.download(filename_africa_region_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [166]:
# ============================================================
# AFRICA REGION
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_AFRICA_REGION_TI = f'''
(
    {journals_part_africa_region}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_AFRICA_REGION_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_AFRICA_REGION_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_africa_region_ti = response.json()

    total_results_africa_region_ti = int(
        data_africa_region_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_africa_region_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0850-3907) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [167]:
# ============================================================
# AFRICA REGION — TECHNOLOGY INTERNALIZATION
# RESULTADO: 0
# ============================================================

columns_africa_region_ti = [
    "Journal", "Title", "Authors", "Year", "ISSN",
    "Volume", "Issue", "Pages", "DOI", "EID",
    "Cited by", "Scopus URL"
]

df_africa_region_ti = pd.DataFrame(
    columns=columns_africa_region_ti
)

search_terms_africa_region_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)

search_log_africa_region_ti = pd.DataFrame({
    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],
    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_africa_region),
        "; ".join(confirmed_africa_region["Journal"].tolist()),
        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_africa_region_ti,
        "ISSN",
        0
    ]
})

filename_africa_region_ti = (
    "Scopus_Africa_Region_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_africa_region_ti,
    engine="openpyxl"
) as writer:

    df_africa_region_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_africa_region_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ EXCEL CRIADO COM SUCESSO")
print("Total de artigos: 0")
print(filename_africa_region_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Scopus_Africa_Region_Technology_Internalization_2004_2026.xlsx


In [168]:
from google.colab import files

files.download(filename_africa_region_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [169]:
# ============================================================
# EASTERN EUROPE REGION
# VERIFICAÇÃO DOS PERIÓDICOS NO SCOPUS
# ============================================================

eastern_europe_journals = {

    "Entrepreneurial Business and Economics Review": [
        "2353-883X", "2353-8821"
    ],

    "Czech Journal of International Relations": [
        "0323-1844", "2570-9429"
    ],

    "Central European Management Journal": [
        "2658-0845", "2658-2430"
    ],

    "New Perspectives": [
        "2336-825X", "2336-8268"
    ],

    "Central European Journal of International and Security Studies": [
        "1802-548X"
    ],

    "RUDN Journal of Russian History": [
        "2312-8674", "2312-8690"
    ],

    "International Organisations Research Journal": [
        "1996-7845", "2542-2081"
    ],

    "Journal of International Studies": [
        "2071-8330", "2306-3483"
    ],

    "Journal of Comparative Politics": [
        "1338-1385"
    ],

    "Politologicky Casopis": [
        "1211-3247", "1805-9503"
    ],

    "Journal of Regional Security": [
        "2217-995X", "2406-0364"
    ],

    "Society Register": [
        "2544-5502"
    ],

    "World Economy and International Relations": [
        "0131-2227"
    ],

    "Transformations in Business and Economics": [
        "1648-4460"
    ],

    "Russia in Global Affairs": [
        "1810-6374"
    ],

    "Eminak": [
        "1998-4634", "2708-0226"
    ]
}


# ============================================================
# CONSULTAR SCOPUS SERIAL TITLE API
# ============================================================

scopus_eastern_europe = []

for journal, issns in eastern_europe_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_eastern_europe.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_eastern_europe.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# DATAFRAME
# ============================================================

scopus_eastern_europe_df = pd.DataFrame(
    scopus_eastern_europe
)


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

display(
    scopus_eastern_europe_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# CONTAGEM
# ============================================================

total_confirmed_eastern_europe = (
    scopus_eastern_europe_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("EASTERN EUROPE REGION")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_eastern_europe} / "
    f"{len(eastern_europe_journals)}"
)


# ============================================================
# NÃO ENCONTRADOS
# ============================================================

not_found_eastern_europe = (
    scopus_eastern_europe_df[
        scopus_eastern_europe_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_eastern_europe) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print(
        "\n⚠️ Periódicos que precisam de verificação:"
    )

    display(
        not_found_eastern_europe[
            ["Journal", "Status"]
        ]
    )

✅ Encontrado: Entrepreneurial Business and Economics Review
✅ Encontrado: Czech Journal of International Relations
✅ Encontrado: Central European Management Journal
✅ Encontrado: New Perspectives
✅ Encontrado: Central European Journal of International and Security Studies
✅ Encontrado: RUDN Journal of Russian History
✅ Encontrado: International Organisations Research Journal
✅ Encontrado: Journal of International Studies
✅ Encontrado: Journal of Comparative Politics
✅ Encontrado: Politologicky Casopis
✅ Encontrado: Journal of Regional Security
✅ Encontrado: Society Register
✅ Encontrado: World Economy and International Relations
✅ Encontrado: Transformations in Business and Economics
✅ Encontrado: Russia in Global Affairs
✅ Encontrado: Eminak


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Entrepreneurial Business and Economics Review,2353-883X,Entrepreneurial Business and Economics Review,21100857389,FOUND IN SCOPUS
1,Czech Journal of International Relations,0323-1844,Mezinarodni Vztahy,19700170036,FOUND IN SCOPUS
2,Central European Management Journal,2658-0845,Central European Management Journal,21101039079,FOUND IN SCOPUS
3,New Perspectives,2336-825X,New Perspectives,6000177137,FOUND IN SCOPUS
4,Central European Journal of International and ...,1802-548X,Central European Journal of International and ...,21100225612,FOUND IN SCOPUS
5,RUDN Journal of Russian History,2312-8674,RUDN Journal of Russian History,21101037141,FOUND IN SCOPUS
6,International Organisations Research Journal,1996-7845,International Organisations Research Journal,21100820733,FOUND IN SCOPUS
7,Journal of International Studies,2071-8330,Journal of International Studies,21100829274,FOUND IN SCOPUS
8,Journal of Comparative Politics,1338-1385,Journal of Comparative Politics,21100846340,FOUND IN SCOPUS
9,Politologicky Casopis,1211-3247,Politologicky Casopis,5700168238,FOUND IN SCOPUS



EASTERN EUROPE REGION
✅ Confirmados no Scopus: 16 / 16

✅ Todos os periódicos foram encontrados no Scopus.


In [170]:
# ============================================================
# EASTERN EUROPE REGION
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

# 1. Selecionar periódicos confirmados
confirmed_eastern_europe = (
    scopus_eastern_europe_df[
        scopus_eastern_europe_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_eastern_europe)}"
)


# ============================================================
# 2. CRIAR FILTRO POR ISSN
# ============================================================

issn_queries_eastern_europe = []

for _, row in confirmed_eastern_europe.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_eastern_europe.append(
            f'ISSN({issn})'
        )

journals_part_eastern_europe = " OR ".join(
    issn_queries_eastern_europe
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_eastern_europe)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_EASTERN_EUROPE_TT = f'''
(
    {journals_part_eastern_europe}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_EASTERN_EUROPE_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_EASTERN_EUROPE_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)


if response.status_code == 200:

    data_eastern_europe_tt = response.json()

    total_results_eastern_europe_tt = int(
        data_eastern_europe_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_eastern_europe_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 16
Periódicos utilizados na busca: 16

CONSULTA GERADA
(     ISSN(2353-883X) OR ISSN(0323-1844) OR ISSN(2658-0845) OR ISSN(2336-825X) OR ISSN(1802-548X) OR ISSN(2312-8674) OR ISSN(1996-7845) OR ISSN(2071-8330) OR ISSN(1338-1385) OR ISSN(1211-3247) OR ISSN(2217-995X) OR ISSN(2544-5502) OR ISSN(0131-2227) OR ISSN(1648-4460) OR ISSN(1810-6374) OR ISSN(1998-4634) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 19


In [171]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# EASTERN EUROPE REGION
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

all_results_eastern_europe_tt = []

count = 25
start = 0

while start < total_results_eastern_europe_tt:

    params = {
        "query": SEARCH_QUERY_EASTERN_EUROPE_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_eastern_europe_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_eastern_europe_tt)} / "
        f"{total_results_eastern_europe_tt}"
    )

    start += count
    time.sleep(1)


# ============================================================
# CONFERÊNCIA DO TOTAL
# ============================================================

print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_eastern_europe_tt)}"
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journals_retrieved_eastern_europe_tt = [
    item.get("prism:publicationName", "")
    for item in all_results_eastern_europe_tt
]

journal_check_eastern_europe_tt = (
    pd.Series(journals_retrieved_eastern_europe_tt)
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

print("\n============================================================")
print("RESULTADOS POR PERIÓDICO")
print("============================================================")

display(journal_check_eastern_europe_tt)


# ============================================================
# MOSTRAR OS 19 REGISTROS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_eastern_europe_tt,
    start=1
):

    print(f"\n{i}. {item.get('dc:title', '')}")

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

    print(
        "   DOI:",
        item.get("prism:doi", "")
    )

Baixados: 19 / 19

DOWNLOAD CONCLUÍDO
Total de registros baixados: 19

RESULTADOS POR PERIÓDICO


,Journal,Number of articles
0,World Economy and International Relations,10
1,Journal of International Studies,3
2,International Organisations Research Journal,3
3,Entrepreneurial Business and Economics Review,2
4,Transformations in Business and Economics,1



ARTIGOS RECUPERADOS

1. Foreign direct investment and multidimensional technological innovation: Empirical evidence from five East Asian economies (1996–2025)
   Periódico: Journal of International Studies
   Data: 2026-01-01
   DOI: 10.14254/2071-8330.2026/19-2/4

2. Countering Political Risks for Transnational ICT Corporations in the Global South With Official Development Cooperation Tools
   Periódico: International Organisations Research Journal
   Data: 2025-01-01
   DOI: 10.17323/1996-7845-2025-02-04

3. Impact of liberalization and globalization on economic growth: Empirical evidence from Türkiye
   Periódico: Journal of International Studies
   Data: 2024-01-01
   DOI: 10.14254/2071-8330.2024/17-1/3

4. TRADE-ECONOMIC AND SCIENTIFIC-TECHNICAL RELATIONS BETWEEN THE USA AND THE USSR DURING THE YEARS OF DETENTE (Review of Foreign Studies, 1972–2023)
   Periódico: World Economy and International Relations
   Data: 2024-01-01
   DOI: 10.20542/0131-2227-2024-68-5-125-141

5. MODEL “

In [172]:
# ============================================================
# EXPORTAR — EASTERN EUROPE REGION
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

records_eastern_europe_tt = []

for item in all_results_eastern_europe_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_eastern_europe_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_eastern_europe_tt = pd.DataFrame(
    records_eastern_europe_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_eastern_europe_tt = (
    df_eastern_europe_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_eastern_europe_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_eastern_europe_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        len(confirmed_eastern_europe),

        "; ".join(
            confirmed_eastern_europe["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_eastern_europe_tt,
        "ISSN",
        len(df_eastern_europe_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_eastern_europe_tt = (
    "Scopus_Eastern_Europe_Region_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_eastern_europe_tt,
    engine="openpyxl"
) as writer:

    df_eastern_europe_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_eastern_europe_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_eastern_europe_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(f"Total de artigos: {len(df_eastern_europe_tt)}")
print(f"Periódicos pesquisados: {len(confirmed_eastern_europe)}")

print("\nArtigos por periódico:")
display(journal_counts_eastern_europe_tt)

print("\nArquivo:")
print(filename_eastern_europe_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 19
Periódicos pesquisados: 16

Artigos por periódico:


,Journal,Number of articles
0,World Economy and International Relations,10
1,Journal of International Studies,3
2,International Organisations Research Journal,3
3,Entrepreneurial Business and Economics Review,2
4,Transformations in Business and Economics,1



Arquivo:
Scopus_Eastern_Europe_Region_Technology_Transfer_2004_2026.xlsx


In [173]:
from google.colab import files

files.download(filename_eastern_europe_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [174]:
# ============================================================
# EASTERN EUROPE REGION
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_EASTERN_EUROPE_TI = f'''
(
    {journals_part_eastern_europe}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_EASTERN_EUROPE_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_EASTERN_EUROPE_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_eastern_europe_ti = response.json()

    total_results_eastern_europe_ti = int(
        data_eastern_europe_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_eastern_europe_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(2353-883X) OR ISSN(0323-1844) OR ISSN(2658-0845) OR ISSN(2336-825X) OR ISSN(1802-548X) OR ISSN(2312-8674) OR ISSN(1996-7845) OR ISSN(2071-8330) OR ISSN(1338-1385) OR ISSN(1211-3247) OR ISSN(2217-995X) OR ISSN(2544-5502) OR ISSN(0131-2227) OR ISSN(1648-4460) OR ISSN(1810-6374) OR ISSN(1998-4634) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [175]:
# ============================================================
# EASTERN EUROPE REGION
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# RESULTADO: 0
# ============================================================

# Criar aba Results vazia
columns_eastern_europe_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_eastern_europe_ti = pd.DataFrame(
    columns=columns_eastern_europe_ti
)


# ============================================================
# TERMOS DE BUSCA
# ============================================================

search_terms_eastern_europe_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_eastern_europe_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),

        len(confirmed_eastern_europe),

        "; ".join(
            confirmed_eastern_europe[
                "Journal"
            ].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_eastern_europe_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR PARA EXCEL
# ============================================================

filename_eastern_europe_ti = (
    "Scopus_Eastern_Europe_Region_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_eastern_europe_ti,
    engine="openpyxl"
) as writer:

    df_eastern_europe_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_eastern_europe_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# CONFERÊNCIA
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")

print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_eastern_europe)}"
)

print("Período: 2004–2026")
print("Document type: Article")
print("Search field: TITLE-ABS-KEY")
print("Número de expressões: 3")

print("\nArquivo:")
print(filename_eastern_europe_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Periódicos pesquisados: 16
Período: 2004–2026
Document type: Article
Search field: TITLE-ABS-KEY
Número de expressões: 3

Arquivo:
Scopus_Eastern_Europe_Region_Technology_Internalization_2004_2026.xlsx


In [176]:
from google.colab import files

files.download(
    filename_eastern_europe_ti
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [177]:
# ============================================================
# PACIFIC REGION
# POLITICAL SCIENCE AND INTERNATIONAL RELATIONS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

pacific_psir_journals = {

    "Asian Economic Policy Review": [
        "1832-8105",
        "1748-3131"
    ],

    "Asian Journal of Business Research": [
        "2463-4522",
        "1178-8933"
    ],

    "Australian and New Zealand Journal of European Studies": [
        "1837-2147"
    ],

    "Landfall": [
        "0023-7930"
    ]
}


# ============================================================
# VERIFICAR NO SCOPUS
# ============================================================

scopus_pacific_psir = []

for journal, issns in pacific_psir_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_pacific_psir.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get(
                        "dc:title", ""
                    ),
                    "Source ID": entry.get(
                        "source-id", ""
                    ),
                    "Publisher": entry.get(
                        "dc:publisher", ""
                    ),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_pacific_psir.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# DATAFRAME
# ============================================================

scopus_pacific_psir_df = pd.DataFrame(
    scopus_pacific_psir
)


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

display(
    scopus_pacific_psir_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# CONTAGEM FINAL
# ============================================================

total_confirmed_pacific_psir = (
    scopus_pacific_psir_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("PACIFIC REGION — POLITICAL SCIENCE / IR")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_pacific_psir} / "
    f"{len(pacific_psir_journals)}"
)


# ============================================================
# NÃO ENCONTRADOS
# ============================================================

not_found_pacific_psir = (
    scopus_pacific_psir_df[
        scopus_pacific_psir_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_pacific_psir) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print(
        "\n⚠️ Periódicos que precisam de verificação:"
    )

    display(
        not_found_pacific_psir[
            ["Journal", "Status"]
        ]
    )

✅ Encontrado: Asian Economic Policy Review
✅ Encontrado: Asian Journal of Business Research
✅ Encontrado: Australian and New Zealand Journal of European Studies
✅ Encontrado: Landfall


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Asian Economic Policy Review,1832-8105,Asian Economic Policy Review,17600155050,FOUND IN SCOPUS
1,Asian Journal of Business Research,2463-4522,Asian Journal of Business Research,21100893575,FOUND IN SCOPUS
2,Australian and New Zealand Journal of European...,1837-2147,Australian and New Zealand Journal of European...,21101266476,FOUND IN SCOPUS
3,Landfall,0023-7930,Landfall,27128,FOUND IN SCOPUS



PACIFIC REGION — POLITICAL SCIENCE / IR
✅ Confirmados no Scopus: 4 / 4

✅ Todos os periódicos foram encontrados no Scopus.


In [178]:
# ============================================================
# PACIFIC REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

# 1. Selecionar periódicos confirmados
confirmed_pacific_psir = (
    scopus_pacific_psir_df[
        scopus_pacific_psir_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_pacific_psir)}"
)


# ============================================================
# 2. CRIAR FILTRO POR ISSN
# ============================================================

issn_queries_pacific_psir = []

for _, row in confirmed_pacific_psir.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_pacific_psir.append(
            f'ISSN({issn})'
        )

journals_part_pacific_psir = " OR ".join(
    issn_queries_pacific_psir
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_pacific_psir)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_PACIFIC_PSIR_TT = f'''
(
    {journals_part_pacific_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_PACIFIC_PSIR_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_PACIFIC_PSIR_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print(
    "Status da API:",
    response.status_code
)


if response.status_code == 200:

    data_pacific_psir_tt = response.json()

    total_results_pacific_psir_tt = int(
        data_pacific_psir_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_pacific_psir_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 4
Periódicos utilizados na busca: 4

CONSULTA GERADA
(     ISSN(1832-8105) OR ISSN(2463-4522) OR ISSN(1837-2147) OR ISSN(0023-7930) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 3


In [179]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# PACIFIC REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

all_results_pacific_psir_tt = []

count = 25
start = 0

while start < total_results_pacific_psir_tt:

    params = {
        "query": SEARCH_QUERY_PACIFIC_PSIR_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_pacific_psir_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_pacific_psir_tt)} / "
        f"{total_results_pacific_psir_tt}"
    )

    start += count
    time.sleep(1)


# ============================================================
# CONFERÊNCIA
# ============================================================

print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_pacific_psir_tt)}"
)


# ============================================================
# MOSTRAR OS 3 ARTIGOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_pacific_psir_tt,
    start=1
):

    print(f"\n{i}. {item.get('dc:title', '')}")

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   ISSN:",
        item.get("prism:issn", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

    print(
        "   DOI:",
        item.get("prism:doi", "")
    )

Baixados: 3 / 3

DOWNLOAD CONCLUÍDO
Total de registros baixados: 3

ARTIGOS RECUPERADOS

1. Measuring and Explaining Innovative Capability: Evidence from Southeast Asia
   Periódico: Asian Economic Policy Review
   ISSN: 18328105
   Data: 2015-01-01
   DOI: 10.1111/aepr.12093

2. Overseas Impacts of China's Outward Direct Investment
   Periódico: Asian Economic Policy Review
   ISSN: 18328105
   Data: 2014-01-01
   DOI: 10.1111/aepr.12065

3. Reaching a Global Agreement on Climate Change: What are the Obstacles?
   Periódico: Asian Economic Policy Review
   ISSN: 18328105
   Data: 2010-06-01
   DOI: 10.1111/j.1748-3131.2010.01144.x


In [180]:
# ============================================================
# EXPORTAR RESULTADOS
# PACIFIC REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

records_pacific_psir_tt = []

for item in all_results_pacific_psir_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_pacific_psir_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_pacific_psir_tt = pd.DataFrame(
    records_pacific_psir_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_pacific_psir_tt = (
    df_pacific_psir_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_pacific_psir_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_pacific_psir_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Pacific Region",
        "Political Science and International Relations",
        len(confirmed_pacific_psir),

        "; ".join(
            confirmed_pacific_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_pacific_psir_tt,
        "ISSN",
        len(df_pacific_psir_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_pacific_psir_tt = (
    "Scopus_Pacific_Region_PSIR_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_pacific_psir_tt,
    engine="openpyxl"
) as writer:

    df_pacific_psir_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_pacific_psir_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_pacific_psir_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(f"Total de artigos: {len(df_pacific_psir_tt)}")
print(f"Periódicos pesquisados: {len(confirmed_pacific_psir)}")

print("\nArtigos por periódico:")
display(journal_counts_pacific_psir_tt)

print("\nArquivo:")
print(filename_pacific_psir_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 3
Periódicos pesquisados: 4

Artigos por periódico:


,Journal,Number of articles
0,Asian Economic Policy Review,3



Arquivo:
Scopus_Pacific_Region_PSIR_Technology_Transfer_2004_2026.xlsx


In [181]:
from google.colab import files

files.download(filename_pacific_psir_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [182]:
# ============================================================
# PACIFIC REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_PACIFIC_PSIR_TI = f'''
(
    {journals_part_pacific_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_PACIFIC_PSIR_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_PACIFIC_PSIR_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_pacific_psir_ti = response.json()

    total_results_pacific_psir_ti = int(
        data_pacific_psir_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_pacific_psir_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(1832-8105) OR ISSN(2463-4522) OR ISSN(1837-2147) OR ISSN(0023-7930) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [183]:
# ============================================================
# PACIFIC REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION
# 2004–2026 — RESULTADO: 0
# ============================================================

columns_pacific_psir_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_pacific_psir_ti = pd.DataFrame(
    columns=columns_pacific_psir_ti
)


# ============================================================
# TERMOS DE BUSCA
# ============================================================

search_terms_pacific_psir_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)


# ============================================================
# SEARCH LOG
# ============================================================

search_log_pacific_psir_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Pacific Region",
        "Political Science and International Relations",
        len(confirmed_pacific_psir),

        "; ".join(
            confirmed_pacific_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_pacific_psir_ti,
        "ISSN",
        0
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_pacific_psir_ti = (
    "Scopus_Pacific_Region_PSIR_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_pacific_psir_ti,
    engine="openpyxl"
) as writer:

    df_pacific_psir_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_pacific_psir_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


# ============================================================
# CONFERÊNCIA
# ============================================================

print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print("Total de artigos: 0")
print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_pacific_psir)}"
)

print("\nArquivo:")
print(filename_pacific_psir_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Periódicos pesquisados: 4

Arquivo:
Scopus_Pacific_Region_PSIR_Technology_Internalization_2004_2026.xlsx


In [184]:
from google.colab import files

files.download(
    "Scopus_Pacific_Region_PSIR_Technology_Internalization_2004_2026.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [185]:
# ============================================================
# WESTERN EUROPE REGION
# POLITICAL SCIENCE AND INTERNATIONAL RELATIONS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

western_europe_psir_journals = {

    "American Journal of Political Science": [
        "0092-5853", "1540-5907"
    ],

    "American Political Science Review": [
        "0003-0554", "1537-5943"
    ],

    "Political Analysis": [
        "1047-1987", "1476-4989"
    ],

    "British Journal of Political Science": [
        "0007-1234", "1469-2112"
    ],

    "West European Politics": [
        "0140-2382", "1743-9655"
    ],

    "Journal of European Public Policy": [
        "1350-1763", "1466-4429"
    ],

    "Philosophy and Public Affairs": [
        "0048-3915", "1088-4963"
    ],

    "Democratization": [
        "1351-0347", "1743-890X"
    ],

    "New Political Economy": [
        "1356-3467", "1469-9923"
    ],

    "Journal of Common Market Studies": [
        "0021-9886", "1468-5965"
    ],

    "European Political Science Review": [
        "1755-7739", "1755-7747"
    ]
}


# ============================================================
# VERIFICAR NO SCOPUS
# ============================================================

scopus_western_europe_psir = []

for journal, issns in western_europe_psir_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_western_europe_psir.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_western_europe_psir.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# DATAFRAME
# ============================================================

scopus_western_europe_psir_df = pd.DataFrame(
    scopus_western_europe_psir
)


# ============================================================
# MOSTRAR RESULTADOS
# ============================================================

display(
    scopus_western_europe_psir_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# CONTAGEM FINAL
# ============================================================

total_confirmed_western_europe_psir = (
    scopus_western_europe_psir_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("WESTERN EUROPE REGION — POLITICAL SCIENCE / IR")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_western_europe_psir} / "
    f"{len(western_europe_psir_journals)}"
)


# ============================================================
# NÃO ENCONTRADOS
# ============================================================

not_found_western_europe_psir = (
    scopus_western_europe_psir_df[
        scopus_western_europe_psir_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_western_europe_psir) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print(
        "\n⚠️ Periódicos que precisam de verificação:"
    )

    display(
        not_found_western_europe_psir[
            ["Journal", "Status"]
        ]
    )

✅ Encontrado: American Journal of Political Science
✅ Encontrado: American Political Science Review
✅ Encontrado: Political Analysis
✅ Encontrado: British Journal of Political Science
✅ Encontrado: West European Politics
✅ Encontrado: Journal of European Public Policy
✅ Encontrado: Philosophy and Public Affairs
✅ Encontrado: Democratization
✅ Encontrado: New Political Economy
✅ Encontrado: Journal of Common Market Studies
✅ Encontrado: European Political Science Review


,Journal,ISSN searched,Scopus title,Source ID,Status
0,American Journal of Political Science,0092-5853,American Journal of Political Science,15555,FOUND IN SCOPUS
1,American Political Science Review,0003-0554,American Political Science Review,15557,FOUND IN SCOPUS
2,Political Analysis,1047-1987,Political Analysis,145558,FOUND IN SCOPUS
3,British Journal of Political Science,0007-1234,British Journal of Political Science,16232,FOUND IN SCOPUS
4,West European Politics,0140-2382,West European Politics,25691,FOUND IN SCOPUS
5,Journal of European Public Policy,1350-1763,Journal of European Public Policy,19878,FOUND IN SCOPUS
6,Philosophy and Public Affairs,0048-3915,Philosophy and Public Affairs,22738,FOUND IN SCOPUS
7,Democratization,1351-0347,Democratization,24034,FOUND IN SCOPUS
8,New Political Economy,1356-3467,New Political Economy,29032,FOUND IN SCOPUS
9,Journal of Common Market Studies,0021-9886,Journal of Common Market Studies,19875,FOUND IN SCOPUS



WESTERN EUROPE REGION — POLITICAL SCIENCE / IR
✅ Confirmados no Scopus: 11 / 11

✅ Todos os periódicos foram encontrados no Scopus.


In [186]:
# ============================================================
# WESTERN EUROPE REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

# 1. Selecionar periódicos confirmados
confirmed_western_europe_psir = (
    scopus_western_europe_psir_df[
        scopus_western_europe_psir_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_western_europe_psir)}"
)


# ============================================================
# 2. CRIAR FILTRO POR ISSN
# ============================================================

issn_queries_western_europe_psir = []

for _, row in confirmed_western_europe_psir.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_western_europe_psir.append(
            f'ISSN({issn})'
        )

journals_part_western_europe_psir = " OR ".join(
    issn_queries_western_europe_psir
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_western_europe_psir)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_WESTERN_EUROPE_PSIR_TT = f'''
(
    {journals_part_western_europe_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_WESTERN_EUROPE_PSIR_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_WESTERN_EUROPE_PSIR_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_western_europe_psir_tt = response.json()

    total_results_western_europe_psir_tt = int(
        data_western_europe_psir_tt[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_western_europe_psir_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 11
Periódicos utilizados na busca: 11

CONSULTA GERADA
(     ISSN(0092-5853) OR ISSN(0003-0554) OR ISSN(1047-1987) OR ISSN(0007-1234) OR ISSN(0140-2382) OR ISSN(1350-1763) OR ISSN(0048-3915) OR ISSN(1351-0347) OR ISSN(1356-3467) OR ISSN(0021-9886) OR ISSN(1755-7739) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 4


In [187]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# WESTERN EUROPE REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER
# 2004–2026
# ============================================================

all_results_western_europe_psir_tt = []

count = 25
start = 0

while start < total_results_western_europe_psir_tt:

    params = {
        "query": SEARCH_QUERY_WESTERN_EUROPE_PSIR_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_western_europe_psir_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_western_europe_psir_tt)} / "
        f"{total_results_western_europe_psir_tt}"
    )

    start += count
    time.sleep(1)


# ============================================================
# CONFERÊNCIA
# ============================================================

print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_western_europe_psir_tt)}"
)


# ============================================================
# MOSTRAR OS 4 ARTIGOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_western_europe_psir_tt,
    start=1
):

    print(f"\n{i}. {item.get('dc:title', '')}")

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   ISSN:",
        item.get("prism:issn", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

    print(
        "   DOI:",
        item.get("prism:doi", "")
    )

Baixados: 4 / 4

DOWNLOAD CONCLUÍDO
Total de registros baixados: 4

ARTIGOS RECUPERADOS

1. Centimanes v. Titans: right-wing populist governments’ treatment of foreign multinationals in East Central Europe
   Periódico: New Political Economy
   ISSN: 13563467
   Data: 2026-01-01
   DOI: 10.1080/13563467.2026.2624414

2. The Two Disjointed Faces of R&D and the Productivity Gap in Europe
   Periódico: Journal of Common Market Studies
   ISSN: 00219886
   Data: 2022-05-01
   DOI: 10.1111/jcms.13260

3. Political Economy of International Policy on the Transfer of Environmentally Sound Technologies in Global Climate Change Regime
   Periódico: New Political Economy
   ISSN: 13563467
   Data: 2019-01-02
   DOI: 10.1080/13563467.2017.1417361

4. The Impact of Patents on Innovation, Technology Transfer and Health: A Pre- and Post-TRIPs Analysis of India's Pharmaceutical Industry
   Periódico: New Political Economy
   ISSN: 13563467
   Data: 2014-01-01
   DOI: 10.1080/13563467.2013.796446


In [188]:
# ============================================================
# EXPORTAR RESULTADOS
# WESTERN EUROPE REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

records_western_europe_psir_tt = []

for item in all_results_western_europe_psir_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_western_europe_psir_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_western_europe_psir_tt = pd.DataFrame(
    records_western_europe_psir_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_western_europe_psir_tt = (
    df_western_europe_psir_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_western_europe_psir_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_western_europe_psir_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Western Europe Region",
        "Political Science and International Relations",
        len(confirmed_western_europe_psir),

        "; ".join(
            confirmed_western_europe_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_western_europe_psir_tt,
        "ISSN",
        len(df_western_europe_psir_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_western_europe_psir_tt = (
    "Scopus_Western_Europe_Region_PSIR_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_western_europe_psir_tt,
    engine="openpyxl"
) as writer:

    df_western_europe_psir_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_western_europe_psir_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_western_europe_psir_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(f"Total de artigos: {len(df_western_europe_psir_tt)}")
print(f"Periódicos pesquisados: {len(confirmed_western_europe_psir)}")

print("\nArtigos por periódico:")
display(journal_counts_western_europe_psir_tt)

print("\nArquivo:")
print(filename_western_europe_psir_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 4
Periódicos pesquisados: 11

Artigos por periódico:


,Journal,Number of articles
0,New Political Economy,3
1,Journal of Common Market Studies,1



Arquivo:
Scopus_Western_Europe_Region_PSIR_Technology_Transfer_2004_2026.xlsx


In [189]:
from google.colab import files

files.download(filename_western_europe_psir_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [190]:
# ============================================================
# WESTERN EUROPE REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION
# 2004–2026
# ============================================================

SEARCH_QUERY_WESTERN_EUROPE_PSIR_TI = f'''
(
    {journals_part_western_europe_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_WESTERN_EUROPE_PSIR_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_WESTERN_EUROPE_PSIR_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_western_europe_psir_ti = response.json()

    total_results_western_europe_psir_ti = int(
        data_western_europe_psir_ti[
            "search-results"
        ][
            "opensearch:totalResults"
        ]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_western_europe_psir_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0092-5853) OR ISSN(0003-0554) OR ISSN(1047-1987) OR ISSN(0007-1234) OR ISSN(0140-2382) OR ISSN(1350-1763) OR ISSN(0048-3915) OR ISSN(1351-0347) OR ISSN(1356-3467) OR ISSN(0021-9886) OR ISSN(1755-7739) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [191]:
# ============================================================
# EXPORTAR RESULTADO ZERO
# WESTERN EUROPE REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION — 2004–2026
# ============================================================

# Aba Results vazia, mantendo as mesmas colunas
columns_western_europe_psir_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_western_europe_psir_ti = pd.DataFrame(
    columns=columns_western_europe_psir_ti
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_western_europe_psir_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)

search_log_western_europe_psir_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Western Europe Region",
        "Political Science and International Relations",
        len(confirmed_western_europe_psir),

        "; ".join(
            confirmed_western_europe_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_western_europe_psir_ti,
        "ISSN",
        total_results_western_europe_psir_ti
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_western_europe_psir_ti = (
    "Scopus_Western_Europe_Region_PSIR_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_western_europe_psir_ti,
    engine="openpyxl"
) as writer:

    df_western_europe_psir_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_western_europe_psir_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{total_results_western_europe_psir_ti}"
)

print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_western_europe_psir)}"
)

print("\nArquivo:")
print(filename_western_europe_psir_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Periódicos pesquisados: 11

Arquivo:
Scopus_Western_Europe_Region_PSIR_Technology_Internalization_2004_2026.xlsx


In [192]:
from google.colab import files

files.download(filename_western_europe_psir_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [193]:
# ============================================================
# NORTHERN AMERICA REGION
# POLITICAL SCIENCE AND INTERNATIONAL RELATIONS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

northern_america_psir_journals = {

    "World Politics": [
        "0043-8871", "1086-3338"
    ],

    "Perspectives on Politics": [
        "1537-5927", "1541-0986"
    ],

    "Review of International Organizations": [
        "1559-7431", "1559-744X"
    ],

    "Quarterly Journal of Political Science": [
        "1554-0626", "1554-0634"
    ],

    "Global Environmental Politics": [
        "1526-3800", "1536-0091"
    ],

    "Politics and Society": [
        "0032-3292", "1552-7514"
    ],

    "Journal of Political Ecology": [
        "1073-0451"
    ],

    "Latin American Politics and Society": [
        "1531-426X", "1548-2456"
    ],

    "Swiss Political Science Review": [
        "1424-7755", "1662-6370"
    ],

    "Territory, Politics, Governance": [
        "2162-2671", "2162-268X"
    ],

    "Foreign Policy Analysis": [
        "1743-8586", "1743-8594"
    ],

    "State Politics and Policy Quarterly": [
        "1532-4400", "1946-1607"
    ],

    "Journal of the Japanese and International Economies": [
        "0889-1583", "1095-8681"
    ],

    "European Policy Analysis": [
        "2380-6567", "2380-6575"
    ],

    "Journal of Law and Courts": [
        "2164-6570", "2164-6589"
    ]
}


# ============================================================
# VERIFICAR NO SCOPUS
# ============================================================

scopus_northern_america_psir = []

for journal, issns in northern_america_psir_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_northern_america_psir.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_northern_america_psir.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# DATAFRAME
# ============================================================

scopus_northern_america_psir_df = pd.DataFrame(
    scopus_northern_america_psir
)


# ============================================================
# MOSTRAR TÍTULO ESPERADO × TÍTULO DEVOLVIDO PELO SCOPUS
# ============================================================

display(
    scopus_northern_america_psir_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# CONTAGEM
# ============================================================

total_confirmed_northern_america_psir = (
    scopus_northern_america_psir_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_northern_america_psir} / "
    f"{len(northern_america_psir_journals)}"
)


# ============================================================
# NÃO ENCONTRADOS
# ============================================================

not_found_northern_america_psir = (
    scopus_northern_america_psir_df[
        scopus_northern_america_psir_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_northern_america_psir) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print("\n⚠️ Periódicos que precisam de verificação:")

    display(
        not_found_northern_america_psir[
            ["Journal", "Status"]
        ]
    )

✅ Encontrado: World Politics
✅ Encontrado: Perspectives on Politics
✅ Encontrado: Review of International Organizations
✅ Encontrado: Quarterly Journal of Political Science
✅ Encontrado: Global Environmental Politics
✅ Encontrado: Politics and Society
✅ Encontrado: Journal of Political Ecology
✅ Encontrado: Latin American Politics and Society
✅ Encontrado: Swiss Political Science Review
✅ Encontrado: Territory, Politics, Governance
✅ Encontrado: Foreign Policy Analysis
✅ Encontrado: State Politics and Policy Quarterly
✅ Encontrado: Journal of the Japanese and International Economies
✅ Encontrado: European Policy Analysis
✅ Encontrado: Journal of Law and Courts


,Journal,ISSN searched,Scopus title,Source ID,Status
0,World Politics,0043-8871,World Politics,25725,FOUND IN SCOPUS
1,Perspectives on Politics,1537-5927,Perspectives on Politics,4000148701,FOUND IN SCOPUS
2,Review of International Organizations,1559-7431,Review of International Organizations,4700152638,FOUND IN SCOPUS
3,Quarterly Journal of Political Science,1554-0626,Quarterly Journal of Political Science,11600153640,FOUND IN SCOPUS
4,Global Environmental Politics,1526-3800,Global Environmental Politics,4900152303,FOUND IN SCOPUS
5,Politics and Society,0032-3292,Politics and Society,23439,FOUND IN SCOPUS
6,Journal of Political Ecology,1073-0451,Journal of Political Ecology,19900193691,FOUND IN SCOPUS
7,Latin American Politics and Society,1531-426X,Latin American Politics and Society,26459,FOUND IN SCOPUS
8,Swiss Political Science Review,1424-7755,Swiss Political Science Review,5700163398,FOUND IN SCOPUS
9,"Territory, Politics, Governance",2162-2671,"Territory, Politics, Governance",21100463834,FOUND IN SCOPUS



NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR
✅ Confirmados no Scopus: 15 / 15

✅ Todos os periódicos foram encontrados no Scopus.


In [194]:
# ============================================================
# NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

# 1. Selecionar periódicos confirmados
confirmed_northern_america_psir = (
    scopus_northern_america_psir_df[
        scopus_northern_america_psir_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_northern_america_psir)}"
)


# ============================================================
# 2. CRIAR FILTRO POR ISSN
# ============================================================

issn_queries_northern_america_psir = []

for _, row in confirmed_northern_america_psir.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_northern_america_psir.append(
            f'ISSN({issn})'
        )

journals_part_northern_america_psir = " OR ".join(
    issn_queries_northern_america_psir
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_northern_america_psir)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER
# ============================================================

SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TT = f'''
(
    {journals_part_northern_america_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_northern_america_psir_tt = response.json()

    total_results_northern_america_psir_tt = int(
        data_northern_america_psir_tt[
            "search-results"
        ]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_northern_america_psir_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 15
Periódicos utilizados na busca: 15

CONSULTA GERADA
(     ISSN(0043-8871) OR ISSN(1537-5927) OR ISSN(1559-7431) OR ISSN(1554-0626) OR ISSN(1526-3800) OR ISSN(0032-3292) OR ISSN(1073-0451) OR ISSN(1531-426X) OR ISSN(1424-7755) OR ISSN(2162-2671) OR ISSN(1743-8586) OR ISSN(1532-4400) OR ISSN(0889-1583) OR ISSN(2380-6567) OR ISSN(2164-6570) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 10


In [195]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

all_results_northern_america_psir_tt = []

count = 25
start = 0

while start < total_results_northern_america_psir_tt:

    params = {
        "query": SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_northern_america_psir_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_northern_america_psir_tt)} / "
        f"{total_results_northern_america_psir_tt}"
    )

    start += count
    time.sleep(1)


# ============================================================
# CONFERÊNCIA
# ============================================================

print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_northern_america_psir_tt)}"
)


# ============================================================
# MOSTRAR OS 10 ARTIGOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_northern_america_psir_tt,
    start=1
):

    print(f"\n{i}. {item.get('dc:title', '')}")

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   ISSN:",
        item.get("prism:issn", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

    print(
        "   DOI:",
        item.get("prism:doi", "")
    )

Baixados: 10 / 10

DOWNLOAD CONCLUÍDO
Total de registros baixados: 10

ARTIGOS RECUPERADOS

1. China in Twenty-Five Years of Global Environmental Politics: The Case of Climate Diplomacy
   Periódico: Global Environmental Politics
   ISSN: 15263800
   Data: 2026-02-01
   DOI: 10.1162/GLEP.a.729

2. Patent or Planet? Rethinking IP for Just Climate Tech Transition in the Global South
   Periódico: Global Environmental Politics
   ISSN: 15263800
   Data: 2025-11-01
   DOI: 10.1162/GLEP.a.25

3. Exporting the Tools of Dictatorship: The Politics of China's Technology Transfers
   Periódico: Perspectives on Politics
   ISSN: 15375927
   Data: 2025-09-01
   DOI: 10.1017/S1537592724002226

4. Behind the screen: Understanding national support for a foreign investment screening mechanism in the European Union
   Periódico: Review of International Organizations
   ISSN: 15597431
   Data: 2022-07-01
   DOI: 10.1007/s11558-021-09436-y

5. Foreign direct investment with host country market structures

In [196]:
# ============================================================
# EXPORTAR RESULTADOS
# NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — 2004–2026
# ============================================================

records_northern_america_psir_tt = []

for item in all_results_northern_america_psir_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_northern_america_psir_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_northern_america_psir_tt = pd.DataFrame(
    records_northern_america_psir_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_northern_america_psir_tt = (
    df_northern_america_psir_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_northern_america_psir_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer"'
)

search_log_northern_america_psir_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Northern America Region",
        "Political Science and International Relations",
        len(confirmed_northern_america_psir),

        "; ".join(
            confirmed_northern_america_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_northern_america_psir_tt,
        "ISSN",
        len(df_northern_america_psir_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_northern_america_psir_tt = (
    "Scopus_Northern_America_Region_PSIR_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_northern_america_psir_tt,
    engine="openpyxl"
) as writer:

    df_northern_america_psir_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_northern_america_psir_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_northern_america_psir_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{len(df_northern_america_psir_tt)}"
)

print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_northern_america_psir)}"
)

print("\nArtigos por periódico:")
display(journal_counts_northern_america_psir_tt)

print("\nArquivo:")
print(filename_northern_america_psir_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 10
Periódicos pesquisados: 15

Artigos por periódico:


,Journal,Number of articles
0,Global Environmental Politics,3
1,Journal of the Japanese and International Econ...,3
2,Review of International Organizations,2
3,Perspectives on Politics,1
4,Journal of Political Ecology,1



Arquivo:
Scopus_Northern_America_Region_PSIR_Technology_Transfer_2004_2026.xlsx


In [197]:
from google.colab import files

files.download(filename_northern_america_psir_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [198]:
# ============================================================
# NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION — 2004–2026
# ============================================================

SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TI = f'''
(
    {journals_part_northern_america_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_NORTHERN_AMERICA_PSIR_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_northern_america_psir_ti = response.json()

    total_results_northern_america_psir_ti = int(
        data_northern_america_psir_ti[
            "search-results"
        ]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_northern_america_psir_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0043-8871) OR ISSN(1537-5927) OR ISSN(1559-7431) OR ISSN(1554-0626) OR ISSN(1526-3800) OR ISSN(0032-3292) OR ISSN(1073-0451) OR ISSN(1531-426X) OR ISSN(1424-7755) OR ISSN(2162-2671) OR ISSN(1743-8586) OR ISSN(1532-4400) OR ISSN(0889-1583) OR ISSN(2380-6567) OR ISSN(2164-6570) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [200]:
# ============================================================
# EXPORTAR RESULTADO ZERO
# NORTHERN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION — 2004–2026
# ============================================================

columns_northern_america_psir_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_northern_america_psir_ti = pd.DataFrame(
    columns=columns_northern_america_psir_ti
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_northern_america_psir_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization"'
)

search_log_northern_america_psir_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Northern America Region",
        "Political Science and International Relations",
        len(confirmed_northern_america_psir),

        "; ".join(
            confirmed_northern_america_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English",
        3,
        search_terms_northern_america_psir_ti,
        "ISSN",
        total_results_northern_america_psir_ti
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_northern_america_psir_ti = (
    "Scopus_Northern_America_Region_PSIR_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_northern_america_psir_ti,
    engine="openpyxl"
) as writer:

    df_northern_america_psir_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_northern_america_psir_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{total_results_northern_america_psir_ti}"
)

print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_northern_america_psir)}"
)

print("\nArquivo:")
print(filename_northern_america_psir_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Periódicos pesquisados: 15

Arquivo:
Scopus_Northern_America_Region_PSIR_Technology_Internalization_2004_2026.xlsx


In [201]:
from google.colab import files

files.download(filename_northern_america_psir_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [202]:
# ============================================================
# LATIN AMERICA REGION
# POLITICAL SCIENCE AND INTERNATIONAL RELATIONS
# VERIFICAÇÃO NO SCOPUS
# ============================================================

latin_america_psir_journals = {

    "Revista de Ciencia Politica": [
        "0716-1417", "0718-090X"
    ],

    "Brazilian Journal of Political Economy": [
        "0101-3157", "1809-4538"
    ],

    "Estudios Fronterizos": [
        "0187-6961", "2395-9134"
    ],

    "Justicia (Barranquilla)": [
        "0124-7441", "2590-4566"
    ],

    "Norteamerica": [
        "1870-3550", "2448-7228"
    ],

    "Revista Brasileira de Politica Internacional": [
        "0034-7329", "1983-3121"
    ],

    "Colombia Internacional": [
        "0121-5612", "1900-6004"
    ],

    "Gestion y Politica Publica": [
        "1405-1079"
    ],

    "Estudios Sociologicos": [
        "0185-4186", "2448-6442"
    ],

    "Ensayos Sobre Politica Economica": [
        "0120-4483"
    ],

    "Izquierdas": [
        "0718-5049"
    ],

    "Analisis Politico": [
        "0121-4705"
    ],

    "Desarrollo y Sociedad": [
        "0120-3584", "1900-7760"
    ],

    "Estudios de Derecho": [
        "0120-1867", "2145-6151"
    ],

    "Revista Justicia y Derecho": [
        "0719-9392"
    ],

    "Economia y Politica": [
        "0719-4803"
    ],

    "Politica y Gobierno": [
        "1665-2037"
    ],

    "Mexico y la Cuenca del Pacifico": [
        "1665-0174", "2007-5308"
    ]
}


# ============================================================
# VERIFICAR NO SCOPUS
# ============================================================

scopus_latin_america_psir = []

for journal, issns in latin_america_psir_journals.items():

    found = False

    for issn in issns:

        params = {
            "issn": issn,
            "view": "STANDARD"
        }

        response = requests.get(
            serial_url,
            headers=headers_serial,
            params=params
        )

        if response.status_code == 200:

            data = response.json()

            entries = data.get(
                "serial-metadata-response",
                {}
            ).get("entry", [])

            if entries:

                entry = entries[0]

                scopus_latin_america_psir.append({
                    "Journal": journal,
                    "ISSN searched": issn,
                    "Scopus title": entry.get("dc:title", ""),
                    "Source ID": entry.get("source-id", ""),
                    "Publisher": entry.get("dc:publisher", ""),
                    "Status": "FOUND IN SCOPUS"
                })

                found = True
                break

        time.sleep(0.2)


    if found:

        print(f"✅ Encontrado: {journal}")

    else:

        scopus_latin_america_psir.append({
            "Journal": journal,
            "ISSN searched": "",
            "Scopus title": "",
            "Source ID": "",
            "Publisher": "",
            "Status": "NOT FOUND"
        })

        print(f"⚠️ Não encontrado: {journal}")


# ============================================================
# DATAFRAME
# ============================================================

scopus_latin_america_psir_df = pd.DataFrame(
    scopus_latin_america_psir
)


# ============================================================
# MOSTRAR TÍTULO ESPERADO × TÍTULO RETORNADO
# ============================================================

display(
    scopus_latin_america_psir_df[
        [
            "Journal",
            "ISSN searched",
            "Scopus title",
            "Source ID",
            "Status"
        ]
    ]
)


# ============================================================
# CONTAGEM
# ============================================================

total_confirmed_latin_america_psir = (
    scopus_latin_america_psir_df["Status"]
    == "FOUND IN SCOPUS"
).sum()


print("\n============================================================")
print("LATIN AMERICA REGION — POLITICAL SCIENCE / IR")
print("============================================================")

print(
    f"✅ Confirmados no Scopus: "
    f"{total_confirmed_latin_america_psir} / "
    f"{len(latin_america_psir_journals)}"
)


# ============================================================
# NÃO ENCONTRADOS
# ============================================================

not_found_latin_america_psir = (
    scopus_latin_america_psir_df[
        scopus_latin_america_psir_df["Status"]
        == "NOT FOUND"
    ]
)

if len(not_found_latin_america_psir) == 0:

    print(
        "\n✅ Todos os periódicos foram encontrados no Scopus."
    )

else:

    print("\n⚠️ Periódicos que precisam de verificação:")

    display(
        not_found_latin_america_psir[
            ["Journal", "Status"]
        ]
    )

✅ Encontrado: Revista de Ciencia Politica
✅ Encontrado: Brazilian Journal of Political Economy
✅ Encontrado: Estudios Fronterizos
✅ Encontrado: Justicia (Barranquilla)
✅ Encontrado: Norteamerica
✅ Encontrado: Revista Brasileira de Politica Internacional
✅ Encontrado: Colombia Internacional
✅ Encontrado: Gestion y Politica Publica
✅ Encontrado: Estudios Sociologicos
✅ Encontrado: Ensayos Sobre Politica Economica
✅ Encontrado: Izquierdas
✅ Encontrado: Analisis Politico
✅ Encontrado: Desarrollo y Sociedad
✅ Encontrado: Estudios de Derecho
✅ Encontrado: Revista Justicia y Derecho
✅ Encontrado: Economia y Politica
✅ Encontrado: Politica y Gobierno
✅ Encontrado: Mexico y la Cuenca del Pacifico


,Journal,ISSN searched,Scopus title,Source ID,Status
0,Revista de Ciencia Politica,0716-1417,Revista de Ciencia Politica,5100152922,FOUND IN SCOPUS
1,Brazilian Journal of Political Economy,0101-3157,Brazilian Journal of Political Economy,5800169372,FOUND IN SCOPUS
2,Estudios Fronterizos,2395-9134,Estudios Fronterizos,21100933830,FOUND IN SCOPUS
3,Justicia (Barranquilla),0124-7441,Justicia (Barranquilla),21101215777,FOUND IN SCOPUS
4,Norteamerica,1870-3550,Norteamerica,21100824449,FOUND IN SCOPUS
5,Revista Brasileira de Politica Internacional,0034-7329,Revista Brasileira de Politica Internacional,5700168481,FOUND IN SCOPUS
6,Colombia Internacional,0121-5612,Colombia Internacional,5800169849,FOUND IN SCOPUS
7,Gestion y Politica Publica,1405-1079,Gestion y Politica Publica,22826,FOUND IN SCOPUS
8,Estudios Sociologicos,0185-4186,Estudios Sociologicos,14377,FOUND IN SCOPUS
9,Ensayos Sobre Politica Economica,0120-4483,Ensayos Sobre Politica Economica,19700169972,FOUND IN SCOPUS



LATIN AMERICA REGION — POLITICAL SCIENCE / IR
✅ Confirmados no Scopus: 18 / 18

✅ Todos os periódicos foram encontrados no Scopus.


In [203]:
# ============================================================
# LATIN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — MULTILINGUAL
# 2004–2026
# ============================================================

# 1. Selecionar periódicos confirmados
confirmed_latin_america_psir = (
    scopus_latin_america_psir_df[
        scopus_latin_america_psir_df["Status"]
        == "FOUND IN SCOPUS"
    ].copy()
)

print(
    f"Periódicos confirmados: "
    f"{len(confirmed_latin_america_psir)}"
)


# ============================================================
# 2. CRIAR FILTRO POR ISSN
# ============================================================

issn_queries_latin_america_psir = []

for _, row in confirmed_latin_america_psir.iterrows():

    issn = row["ISSN searched"]

    if issn:
        issn_queries_latin_america_psir.append(
            f'ISSN({issn})'
        )

journals_part_latin_america_psir = " OR ".join(
    issn_queries_latin_america_psir
)

print(
    f"Periódicos utilizados na busca: "
    f"{len(issn_queries_latin_america_psir)}"
)


# ============================================================
# 3. TECHNOLOGY TRANSFER — 7 EXPRESSÕES
# ============================================================

SEARCH_QUERY_LATIN_AMERICA_PSIR_TT = f'''
(
    {journals_part_latin_america_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Transfer"
    OR
    "Transfer of Technology"
    OR
    "Technological Transfer"
    OR
    "Transferência de Tecnologia"
    OR
    "Transferencia de Tecnología"
    OR
    "Transferência Tecnológica"
    OR
    "Transferencia Tecnológica"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# 4. MOSTRAR CONSULTA
# ============================================================

print("\n============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_LATIN_AMERICA_PSIR_TT)


# ============================================================
# 5. EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_LATIN_AMERICA_PSIR_TT,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_latin_america_psir_tt = response.json()

    total_results_latin_america_psir_tt = int(
        data_latin_america_psir_tt[
            "search-results"
        ]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_latin_america_psir_tt}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

Periódicos confirmados: 18
Periódicos utilizados na busca: 18

CONSULTA GERADA
(     ISSN(0716-1417) OR ISSN(0101-3157) OR ISSN(2395-9134) OR ISSN(0124-7441) OR ISSN(1870-3550) OR ISSN(0034-7329) OR ISSN(0121-5612) OR ISSN(1405-1079) OR ISSN(0185-4186) OR ISSN(0120-4483) OR ISSN(0718-5049) OR ISSN(0121-4705) OR ISSN(0120-3584) OR ISSN(0120-1867) OR ISSN(0719-9392) OR ISSN(0719-4803) OR ISSN(1665-2037) OR ISSN(1665-0174) ) AND TITLE-ABS-KEY(     "Technology Transfer"     OR     "Transfer of Technology"     OR     "Technological Transfer"     OR     "Transferência de Tecnologia"     OR     "Transferencia de Tecnología"     OR     "Transferência Tecnológica"     OR     "Transferencia Tecnológica" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 5


In [205]:
# ============================================================
# BAIXAR E CONFERIR RESULTADOS
# LATIN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — MULTILINGUAL — 2004–2026
# ============================================================

all_results_latin_america_psir_tt = []

count = 25
start = 0

while start < total_results_latin_america_psir_tt:

    params = {
        "query": SEARCH_QUERY_LATIN_AMERICA_PSIR_TT,
        "count": count,
        "start": start
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    if response.status_code != 200:
        print("❌ Erro:", response.status_code)
        print(response.text)
        break

    page_data = response.json()

    entries = page_data[
        "search-results"
    ].get("entry", [])

    all_results_latin_america_psir_tt.extend(entries)

    print(
        f"Baixados: "
        f"{len(all_results_latin_america_psir_tt)} / "
        f"{total_results_latin_america_psir_tt}"
    )

    start += count
    time.sleep(1)


# ============================================================
# CONFERÊNCIA
# ============================================================

print("\n============================================================")
print("DOWNLOAD CONCLUÍDO")
print("============================================================")

print(
    f"Total de registros baixados: "
    f"{len(all_results_latin_america_psir_tt)}"
)


# ============================================================
# MOSTRAR OS 5 ARTIGOS
# ============================================================

print("\n============================================================")
print("ARTIGOS RECUPERADOS")
print("============================================================")

for i, item in enumerate(
    all_results_latin_america_psir_tt,
    start=1
):

    print(f"\n{i}. {item.get('dc:title', '')}")

    print(
        "   Periódico:",
        item.get("prism:publicationName", "")
    )

    print(
        "   ISSN:",
        item.get("prism:issn", "")
    )

    print(
        "   Data:",
        item.get("prism:coverDate", "")
    )

    print(
        "   DOI:",
        item.get("prism:doi", "")
    )

Baixados: 5 / 5

DOWNLOAD CONCLUÍDO
Total de registros baixados: 5

ARTIGOS RECUPERADOS

1. The Limits of Quality Investment in Mexico under the USMCA
   Periódico: Norteamerica
   ISSN: 18703550
   Data: 2026-01-01
   DOI: 10.22201/cisan.24487228e.2026.1.784

2. Analysis of the impact of cooperation in science and technology between north America and the European Union. Parallels between Hungary and Mexico edit
   Periódico: Norteamerica
   ISSN: 18703550
   Data: 2019-01-01
   DOI: 10.22201/cisan.24487228e.2019.1.372

3. The structural inertia in the national productive base: The FDI and the international transference of technology
   Periódico: Revista De Economia Politica
   ISSN: 01013157
   Data: 2016-04-01
   DOI: 10.1590/0101-31572016v36n02a03

4. On the transference of organizational models
   Periódico: Gestion Y Politica Publica
   ISSN: 14051079
   Data: 2014-01-01
   DOI: 

5. Knowledge and technology transfer strategies: Best practices in Spanish entrepreneurial universit

In [206]:
# ============================================================
# EXPORTAR RESULTADOS
# LATIN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY TRANSFER — MULTILINGUAL — 2004–2026
# ============================================================

records_latin_america_psir_tt = []

for item in all_results_latin_america_psir_tt:

    cover_date = item.get("prism:coverDate", "")
    year = cover_date[:4] if cover_date else ""

    scopus_url = ""

    for link in item.get("link", []):
        if link.get("@ref") == "scopus":
            scopus_url = link.get("@href", "")

    records_latin_america_psir_tt.append({
        "Journal": item.get("prism:publicationName", ""),
        "Title": item.get("dc:title", ""),
        "Authors": item.get("dc:creator", ""),
        "Year": year,
        "ISSN": item.get("prism:issn", ""),
        "Volume": item.get("prism:volume", ""),
        "Issue": item.get("prism:issueIdentifier", ""),
        "Pages": item.get("prism:pageRange", ""),
        "DOI": item.get("prism:doi", ""),
        "EID": item.get("eid", ""),
        "Cited by": item.get("citedby-count", ""),
        "Scopus URL": scopus_url
    })


df_latin_america_psir_tt = pd.DataFrame(
    records_latin_america_psir_tt
)


# ============================================================
# CONTAGEM POR PERIÓDICO
# ============================================================

journal_counts_latin_america_psir_tt = (
    df_latin_america_psir_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_latin_america_psir_tt = (
    '"Technology Transfer" OR '
    '"Transfer of Technology" OR '
    '"Technological Transfer" OR '
    '"Transferência de Tecnologia" OR '
    '"Transferencia de Tecnología" OR '
    '"Transferência Tecnológica" OR '
    '"Transferencia Tecnológica"'
)

search_log_latin_america_psir_tt = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Latin America Region",
        "Political Science and International Relations",
        len(confirmed_latin_america_psir),

        "; ".join(
            confirmed_latin_america_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English; Portuguese; Spanish",
        7,
        search_terms_latin_america_psir_tt,
        "ISSN",
        len(df_latin_america_psir_tt)
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_latin_america_psir_tt = (
    "Scopus_Latin_America_Region_PSIR_"
    "Technology_Transfer_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_latin_america_psir_tt,
    engine="openpyxl"
) as writer:

    df_latin_america_psir_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_latin_america_psir_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_latin_america_psir_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{len(df_latin_america_psir_tt)}"
)

print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_latin_america_psir)}"
)

print("\nArtigos por periódico:")
display(journal_counts_latin_america_psir_tt)

print("\nArquivo:")
print(filename_latin_america_psir_tt)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 5
Periódicos pesquisados: 18

Artigos por periódico:


,Journal,Number of articles
0,Norteamerica,2
1,Gestion Y Politica Publica,2
2,Revista De Economia Politica,1



Arquivo:
Scopus_Latin_America_Region_PSIR_Technology_Transfer_2004_2026.xlsx


In [207]:
# ============================================================
# PADRONIZAR NOME DO PERIÓDICO
# ============================================================

df_latin_america_psir_tt["Journal"] = (
    df_latin_america_psir_tt["Journal"]
    .replace({
        "Revista De Economia Politica":
            "Brazilian Journal of Political Economy",
        "Revista de Economia Politica":
            "Brazilian Journal of Political Economy"
    })
)

# Refazer a contagem por periódico
journal_counts_latin_america_psir_tt = (
    df_latin_america_psir_tt["Journal"]
    .value_counts()
    .rename_axis("Journal")
    .reset_index(name="Number of articles")
)

display(journal_counts_latin_america_psir_tt)

,Journal,Number of articles
0,Norteamerica,2
1,Gestion Y Politica Publica,2
2,Brazilian Journal of Political Economy,1


In [208]:
with pd.ExcelWriter(
    filename_latin_america_psir_tt,
    engine="openpyxl"
) as writer:

    df_latin_america_psir_tt.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    journal_counts_latin_america_psir_tt.to_excel(
        writer,
        sheet_name="Journal Counts",
        index=False
    )

    search_log_latin_america_psir_tt.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )

print("✅ Planilha atualizada com o nome padronizado.")
print(filename_latin_america_psir_tt)

✅ Planilha atualizada com o nome padronizado.
Scopus_Latin_America_Region_PSIR_Technology_Transfer_2004_2026.xlsx


In [209]:
from google.colab import files
files.download(filename_latin_america_psir_tt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [210]:
# ============================================================
# LATIN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION — MULTILINGUAL
# 2004–2026
# ============================================================

SEARCH_QUERY_LATIN_AMERICA_PSIR_TI = f'''
(
    {journals_part_latin_america_psir}
)
AND
TITLE-ABS-KEY(
    "Technology Internalization"
    OR
    "Internalization of Technology"
    OR
    "Technological Internalization"
    OR
    "Internalização de Tecnologia"
    OR
    "Internalización de Tecnología"
    OR
    "Internalização Tecnológica"
    OR
    "Internalización Tecnológica"
)
AND
PUBYEAR > 2003
AND
PUBYEAR < 2027
AND
DOCTYPE(ar)
'''.replace("\n", " ").strip()


# ============================================================
# MOSTRAR CONSULTA
# ============================================================

print("============================================================")
print("CONSULTA GERADA")
print("============================================================")

print(SEARCH_QUERY_LATIN_AMERICA_PSIR_TI)


# ============================================================
# EXECUTAR CONSULTA
# ============================================================

params = {
    "query": SEARCH_QUERY_LATIN_AMERICA_PSIR_TI,
    "count": 25,
    "start": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

print("\n============================================================")
print("RESULTADO DA CONSULTA")
print("============================================================")

print("Status da API:", response.status_code)


if response.status_code == 200:

    data_latin_america_psir_ti = response.json()

    total_results_latin_america_psir_ti = int(
        data_latin_america_psir_ti[
            "search-results"
        ]["opensearch:totalResults"]
    )

    print("\n✅ CONSULTA FUNCIONOU!")

    print(
        f"Total de resultados encontrados: "
        f"{total_results_latin_america_psir_ti}"
    )

else:

    print("\n❌ ERRO NA CONSULTA")
    print(response.text)

CONSULTA GERADA
(     ISSN(0716-1417) OR ISSN(0101-3157) OR ISSN(2395-9134) OR ISSN(0124-7441) OR ISSN(1870-3550) OR ISSN(0034-7329) OR ISSN(0121-5612) OR ISSN(1405-1079) OR ISSN(0185-4186) OR ISSN(0120-4483) OR ISSN(0718-5049) OR ISSN(0121-4705) OR ISSN(0120-3584) OR ISSN(0120-1867) OR ISSN(0719-9392) OR ISSN(0719-4803) OR ISSN(1665-2037) OR ISSN(1665-0174) ) AND TITLE-ABS-KEY(     "Technology Internalization"     OR     "Internalization of Technology"     OR     "Technological Internalization"     OR     "Internalização de Tecnologia"     OR     "Internalización de Tecnología"     OR     "Internalização Tecnológica"     OR     "Internalización Tecnológica" ) AND PUBYEAR > 2003 AND PUBYEAR < 2027 AND DOCTYPE(ar)

RESULTADO DA CONSULTA
Status da API: 200

✅ CONSULTA FUNCIONOU!
Total de resultados encontrados: 0


In [211]:
# ============================================================
# EXPORTAR RESULTADO ZERO
# LATIN AMERICA REGION — POLITICAL SCIENCE / IR
# TECHNOLOGY INTERNALIZATION — MULTILINGUAL — 2004–2026
# ============================================================

columns_latin_america_psir_ti = [
    "Journal",
    "Title",
    "Authors",
    "Year",
    "ISSN",
    "Volume",
    "Issue",
    "Pages",
    "DOI",
    "EID",
    "Cited by",
    "Scopus URL"
]

df_latin_america_psir_ti = pd.DataFrame(
    columns=columns_latin_america_psir_ti
)


# ============================================================
# SEARCH LOG
# ============================================================

search_terms_latin_america_psir_ti = (
    '"Technology Internalization" OR '
    '"Internalization of Technology" OR '
    '"Technological Internalization" OR '
    '"Internalização de Tecnologia" OR '
    '"Internalización de Tecnología" OR '
    '"Internalização Tecnológica" OR '
    '"Internalización Tecnológica"'
)

search_log_latin_america_psir_ti = pd.DataFrame({

    "Field": [
        "Database",
        "Search date",
        "Region",
        "Subject area",
        "Number of journals",
        "Journals",
        "Publication years",
        "Document type",
        "Search field",
        "Languages",
        "Number of search phrases",
        "Search terms",
        "Source identification",
        "Total results"
    ],

    "Value": [
        "Scopus",
        pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Latin America Region",
        "Political Science and International Relations",
        len(confirmed_latin_america_psir),

        "; ".join(
            confirmed_latin_america_psir["Journal"].tolist()
        ),

        "2004–2026",
        "Article",
        "TITLE-ABS-KEY",
        "English; Portuguese; Spanish",
        7,
        search_terms_latin_america_psir_ti,
        "ISSN",
        total_results_latin_america_psir_ti
    ]
})


# ============================================================
# EXPORTAR EXCEL
# ============================================================

filename_latin_america_psir_ti = (
    "Scopus_Latin_America_Region_PSIR_"
    "Technology_Internalization_2004_2026.xlsx"
)

with pd.ExcelWriter(
    filename_latin_america_psir_ti,
    engine="openpyxl"
) as writer:

    df_latin_america_psir_ti.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    search_log_latin_america_psir_ti.to_excel(
        writer,
        sheet_name="Search Log",
        index=False
    )


print("============================================================")
print("✅ EXCEL CRIADO COM SUCESSO")
print("============================================================")

print(
    f"Total de artigos: "
    f"{total_results_latin_america_psir_ti}"
)

print(
    f"Periódicos pesquisados: "
    f"{len(confirmed_latin_america_psir)}"
)

print("\nArquivo:")
print(filename_latin_america_psir_ti)

✅ EXCEL CRIADO COM SUCESSO
Total de artigos: 0
Periódicos pesquisados: 18

Arquivo:
Scopus_Latin_America_Region_PSIR_Technology_Internalization_2004_2026.xlsx


In [212]:
from google.colab import files

files.download(filename_latin_america_psir_ti)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>